<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/02_Preprocessing_Encoding_%26_Data_Splits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
# ==================================================================================================
# NOTEBOOK 02 — PREPROCESSING, ENCODING & DATA SPLITS
# ==================================================================================================
#
# Project:
#   SPP-GAN — Privacy-Preserving Synthetic Tabular Data Generation with GANs
#
# Notebook Purpose:
#   This notebook converts the validated raw datasets from Notebook 01 into reproducible,
#   leakage-controlled datasets suitable for downstream statistical, generative,
#   privacy-preserving, and machine-learning utility experiments.
#
# Processing stages:
#   1. Load validated dataset registry
#   2. Normalize recognized missing-value markers
#   3. Validate feature types
#   4. Validate target variables
#   5. Create immutable original row identifiers
#   6. Create stratified train/validation/test splits
#   7. Verify split integrity
#   8. Fit preprocessing objects on TRAINING DATA ONLY
#   9. Transform train/validation/test datasets
#  10. Save native and encoded datasets
#  11. Save preprocessing objects, schemas, feature mappings and manifests
#  12. Reload saved artifacts and verify reproducibility
#
# IMPORTANT METHODOLOGICAL BOUNDARY:
#   This notebook does NOT:
#       - train GANs
#       - generate synthetic data
#       - perform statistical baseline experiments
#       - perform privacy accounting
#       - evaluate synthetic-data quality
#       - remove rows based on outlier detection
#       - perform target-based feature selection
#
# Leakage-control principle:
#   Any learned preprocessing parameter MUST be fitted using TRAINING DATA ONLY.
#
# ==================================================================================================

print("=" * 100)
print("NOTEBOOK 02 — PREPROCESSING, ENCODING & DATA SPLITS")
print("=" * 100)

print()
print("✓ Notebook scope initialized.")
print("✓ Raw-data validation boundary inherited from Notebook 01.")
print("✓ Leakage-controlled preprocessing will be enforced.")

NOTEBOOK 02 — PREPROCESSING, ENCODING & DATA SPLITS

✓ Notebook scope initialized.
✓ Raw-data validation boundary inherited from Notebook 01.
✓ Leakage-controlled preprocessing will be enforced.


In [33]:
# ==================================================================================================
# 2. IMPORT LIBRARIES
# ==================================================================================================

import os
import sys
import json
import hashlib
import pickle
import warnings
import platform
import time

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.impute import SimpleImputer

import joblib

warnings.filterwarnings("ignore")

print("=" * 100)
print("LIBRARY IMPORT STATUS")
print("=" * 100)

print(f"✓ Python       : {platform.python_version()}")
print(f"✓ NumPy        : {np.__version__}")
print(f"✓ Pandas       : {pd.__version__}")
print(f"✓ Scikit-learn: imported")
print(f"✓ Joblib       : {joblib.__version__}")

print()
print("✓ All required libraries imported successfully.")

LIBRARY IMPORT STATUS
✓ Python       : 3.13.15
✓ NumPy        : 2.1.3
✓ Pandas       : 2.2.3
✓ Scikit-learn: imported
✓ Joblib       : 1.5.3

✓ All required libraries imported successfully.


In [34]:
# ==================================================================================================
# 3. LOAD NOTEBOOK 00 CONFIGURATION
# ==================================================================================================

print("=" * 100)
print("3. LOAD NOTEBOOK 00 CONFIGURATION")
print("=" * 100)

from pathlib import Path


# --------------------------------------------------------------------------------------------------
# 3.1 Google Drive
# --------------------------------------------------------------------------------------------------

if "google.colab" in sys.modules:

    from google.colab import drive

    DRIVE_MOUNT_POINT = Path("/content/drive")
    DRIVE_MYDRIVE = DRIVE_MOUNT_POINT / "MyDrive"

    if not DRIVE_MYDRIVE.exists():

        print()
        print("Mounting Google Drive...")

        drive.mount(
            str(DRIVE_MOUNT_POINT),
            force_remount=False
        )

    if not DRIVE_MYDRIVE.exists():

        raise RuntimeError(
            "Google Drive MyDrive could not be verified."
        )

    print("✓ Google Drive verified.")

else:

    DRIVE_MOUNT_POINT = Path("/content/drive")
    DRIVE_MYDRIVE = DRIVE_MOUNT_POINT / "MyDrive"

    print("⚠ Non-Colab environment detected.")


# --------------------------------------------------------------------------------------------------
# 3.2 Resolve project root
# --------------------------------------------------------------------------------------------------

if "PROJECT_ROOT" in globals():

    PROJECT_ROOT = Path(PROJECT_ROOT)

else:

    PROJECT_ROOT = DRIVE_MYDRIVE / "SPP_GAN_Research"


PROJECT_ROOT = PROJECT_ROOT.resolve()


# --------------------------------------------------------------------------------------------------
# 3.3 Resolve directory registry
# --------------------------------------------------------------------------------------------------

if "DIRECTORIES" not in globals():

    DIRECTORIES = {
        "data": PROJECT_ROOT / "data",
        "raw_data": PROJECT_ROOT / "data" / "raw",
        "processed_data": PROJECT_ROOT / "data" / "processed",

        "models": PROJECT_ROOT / "models",
        "tvae_models": PROJECT_ROOT / "models" / "tvae",
        "ctgan_models": PROJECT_ROOT / "models" / "ctgan",
        "dp_ctgan_models": PROJECT_ROOT / "models" / "dp_ctgan",
        "spp_gan_models": PROJECT_ROOT / "models" / "spp_gan",

        "synthetic_data": PROJECT_ROOT / "synthetic_data",
        "tvae_synthetic": PROJECT_ROOT / "synthetic_data" / "tvae",
        "ctgan_synthetic": PROJECT_ROOT / "synthetic_data" / "ctgan",
        "dp_ctgan_synthetic": PROJECT_ROOT / "synthetic_data" / "dp_ctgan",
        "spp_gan_synthetic": PROJECT_ROOT / "synthetic_data" / "spp_gan",

        "results": PROJECT_ROOT / "results",
        "statistical_results": PROJECT_ROOT / "results" / "statistical",
        "ml_results": PROJECT_ROOT / "results" / "machine_learning",
        "privacy_results": PROJECT_ROOT / "results" / "privacy",
        "utility_results": PROJECT_ROOT / "results" / "utility",
        "comparative_results": PROJECT_ROOT / "results" / "comparative",
        "raw_validation_results": PROJECT_ROOT / "results" / "raw_validation",

        "logs": PROJECT_ROOT / "logs",
        "checkpoints": PROJECT_ROOT / "checkpoints",
        "manifests": PROJECT_ROOT / "manifests",

        "figures": PROJECT_ROOT / "paper" / "figures",
        "tables": PROJECT_ROOT / "paper" / "tables",

        "config": PROJECT_ROOT / "config",
    }


DIRECTORIES = {
    key: Path(value)
    for key, value in DIRECTORIES.items()
}


# --------------------------------------------------------------------------------------------------
# 3.4 Configuration constants
# --------------------------------------------------------------------------------------------------

MASTER_SEED = 2025

print()
print(f"Project root : {PROJECT_ROOT}")
print(f"Master seed  : {MASTER_SEED}")

print()
print("✓ Notebook 00 configuration context resolved.")

3. LOAD NOTEBOOK 00 CONFIGURATION
✓ Google Drive verified.

Project root : /content/drive/MyDrive/SPP_GAN_Research
Master seed  : 2025

✓ Notebook 00 configuration context resolved.


In [35]:
# ==================================================================================================
# SECTION 4 — LOAD NOTEBOOK 01 VALIDATED REGISTRY
# ==================================================================================================

print("=" * 100)
print("4. LOAD NOTEBOOK 01 VALIDATED REGISTRY")
print("=" * 100)

from pathlib import Path
import pandas as pd
import json
import re


# --------------------------------------------------------------------------------------------------
# 4.1 VERIFY PROJECT ROOT
# --------------------------------------------------------------------------------------------------

if "PROJECT_ROOT" not in globals():
    raise RuntimeError(
        "PROJECT_ROOT is not available. "
        "Execute Section 3 — Load Notebook 00 Configuration first."
    )

PROJECT_ROOT = Path(PROJECT_ROOT)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_VALIDATION_DIR = PROJECT_ROOT / "results" / "raw_validation"

print(f"PROJECT_ROOT       : {PROJECT_ROOT}")
print(f"RAW_DATA_DIR       : {RAW_DATA_DIR}")
print(f"RAW_VALIDATION_DIR : {RAW_VALIDATION_DIR}")

if not PROJECT_ROOT.exists():
    raise RuntimeError(
        f"PROJECT_ROOT does not exist:\n{PROJECT_ROOT}"
    )

if not RAW_DATA_DIR.exists():
    raise RuntimeError(
        f"Raw data directory does not exist:\n{RAW_DATA_DIR}"
    )

if not RAW_VALIDATION_DIR.exists():
    raise RuntimeError(
        f"Notebook 01 validation directory does not exist:\n"
        f"{RAW_VALIDATION_DIR}"
    )

print("✓ Project root verified")
print("✓ Raw data directory verified")
print("✓ Notebook 01 validation directory verified")


# --------------------------------------------------------------------------------------------------
# 4.2 CANONICAL DATASET LIST
# --------------------------------------------------------------------------------------------------

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

print("\nRequired datasets:")

for dataset_id in DATASET_IDS:
    print(f"  - {dataset_id}")


# --------------------------------------------------------------------------------------------------
# 4.3 CANONICAL TARGET POLICY
# --------------------------------------------------------------------------------------------------

TARGET_COLUMNS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

print("\nTarget policy:")

for dataset_id in DATASET_IDS:
    print(
        f"  {dataset_id:20s} -> {TARGET_COLUMNS[dataset_id]}"
    )


# --------------------------------------------------------------------------------------------------
# 4.4 CANONICAL IDENTIFIER POLICY
# --------------------------------------------------------------------------------------------------

IDENTIFIER_COLUMNS = {
    "adult_income": [],
    "bank_marketing": [],
    "diabetes_130us": [
        "encounter_id",
        "patient_nbr",
    ],
}

print("\nIdentifier policy:")

for dataset_id in DATASET_IDS:
    print(
        f"  {dataset_id:20s} -> {IDENTIFIER_COLUMNS[dataset_id]}"
    )


# --------------------------------------------------------------------------------------------------
# 4.5 NOTEBOOK 01 VALIDATION ARTIFACTS
# --------------------------------------------------------------------------------------------------

STRUCTURAL_DIR = RAW_VALIDATION_DIR / "structural"
FEATURE_DIR = RAW_VALIDATION_DIR / "feature_inventory"
MISSINGNESS_DIR = RAW_VALIDATION_DIR / "missingness"

STRUCTURAL_REPORT_PATH = (
    STRUCTURAL_DIR / "structural_validation_report.csv"
)

DATASET_VALIDATION_REPORT_PATH = (
    STRUCTURAL_DIR / "dataset_validation_report.csv"
)

FEATURE_INVENTORY_PATH = (
    FEATURE_DIR / "feature_inventory.csv"
)

MISSINGNESS_REPORT_PATH = (
    MISSINGNESS_DIR / "raw_missingness_report.csv"
)

print("\nNotebook 01 artifacts:")

artifact_paths = {
    "Structural validation": STRUCTURAL_REPORT_PATH,
    "Dataset validation": DATASET_VALIDATION_REPORT_PATH,
    "Feature inventory": FEATURE_INVENTORY_PATH,
    "Missingness report": MISSINGNESS_REPORT_PATH,
}

for name, path in artifact_paths.items():

    if path.exists():
        print(f"  ✓ {name}: {path}")
    else:
        print(f"  - {name}: not found")


# --------------------------------------------------------------------------------------------------
# 4.6 LOAD AVAILABLE VALIDATION REPORTS
# --------------------------------------------------------------------------------------------------

NOTEBOOK_01_REPORTS = {}

for name, path in artifact_paths.items():

    if path.exists():

        try:

            NOTEBOOK_01_REPORTS[name] = pd.read_csv(
                path,
                low_memory=False
            )

            print(
                f"✓ Loaded {name}: "
                f"{NOTEBOOK_01_REPORTS[name].shape}"
            )

        except Exception as exc:

            raise RuntimeError(
                f"Failed to load Notebook 01 artifact:\n"
                f"{path}\n"
                f"Error: {repr(exc)}"
            ) from exc


# --------------------------------------------------------------------------------------------------
# 4.7 DISCOVER RAW CSV FILES
# --------------------------------------------------------------------------------------------------

raw_csv_files = sorted(
    RAW_DATA_DIR.rglob("*.csv")
)

raw_csv_files = [
    path
    for path in raw_csv_files
    if path.is_file()
]

print(
    f"\nCSV files discovered under raw data directory: "
    f"{len(raw_csv_files)}"
)

for path in raw_csv_files:
    print(f"  - {path}")


# --------------------------------------------------------------------------------------------------
# 4.8 NORMALIZE FILENAMES
# --------------------------------------------------------------------------------------------------

def normalize_name(value):

    value = str(value).lower()

    value = re.sub(
        r"[^a-z0-9]+",
        "_",
        value
    )

    return value.strip("_")


normalized_raw_files = {
    path: normalize_name(path.stem)
    for path in raw_csv_files
}


# --------------------------------------------------------------------------------------------------
# 4.9 DATASET FILE PATTERNS
# --------------------------------------------------------------------------------------------------

DATASET_FILE_PATTERNS = {

    "adult_income": [
        "adult_income",
        "adult-income",
        "adult",
    ],

    "bank_marketing": [
        "bank_marketing",
        "bank-marketing",
        "bankmarketing",
        "bank",
    ],

    "diabetes_130us": [
        "diabetes_130us",
        "diabetes-130us",
        "diabetes",
    ],
}


# --------------------------------------------------------------------------------------------------
# 4.10 RESOLVE RAW DATASET PATHS
# --------------------------------------------------------------------------------------------------

RAW_DATASET_PATHS = {}

for dataset_id in DATASET_IDS:

    patterns = DATASET_FILE_PATTERNS[dataset_id]

    candidates = []

    for path, normalized_filename in normalized_raw_files.items():

        for pattern in patterns:

            normalized_pattern = normalize_name(pattern)

            if normalized_pattern in normalized_filename:

                candidates.append(path)
                break

    candidates = sorted(
        set(candidates)
    )

    print(
        f"\nCandidates for {dataset_id}:"
    )

    for candidate in candidates:
        print(f"  - {candidate}")

    if len(candidates) == 0:

        raise RuntimeError(
            f"No raw CSV file could be identified for "
            f"dataset '{dataset_id}'.\n"
            f"Raw data directory: {RAW_DATA_DIR}"
        )

    if len(candidates) == 1:

        selected_path = candidates[0]

    else:

        exact_matches = [
            path
            for path in candidates
            if normalize_name(path.stem)
            == normalize_name(dataset_id)
        ]

        if len(exact_matches) == 1:

            selected_path = exact_matches[0]

        else:

            raise RuntimeError(
                f"Multiple raw-file candidates were found for "
                f"dataset '{dataset_id}'.\n\n"
                + "\n".join(
                    f"  - {path}"
                    for path in candidates
                )
                + "\n\n"
                + "Notebook 02 will not guess between files."
            )

    RAW_DATASET_PATHS[dataset_id] = selected_path

    print(
        f"✓ Selected: {selected_path}"
    )


# --------------------------------------------------------------------------------------------------
# 4.11 LOAD RAW DATASETS WITH PARSING VALIDATION
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("4.11 LOAD RAW DATASETS WITH PARSING VALIDATION")
print("=" * 100)


def load_validated_csv(
    file_path,
    dataset_id,
    expected_columns,
):
    """
    Load a CSV using controlled delimiter detection.

    The loader first attempts the standard comma separator.
    If the result contains only one column, alternative delimiters
    are tested.

    The resulting dataframe must contain the expected number of columns.
    """

    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"Dataset file does not exist:\n{file_path}"
        )

    expected_n_columns = len(expected_columns)

    # ----------------------------------------------------------------------------------------------
    # Candidate separators
    # ----------------------------------------------------------------------------------------------

    candidate_separators = [
        ",",
        ";",
        "\t",
        "|",
    ]

    successful_attempts = []

    # ----------------------------------------------------------------------------------------------
    # Try candidate separators
    # ----------------------------------------------------------------------------------------------

    for separator in candidate_separators:

        try:

            df_candidate = pd.read_csv(
                file_path,
                sep=separator,
                low_memory=False
            )

            n_columns = len(df_candidate.columns)

            successful_attempts.append({
                "separator": separator,
                "columns": n_columns,
                "dataframe": df_candidate,
            })

        except Exception as exc:

            print(
                f"  Separator {repr(separator)} failed: "
                f"{repr(exc)}"
            )


    # ----------------------------------------------------------------------------------------------
    # Prefer exact expected column count
    # ----------------------------------------------------------------------------------------------

    exact_matches = [
        attempt
        for attempt in successful_attempts
        if attempt["columns"] == expected_n_columns
    ]


    if len(exact_matches) == 1:

        selected_attempt = exact_matches[0]

    elif len(exact_matches) > 1:

        # If multiple delimiters produce the same expected number
        # of columns, prefer comma, then semicolon.
        separator_priority = {
            ",": 0,
            ";": 1,
            "\t": 2,
            "|": 3,
        }

        exact_matches = sorted(
            exact_matches,
            key=lambda x: separator_priority.get(
                x["separator"],
                99
            )
        )

        selected_attempt = exact_matches[0]

    else:

        # No separator produced the expected dimensionality.
        diagnostics = "\n".join(
            [
                f"    separator={repr(a['separator'])} "
                f"-> {a['columns']} columns"
                for a in successful_attempts
            ]
        )

        raise RuntimeError(
            f"Could not correctly parse dataset '{dataset_id}'.\n\n"
            f"Expected columns : {expected_n_columns}\n"
            f"Expected names   : {expected_columns}\n\n"
            f"Parsing attempts:\n"
            f"{diagnostics}\n\n"
            f"File:\n{file_path}"
        )


    # ----------------------------------------------------------------------------------------------
    # Extract selected dataframe
    # ----------------------------------------------------------------------------------------------

    df = selected_attempt["dataframe"]
    selected_separator = selected_attempt["separator"]


    # ----------------------------------------------------------------------------------------------
    # Normalize column names
    # ----------------------------------------------------------------------------------------------

    df.columns = [
        str(column).strip()
        for column in df.columns
    ]


    # ----------------------------------------------------------------------------------------------
    # Validate exact column count
    # ----------------------------------------------------------------------------------------------

    if len(df.columns) != expected_n_columns:

        raise RuntimeError(
            f"Column-count validation failed for '{dataset_id}'.\n"
            f"Expected : {expected_n_columns}\n"
            f"Actual   : {len(df.columns)}\n"
            f"Separator: {repr(selected_separator)}"
        )


    # ----------------------------------------------------------------------------------------------
    # Validate expected columns when available
    # ----------------------------------------------------------------------------------------------

    missing_expected_columns = [
        column
        for column in expected_columns
        if column not in df.columns
    ]

    if missing_expected_columns:

        raise RuntimeError(
            f"Expected column names are missing for '{dataset_id}'.\n"
            f"Missing : {missing_expected_columns}\n"
            f"Actual  : {list(df.columns)}\n"
            f"Separator used: {repr(selected_separator)}"
        )


    # ----------------------------------------------------------------------------------------------
    # Return validated dataframe and parser information
    # ----------------------------------------------------------------------------------------------

    parser_info = {
        "separator": selected_separator,
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
    }

    return df, parser_info


# ==================================================================================================
# EXPECTED COLUMN DEFINITIONS
# ==================================================================================================

EXPECTED_COLUMNS = {

    "adult_income": [
        "age",
        "workclass",
        "fnlwgt",
        "education",
        "education_num",
        "marital_status",
        "occupation",
        "relationship",
        "race",
        "sex",
        "capital_gain",
        "capital_loss",
        "hours_per_week",
        "native_country",
        "income",
    ],

    "bank_marketing": [
        "age",
        "job",
        "marital",
        "education",
        "default",
        "balance",
        "housing",
        "loan",
        "contact",
        "day",
        "month",
        "duration",
        "campaign",
        "pdays",
        "previous",
        "poutcome",
        "y",
    ],

    "diabetes_130us": [
        "encounter_id",
        "patient_nbr",
        "race",
        "gender",
        "age",
        "weight",
        "admission_type_id",
        "discharge_disposition_id",
        "admission_source_id",
        "time_in_hospital",
        "payer_code",
        "medical_specialty",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "diag_1",
        "diag_2",
        "diag_3",
        "number_diagnoses",
        "max_glu_serum",
        "A1Cresult",
        "metformin",
        "repaglinide",
        "nateglinide",
        "chlorpropamide",
        "glimepiride",
        "acetohexamide",
        "glipizide",
        "glyburide",
        "tolbutamide",
        "pioglitazone",
        "rosiglitazone",
        "acarbose",
        "miglitol",
        "troglitazone",
        "tolazamide",
        "examide",
        "citoglipton",
        "insulin",
        "glyburide-metformin",
        "glipizide-metformin",
        "glimepiride-pioglitazone",
        "metformin-rosiglitazone",
        "metformin-pioglitazone",
        "change",
        "diabetesMed",
        "readmitted",
    ],
}


# ==================================================================================================
# LOAD ALL DATASETS
# ==================================================================================================

RAW_DATASETS = {}
DATASET_PARSING_INFO = {}

for dataset_id in DATASET_IDS:

    raw_path = RAW_DATASET_PATHS[dataset_id]

    print("\n" + "-" * 100)
    print(f"Dataset : {dataset_id}")
    print(f"File    : {raw_path}")

    df, parser_info = load_validated_csv(
        file_path=raw_path,
        dataset_id=dataset_id,
        expected_columns=EXPECTED_COLUMNS[dataset_id],
    )

    RAW_DATASETS[dataset_id] = df
    DATASET_PARSING_INFO[dataset_id] = parser_info

    print(
        f"✓ Separator detected : "
        f"{repr(parser_info['separator'])}"
    )

    print(
        f"✓ Shape              : "
        f"{parser_info['rows']:,} × "
        f"{parser_info['columns']}"
    )

    print(
        f"✓ Column schema       : validated"
    )


# ==================================================================================================
# FINAL PARSING SUMMARY
# ==================================================================================================

print("\n" + "=" * 100)
print("DATASET PARSING SUMMARY")
print("=" * 100)

for dataset_id in DATASET_IDS:

    info = DATASET_PARSING_INFO[dataset_id]

    print(
        f"{dataset_id:20s} | "
        f"separator={repr(info['separator']):6s} | "
        f"{info['rows']:>8,} rows × "
        f"{info['columns']:>2} columns"
    )

print("\n✓ All datasets loaded with validated parsing")


# --------------------------------------------------------------------------------------------------
# 4.12 EXPECTED DATASET SHAPES
# --------------------------------------------------------------------------------------------------

EXPECTED_SHAPES = {

    "adult_income": (
        48842,
        15,
    ),

    "bank_marketing": (
        45211,
        17,
    ),

    "diabetes_130us": (
        101766,
        50,
    ),
}


# --------------------------------------------------------------------------------------------------
# 4.13 VALIDATE DATASET SHAPES
# --------------------------------------------------------------------------------------------------

print("\nDataset shape validation:")

for dataset_id in DATASET_IDS:

    actual_shape = RAW_DATASETS[dataset_id].shape
    expected_shape = EXPECTED_SHAPES[dataset_id]

    if actual_shape != expected_shape:

        raise RuntimeError(
            f"Dataset shape mismatch.\n"
            f"Dataset  : {dataset_id}\n"
            f"Expected : {expected_shape}\n"
            f"Actual   : {actual_shape}\n\n"
            "Check the raw file and CSV parsing configuration "
            "before continuing."
        )

    print(
        f"  ✓ {dataset_id:20s} "
        f"{actual_shape[0]:>8,} × "
        f"{actual_shape[1]:>3}"
    )


# --------------------------------------------------------------------------------------------------
# 4.14 VALIDATE TARGET COLUMNS
# --------------------------------------------------------------------------------------------------

print("\nTarget-column validation:")

for dataset_id in DATASET_IDS:

    df = RAW_DATASETS[dataset_id]

    target_column = TARGET_COLUMNS[dataset_id]

    if target_column not in df.columns:

        raise RuntimeError(
            f"Target column not found.\n"
            f"Dataset : {dataset_id}\n"
            f"Target  : {target_column}\n"
            f"Columns : {list(df.columns)}"
        )

    unique_values = df[target_column].nunique(
        dropna=False
    )

    print(
        f"  ✓ {dataset_id:20s} "
        f"target={target_column} "
        f"| unique={unique_values}"
    )


# --------------------------------------------------------------------------------------------------
# 4.15 VALIDATE IDENTIFIER COLUMNS
# --------------------------------------------------------------------------------------------------

print("\nIdentifier-column validation:")

for dataset_id in DATASET_IDS:

    df = RAW_DATASETS[dataset_id]

    identifiers = IDENTIFIER_COLUMNS[dataset_id]

    missing_identifiers = [
        column
        for column in identifiers
        if column not in df.columns
    ]

    if missing_identifiers:

        raise RuntimeError(
            f"Identifier column(s) not found.\n"
            f"Dataset : {dataset_id}\n"
            f"Missing : {missing_identifiers}"
        )

    print(
        f"  ✓ {dataset_id:20s} "
        f"identifiers={identifiers}"
    )


# --------------------------------------------------------------------------------------------------
# 4.16 CONSTRUCT CANONICAL VALIDATED REGISTRY
# --------------------------------------------------------------------------------------------------

registry_records = []

for dataset_id in DATASET_IDS:

    df = RAW_DATASETS[dataset_id]

    registry_records.append({

        "dataset_id": dataset_id,

        "raw_path": str(
            RAW_DATASET_PATHS[dataset_id]
        ),

        "rows": int(
            df.shape[0]
        ),

        "columns": int(
            df.shape[1]
        ),

        "target_column": TARGET_COLUMNS[dataset_id],

        "identifier_columns": json.dumps(
            IDENTIFIER_COLUMNS[dataset_id]
        ),

        "validated_by_notebook_01": True,

        "reconstructed_by_notebook_02": True,
    })


CANONICAL_DATASET_REGISTRY_DF = pd.DataFrame(
    registry_records
)


# --------------------------------------------------------------------------------------------------
# 4.17 SAVE RECONSTRUCTED REGISTRY
# --------------------------------------------------------------------------------------------------

RECONSTRUCTED_REGISTRY_PATH = (
    RAW_VALIDATION_DIR
    / "validated_dataset_registry.csv"
)

CANONICAL_DATASET_REGISTRY_DF.to_csv(
    RECONSTRUCTED_REGISTRY_PATH,
    index=False
)

print(
    "\n✓ Reconstructed validated registry saved:"
)

print(
    f"  {RECONSTRUCTED_REGISTRY_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 4.18 RELOAD REGISTRY
# --------------------------------------------------------------------------------------------------

if not RECONSTRUCTED_REGISTRY_PATH.exists():

    raise RuntimeError(
        "The reconstructed validated registry was not saved successfully."
    )

RELOADED_REGISTRY = pd.read_csv(
    RECONSTRUCTED_REGISTRY_PATH
)

required_columns = [
    "dataset_id",
    "raw_path",
    "rows",
    "columns",
    "target_column",
    "identifier_columns",
]

missing_columns = [
    column
    for column in required_columns
    if column not in RELOADED_REGISTRY.columns
]

if missing_columns:

    raise RuntimeError(
        "Reconstructed registry is missing required columns:\n"
        f"{missing_columns}"
    )

print(
    "✓ Reconstructed registry reloaded successfully"
)

print(
    f"✓ Registry rows: {len(RELOADED_REGISTRY)}"
)


# --------------------------------------------------------------------------------------------------
# 4.19 FINAL RUNTIME VERIFICATION
# --------------------------------------------------------------------------------------------------

required_objects = [
    "DATASET_IDS",
    "RAW_DATASETS",
    "RAW_DATASET_PATHS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
    "CANONICAL_DATASET_REGISTRY_DF",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Required Section 4 runtime objects are missing:\n"
        f"{missing_objects}"
    )


# --------------------------------------------------------------------------------------------------
# 4.20 SECTION SUMMARY
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 4 VERIFICATION")
print("=" * 100)

print(
    f"Dataset count       : {len(DATASET_IDS)}"
)

print(
    f"Raw datasets loaded : {len(RAW_DATASETS)}"
)

print(
    f"Registry rows        : "
    f"{len(CANONICAL_DATASET_REGISTRY_DF)}"
)

print("\nValidated datasets:")

for dataset_id in DATASET_IDS:

    df = RAW_DATASETS[dataset_id]

    print(
        f"  {dataset_id:20s} | "
        f"{len(df):>8,} rows | "
        f"{len(df.columns):>3} columns | "
        f"target={TARGET_COLUMNS[dataset_id]}"
    )

print("\n" + "=" * 100)
print("✓ SECTION 4 — LOAD NOTEBOOK 01 VALIDATED REGISTRY : PASS")
print("=" * 100)

4. LOAD NOTEBOOK 01 VALIDATED REGISTRY
PROJECT_ROOT       : /content/drive/MyDrive/SPP_GAN_Research
RAW_DATA_DIR       : /content/drive/MyDrive/SPP_GAN_Research/data/raw
RAW_VALIDATION_DIR : /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation
✓ Project root verified
✓ Raw data directory verified
✓ Notebook 01 validation directory verified

Required datasets:
  - adult_income
  - bank_marketing
  - diabetes_130us

Target policy:
  adult_income         -> income
  bank_marketing       -> y
  diabetes_130us       -> readmitted

Identifier policy:
  adult_income         -> []
  bank_marketing       -> []
  diabetes_130us       -> ['encounter_id', 'patient_nbr']

Notebook 01 artifacts:
  ✓ Structural validation: /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/structural/structural_validation_report.csv
  ✓ Dataset validation: /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/structural/dataset_validation_report.csv
  ✓ Feature inventory: /content/drive

In [36]:
# ==================================================================================================
# 5. CREATE NOTEBOOK 02 DIRECTORIES
# ==================================================================================================

print("=" * 100)
print("5. CREATE NOTEBOOK 02 DIRECTORIES")
print("=" * 100)


NB02_DIRECTORIES = {

    "processed_root":
        PROJECT_ROOT / "data" / "processed" / "notebook_02",

    "native":
        PROJECT_ROOT / "data" / "processed" / "notebook_02" / "native",

    "encoded":
        PROJECT_ROOT / "data" / "processed" / "notebook_02" / "encoded",

    "splits":
        PROJECT_ROOT / "data" / "processed" / "notebook_02" / "splits",

    "preprocessors":
        PROJECT_ROOT / "data" / "processed" / "notebook_02" / "preprocessors",

    "schemas":
        PROJECT_ROOT / "data" / "processed" / "notebook_02" / "schemas",

    "feature_mapping":
        PROJECT_ROOT / "data" / "processed" / "notebook_02" / "feature_mapping",

    "manifests":
        PROJECT_ROOT / "results" / "raw_validation" / "notebook_02_manifests",
}


for directory in NB02_DIRECTORIES.values():

    Path(directory).mkdir(
        parents=True,
        exist_ok=True
    )


print()

for name, directory in NB02_DIRECTORIES.items():

    print(
        f"✓ {name:<20} → {directory}"
    )


print()
print("✓ Notebook 02 directories ready.")

5. CREATE NOTEBOOK 02 DIRECTORIES

✓ processed_root       → /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
✓ native               → /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native
✓ encoded              → /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/encoded
✓ splits               → /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/splits
✓ preprocessors        → /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/preprocessors
✓ schemas              → /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas
✓ feature_mapping      → /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/feature_mapping
✓ manifests            → /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/notebook_02_manifests

✓ Notebook 02 directories ready.


In [37]:
# ==================================================================================================
# 6. LOAD SPLIT CONFIGURATION
# ==================================================================================================

print("=" * 100)
print("6. LOAD SPLIT CONFIGURATION")
print("=" * 100)


SPLIT_CONFIG = {

    "train_size": 0.70,
    "validation_size": 0.15,
    "test_size": 0.15,

    "random_state": MASTER_SEED,

    "stratify": True,

    "shuffle": True,
}


# --------------------------------------------------------------------------------------------------
# Validate split configuration
# --------------------------------------------------------------------------------------------------

split_total = (
    SPLIT_CONFIG["train_size"]
    + SPLIT_CONFIG["validation_size"]
    + SPLIT_CONFIG["test_size"]
)


if not np.isclose(split_total, 1.0):

    raise ValueError(
        f"Train/validation/test proportions must sum to 1.0. "
        f"Current value: {split_total}"
    )


if SPLIT_CONFIG["train_size"] <= 0:
    raise ValueError("Training proportion must be > 0.")


if SPLIT_CONFIG["validation_size"] <= 0:
    raise ValueError("Validation proportion must be > 0.")


if SPLIT_CONFIG["test_size"] <= 0:
    raise ValueError("Test proportion must be > 0.")


print()
print(f"Train fraction      : {SPLIT_CONFIG['train_size']:.2f}")
print(f"Validation fraction : {SPLIT_CONFIG['validation_size']:.2f}")
print(f"Test fraction       : {SPLIT_CONFIG['test_size']:.2f}")
print(f"Random seed         : {SPLIT_CONFIG['random_state']}")
print(f"Stratification      : {SPLIT_CONFIG['stratify']}")
print(f"Shuffle             : {SPLIT_CONFIG['shuffle']}")

print()
print("✓ Split configuration validated.")

6. LOAD SPLIT CONFIGURATION

Train fraction      : 0.70
Validation fraction : 0.15
Test fraction       : 0.15
Random seed         : 2025
Stratification      : True
Shuffle             : True

✓ Split configuration validated.


In [38]:
# ==================================================================================================
# 7. DEFINE MISSING-VALUE POLICY
# ==================================================================================================

print("=" * 100)
print("7. DEFINE MISSING-VALUE POLICY")
print("=" * 100)


MISSING_VALUE_POLICY = {

    # Values that are universally interpreted as missing.
    "global_missing_markers": [
        "",
        "NA",
        "N/A",
        "na",
        "n/a",
        "NaN",
        "nan",
        "NULL",
        "null",
        "None",
        "none",
    ],

    # Dataset-specific missing markers.
    #
    # Adult Income uses '?' as an unknown/missing marker.
    # Diabetes 130-US uses '?' extensively for missing categorical fields.
    #
    # Bank Marketing's 'unknown' is retained as an observed category.
    "dataset_specific_markers": {

        "adult_income": [
            "?"
        ],

        "bank_marketing": [
            # Deliberately empty.
            # 'unknown' is treated as an observed category.
        ],

        "diabetes_130us": [
            "?"
        ],
    },

    # Missing-value policy after normalization.
    "numeric_imputation": "median",

    "categorical_imputation": "most_frequent",

    "unknown_categories": "preserve",

    "target_missing_policy": "fail",
}


print()
print("Global missing markers:")
print(
    MISSING_VALUE_POLICY["global_missing_markers"]
)

print()
print("Dataset-specific markers:")

for dataset_id, markers in (
    MISSING_VALUE_POLICY["dataset_specific_markers"]
).items():

    print(
        f"  {dataset_id:<20}: {markers}"
    )

print()
print(
    "✓ Missing-value policy defined."
)
print(
    "✓ Legitimate 'unknown' categories are preserved."
)

7. DEFINE MISSING-VALUE POLICY

Global missing markers:
['', 'NA', 'N/A', 'na', 'n/a', 'NaN', 'nan', 'NULL', 'null', 'None', 'none']

Dataset-specific markers:
  adult_income        : ['?']
  bank_marketing      : []
  diabetes_130us      : ['?']

✓ Missing-value policy defined.
✓ Legitimate 'unknown' categories are preserved.


In [39]:
# ==================================================================================================
# 8. DEFINE IDENTIFIER / TARGET POLICY
# ==================================================================================================

print("=" * 100)
print("8. DEFINE IDENTIFIER / TARGET POLICY")
print("=" * 100)


IDENTIFIER_POLICY = {

    # Explicit research identifiers.
    #
    # These are excluded from modeling/preprocessing features because they
    # represent record-level identity rather than useful generative structure.
    "explicit_identifiers": {

        "adult_income": [],

        "bank_marketing": [],

        "diabetes_130us": [
            "encounter_id",
            "patient_nbr",
        ],
    },

    # Target variables are retained as analytical variables.
    "retain_target": True,

    # Targets are used for stratification.
    "stratify_on_target": True,

    # Create an immutable internal row ID before splitting.
    "create_original_row_id": True,

    # Original row ID must never become a model feature.
    "exclude_original_row_id_from_model": True,

    # Do not automatically remove heuristic identifier candidates.
    "heuristic_identifiers_are_flags_only": True,
}


# --------------------------------------------------------------------------------------------------
# Validate explicit identifier columns
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    df = RAW_DATASETS[dataset_id]

    for column in IDENTIFIER_POLICY["explicit_identifiers"].get(
        dataset_id,
        []
    ):

        if column not in df.columns:

            raise ValueError(
                f"Configured identifier '{column}' not found in "
                f"dataset '{dataset_id}'."
            )


print()
for dataset_id in DATASET_IDS:

    print(
        f"{dataset_id:<20} "
        f"Target = {TARGET_COLUMNS[dataset_id]!r} | "
        f"Excluded identifiers = "
        f"{IDENTIFIER_POLICY['explicit_identifiers'].get(dataset_id, [])}"
    )


print()
print("✓ Identifier and target policy validated.")

8. DEFINE IDENTIFIER / TARGET POLICY

adult_income         Target = 'income' | Excluded identifiers = []
bank_marketing       Target = 'y' | Excluded identifiers = []
diabetes_130us       Target = 'readmitted' | Excluded identifiers = ['encounter_id', 'patient_nbr']

✓ Identifier and target policy validated.


In [40]:
# ==================================================================================================
# 9. NORMALIZE MISSING VALUES
# ==================================================================================================

print("=" * 100)
print("9. NORMALIZE MISSING VALUES")
print("=" * 100)


NORMALIZED_DATASETS = {}

MISSING_NORMALIZATION_REPORT = []


for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    source_df = RAW_DATASETS[dataset_id]

    # Copy so RAW_DATASETS remains untouched.
    df = source_df.copy(deep=True)

    markers = set(
        MISSING_VALUE_POLICY["global_missing_markers"]
        + MISSING_VALUE_POLICY["dataset_specific_markers"].get(
            dataset_id,
            []
        )
    )

    before_missing = int(
        df.isna().sum().sum()
    )

    # ----------------------------------------------------------------------------------------------
    # Normalize object/string columns only.
    # ----------------------------------------------------------------------------------------------

    for column in df.columns:

        if (
            pd.api.types.is_object_dtype(df[column])
            or pd.api.types.is_string_dtype(df[column])
        ):

            # Strip surrounding whitespace without converting legitimate
            # internal content.
            df[column] = df[column].map(
                lambda x: x.strip()
                if isinstance(x, str)
                else x
            )

            df[column] = df[column].replace(
                list(markers),
                np.nan
            )

    after_missing = int(
        df.isna().sum().sum()
    )

    newly_detected = after_missing - before_missing

    NORMALIZED_DATASETS[dataset_id] = df

    MISSING_NORMALIZATION_REPORT.append(
        {
            "dataset_id": dataset_id,
            "rows": len(df),
            "columns": len(df.columns),
            "missing_before": before_missing,
            "missing_after": after_missing,
            "newly_normalized_missing": newly_detected,
        }
    )

    print(
        f"Missing before : {before_missing:,}"
    )

    print(
        f"Missing after  : {after_missing:,}"
    )

    print(
        f"Newly normalized : {newly_detected:,}"
    )


MISSING_NORMALIZATION_DF = pd.DataFrame(
    MISSING_NORMALIZATION_REPORT
)


print()
print("✓ Missing-value markers normalized.")
print("✓ No imputation performed.")
print("✓ Original RAW_DATASETS remain unchanged.")

9. NORMALIZE MISSING VALUES

----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
Missing before : 6,465
Missing after  : 6,465
Newly normalized : 0

----------------------------------------------------------------------------------------------------
Dataset: bank_marketing
----------------------------------------------------------------------------------------------------
Missing before : 0
Missing after  : 0
Newly normalized : 0

----------------------------------------------------------------------------------------------------
Dataset: diabetes_130us
----------------------------------------------------------------------------------------------------
Missing before : 181,168
Missing after  : 374,017
Newly normalized : 192,849

✓ Missing-value markers normalized.
✓ No imputation performed.
✓ Original RAW_DATASETS re

In [41]:
# ==============================================================================
# SECTION 10 — VALIDATE FEATURE TYPES
# ==============================================================================

print("=" * 100)
print("10. VALIDATE FEATURE TYPES")
print("=" * 100)

import pandas as pd
import numpy as np


# ==============================================================================
# 10.1 — REQUIRED OBJECT VALIDATION
# ==============================================================================

REQUIRED_OBJECTS = [
    "RAW_DATASETS",
    "DATASET_IDS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
]

_missing_objects = [
    obj
    for obj in REQUIRED_OBJECTS
    if obj not in globals()
]

if _missing_objects:
    raise RuntimeError(
        "Required Notebook 02 objects are missing:\n"
        f"{_missing_objects}\n\n"
        "Please execute the preceding Notebook 02 sections first."
    )

print("[PASS] Required Notebook 02 objects are available.")


# ==============================================================================
# 10.2 — INITIALIZE FEATURE-TYPE CONTAINERS
# ==============================================================================

FEATURE_TYPE_REPORTS = {}
FEATURE_TYPE_SUMMARY = {}

print("[PASS] Feature-type validation containers initialized.")


# ==============================================================================
# 10.3 — DATASET-WISE FEATURE-TYPE VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    # --------------------------------------------------------------------------
    # Load Dataset
    # --------------------------------------------------------------------------

    if dataset_id not in RAW_DATASETS:
        raise RuntimeError(
            f"[FAIL] RAW_DATASETS does not contain '{dataset_id}'."
        )

    df = RAW_DATASETS[dataset_id]

    if not isinstance(df, pd.DataFrame):
        raise TypeError(
            f"[FAIL] RAW_DATASETS['{dataset_id}'] is not a pandas DataFrame."
        )

    # --------------------------------------------------------------------------
    # Retrieve Target and Identifier Configuration
    # --------------------------------------------------------------------------

    target_column = TARGET_COLUMNS.get(dataset_id)

    if target_column is None:
        raise RuntimeError(
            f"[FAIL] Target column is not defined for dataset '{dataset_id}'."
        )

    identifier_columns = IDENTIFIER_COLUMNS.get(dataset_id, [])

    # Ensure identifier configuration is a list
    if identifier_columns is None:
        identifier_columns = []

    identifier_columns = list(identifier_columns)

    # --------------------------------------------------------------------------
    # Validate Target Column
    # --------------------------------------------------------------------------

    if target_column not in df.columns:

        raise RuntimeError(
            f"[FAIL] Target column '{target_column}' "
            f"not found in dataset '{dataset_id}'.\n"
            f"Available columns:\n{list(df.columns)}"
        )

    # --------------------------------------------------------------------------
    # Validate Identifier Columns
    # --------------------------------------------------------------------------

    missing_identifiers = [
        column
        for column in identifier_columns
        if column not in df.columns
    ]

    if missing_identifiers:

        raise RuntimeError(
            f"[FAIL] Identifier columns missing from "
            f"'{dataset_id}': {missing_identifiers}"
        )

    # --------------------------------------------------------------------------
    # Detect Duplicate Column Names
    # --------------------------------------------------------------------------

    duplicate_columns = df.columns[
        df.columns.duplicated()
    ].tolist()

    if duplicate_columns:

        raise RuntimeError(
            f"[FAIL] Duplicate column names detected in "
            f"'{dataset_id}': {duplicate_columns}"
        )

    # --------------------------------------------------------------------------
    # Build Feature-Type Records
    # --------------------------------------------------------------------------

    records = []

    for column in df.columns:

        series = df[column]

        # ----------------------------------------------------------------------
        # Feature Role
        # ----------------------------------------------------------------------

        is_target = column == target_column

        is_identifier = column in identifier_columns

        # ----------------------------------------------------------------------
        # Raw Pandas Data Type
        # ----------------------------------------------------------------------

        pandas_dtype = str(series.dtype)

        # ----------------------------------------------------------------------
        # Semantic Feature Type
        # ----------------------------------------------------------------------
        #
        # Priority:
        #
        # 1. Identifier
        # 2. Target
        # 3. Numeric
        # 4. Categorical
        #
        # The target is intentionally treated as categorical because the
        # synthetic-data pipeline must preserve the joint feature-target
        # distribution and the targets in these datasets are categorical.
        #
        # Identifiers are retained only for audit purposes and excluded from
        # model-training / synthetic-data generation.
        # ----------------------------------------------------------------------

        if is_identifier:

            semantic_type = "identifier"

        elif is_target:

            semantic_type = "categorical"

        elif pd.api.types.is_bool_dtype(series):

            semantic_type = "categorical"

        elif pd.api.types.is_numeric_dtype(series):

            semantic_type = "numeric"

        else:

            semantic_type = "categorical"

        # ----------------------------------------------------------------------
        # Cardinality
        # ----------------------------------------------------------------------

        n_unique = int(
            series.nunique(dropna=True)
        )

        # ----------------------------------------------------------------------
        # Missingness
        # ----------------------------------------------------------------------

        n_missing = int(
            series.isna().sum()
        )

        missing_rate = (
            float(n_missing / len(series))
            if len(series) > 0
            else np.nan
        )

        # ----------------------------------------------------------------------
        # Record
        # ----------------------------------------------------------------------

        records.append({
            "dataset_id": dataset_id,
            "column": column,
            "pandas_dtype": pandas_dtype,
            "semantic_type": semantic_type,
            "is_target": bool(is_target),
            "is_identifier": bool(is_identifier),
            "n_unique": n_unique,
            "n_missing": n_missing,
            "missing_rate": missing_rate,
        })

    # ==============================================================================
    # 10.4 — CREATE DATASET FEATURE-TYPE REPORT
    # ==============================================================================

    feature_report = pd.DataFrame(records)

    FEATURE_TYPE_REPORTS[dataset_id] = feature_report

    # ==============================================================================
    # 10.5 — DERIVE FEATURE COUNTS
    # ==============================================================================

    # Modeling features exclude identifiers.
    modeling_features = feature_report[
        ~feature_report["is_identifier"].astype(bool)
    ].copy()

    numeric_features = modeling_features[
        modeling_features["semantic_type"] == "numeric"
    ].copy()

    categorical_features = modeling_features[
        modeling_features["semantic_type"] == "categorical"
    ].copy()

    identifier_features = feature_report[
        feature_report["is_identifier"].astype(bool)
    ].copy()

    target_features = feature_report[
        feature_report["is_target"].astype(bool)
    ].copy()

    # ==============================================================================
    # 10.6 — CREATE DATASET SUMMARY
    # ==============================================================================

    target_series = df[target_column]

    FEATURE_TYPE_SUMMARY[dataset_id] = {

        "rows":
            int(len(df)),

        "total_columns":
            int(len(df.columns)),

        "modeling_columns":
            int(len(modeling_features)),

        "numeric_columns":
            int(len(numeric_features)),

        "categorical_columns":
            int(len(categorical_features)),

        "identifier_columns":
            int(len(identifier_features)),

        "target_column":
            target_column,

        "target_dtype":
            str(target_series.dtype),

        "target_unique_values":
            int(target_series.nunique(dropna=True)),

        "target_missing_values":
            int(target_series.isna().sum()),

        "target_missing_rate":
            float(
                target_series.isna().mean()
            ),
    }

    # ==============================================================================
    # 10.7 — DISPLAY DATASET SUMMARY
    # ==============================================================================

    print(
        f"Rows                    : {len(df):,}"
    )

    print(
        f"Total columns           : {len(df.columns)}"
    )

    print(
        f"Modeling columns        : {len(modeling_features)}"
    )

    print(
        f"Numeric features        : {len(numeric_features)}"
    )

    print(
        f"Categorical features    : {len(categorical_features)}"
    )

    print(
        f"Identifier columns      : {len(identifier_features)}"
    )

    print(
        f"Target                  : {target_column}"
    )

    print(
        f"Target unique values    : "
        f"{target_series.nunique(dropna=True):,}"
    )

    print(
        f"Target missing values   : "
        f"{target_series.isna().sum():,}"
    )

    # ==============================================================================
    # 10.8 — VALIDATION GATES
    # ==============================================================================

    # --------------------------------------------------------------------------
    # Gate 1 — Number of records
    # --------------------------------------------------------------------------

    assert len(feature_report) == len(df.columns), (
        f"Feature report column count mismatch for {dataset_id}."
    )

    # --------------------------------------------------------------------------
    # Gate 2 — Every column must be represented
    # --------------------------------------------------------------------------

    assert set(feature_report["column"]) == set(df.columns), (
        f"Feature report does not contain exactly the columns "
        f"of {dataset_id}."
    )

    # --------------------------------------------------------------------------
    # Gate 3 — Target must exist
    # --------------------------------------------------------------------------

    assert target_column in feature_report["column"].values, (
        f"Target column '{target_column}' missing from feature report."
    )

    # --------------------------------------------------------------------------
    # Gate 4 — Exactly one target
    # --------------------------------------------------------------------------

    assert int(
        feature_report["is_target"].sum()
    ) == 1, (
        f"Expected exactly one target for {dataset_id}."
    )

    # --------------------------------------------------------------------------
    # Gate 5 — Target must be categorical
    # --------------------------------------------------------------------------

    target_row = feature_report[
        feature_report["column"] == target_column
    ].iloc[0]

    assert target_row["semantic_type"] == "categorical", (
        f"Target '{target_column}' must be classified as categorical."
    )

    # --------------------------------------------------------------------------
    # Gate 6 — Identifier count
    # --------------------------------------------------------------------------

    assert int(
        feature_report["is_identifier"].sum()
    ) == len(identifier_columns), (
        f"Identifier count mismatch for {dataset_id}."
    )

    # --------------------------------------------------------------------------
    # Gate 7 — Identifier classification
    # --------------------------------------------------------------------------

    for identifier in identifier_columns:

        identifier_row = feature_report[
            feature_report["column"] == identifier
        ].iloc[0]

        # IMPORTANT:
        # Use bool(...) rather than `is True` because Pandas may return
        # numpy.bool_ instead of the native Python bool object.
        assert bool(
            identifier_row["is_identifier"]
        ) is True, (
            f"Column '{identifier}' was not marked as an identifier."
        )

        assert identifier_row["semantic_type"] == "identifier", (
            f"Column '{identifier}' must have semantic type 'identifier'."
        )

    # --------------------------------------------------------------------------
    # Gate 8 — Identifiers must not be modeling features
    # --------------------------------------------------------------------------

    modeling_identifier_count = int(
        modeling_features["is_identifier"].astype(bool).sum()
    )

    assert modeling_identifier_count == 0, (
        f"Identifier leakage detected in modeling features "
        f"for {dataset_id}."
    )

    # --------------------------------------------------------------------------
    # Gate 9 — Every column must have a semantic type
    # --------------------------------------------------------------------------

    assert feature_report["semantic_type"].notna().all(), (
        f"Missing semantic feature type detected in {dataset_id}."
    )

    # --------------------------------------------------------------------------
    # Gate 10 — Only allowed semantic types
    # --------------------------------------------------------------------------

    allowed_semantic_types = {
        "numeric",
        "categorical",
        "identifier",
    }

    observed_semantic_types = set(
        feature_report["semantic_type"].dropna().unique()
    )

    assert observed_semantic_types.issubset(
        allowed_semantic_types
    ), (
        f"Unexpected semantic feature types in {dataset_id}: "
        f"{observed_semantic_types - allowed_semantic_types}"
    )

    # --------------------------------------------------------------------------
    # Gate 11 — Modeling feature accounting
    # --------------------------------------------------------------------------

    expected_modeling_columns = (
        len(df.columns) - len(identifier_columns)
    )

    assert len(modeling_features) == expected_modeling_columns, (
        f"Modeling feature count mismatch for {dataset_id}."
    )

    # --------------------------------------------------------------------------
    # Gate 12 — Numeric + categorical accounting
    # --------------------------------------------------------------------------

    assert (
        len(numeric_features) +
        len(categorical_features)
        ==
        len(modeling_features)
    ), (
        f"Numeric/categorical feature accounting mismatch "
        f"for {dataset_id}."
    )

    # --------------------------------------------------------------------------
    # Gate 13 — No column can simultaneously be target and identifier
    # --------------------------------------------------------------------------

    overlapping_roles = feature_report[
        feature_report["is_target"].astype(bool)
        &
        feature_report["is_identifier"].astype(bool)
    ]

    assert len(overlapping_roles) == 0, (
        f"Target/identifier role overlap detected in {dataset_id}: "
        f"{overlapping_roles['column'].tolist()}"
    )

    # --------------------------------------------------------------------------
    # Dataset validation passed
    # --------------------------------------------------------------------------

    print(
        "\n[PASS] Feature-type validation completed."
    )


# ==============================================================================
# 10.9 — COMBINE ALL FEATURE-TYPE REPORTS
# ==============================================================================

FEATURE_TYPE_DF = pd.concat(
    FEATURE_TYPE_REPORTS.values(),
    ignore_index=True
)


# ==============================================================================
# 10.10 — CREATE FEATURE-TYPE SUMMARY DATAFRAME
# ==============================================================================

FEATURE_TYPE_SUMMARY_DF = (
    pd.DataFrame.from_dict(
        FEATURE_TYPE_SUMMARY,
        orient="index"
    )
    .reset_index()
    .rename(
        columns={"index": "dataset_id"}
    )
)


# ==============================================================================
# 10.11 — GLOBAL VALIDATION
# ==============================================================================

print("\n" + "=" * 100)
print("GLOBAL FEATURE-TYPE VALIDATION")
print("=" * 100)

# --------------------------------------------------------------------------
# Expected datasets
# --------------------------------------------------------------------------

assert set(FEATURE_TYPE_REPORTS.keys()) == set(DATASET_IDS), (
    "Feature-type reports do not cover exactly the configured datasets."
)

# --------------------------------------------------------------------------
# Expected total feature records
# --------------------------------------------------------------------------

expected_total_columns = sum(
    len(RAW_DATASETS[dataset_id].columns)
    for dataset_id in DATASET_IDS
)

assert len(FEATURE_TYPE_DF) == expected_total_columns, (
    "Combined feature-type report has an unexpected number of rows."
)

# --------------------------------------------------------------------------
# Validate dataset IDs
# --------------------------------------------------------------------------

assert set(
    FEATURE_TYPE_DF["dataset_id"].unique()
) == set(DATASET_IDS), (
    "Combined feature-type report contains unexpected dataset IDs."
)

# --------------------------------------------------------------------------
# Validate unique dataset-column combinations
# --------------------------------------------------------------------------

duplicate_feature_records = FEATURE_TYPE_DF[
    FEATURE_TYPE_DF.duplicated(
        subset=["dataset_id", "column"],
        keep=False
    )
]

assert duplicate_feature_records.empty, (
    "Duplicate dataset-column records detected in FEATURE_TYPE_DF."
)

print(
    "[PASS] Global feature-type validation passed."
)


# ==============================================================================
# 10.12 — DISPLAY FINAL SUMMARY
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 10 SUMMARY")
print("=" * 100)

display(
    FEATURE_TYPE_SUMMARY_DF[
        [
            "dataset_id",
            "rows",
            "total_columns",
            "modeling_columns",
            "numeric_columns",
            "categorical_columns",
            "identifier_columns",
            "target_column",
            "target_dtype",
            "target_unique_values",
            "target_missing_values",
        ]
    ]
)


# ==============================================================================
# 10.13 — SEMANTIC TYPE DISTRIBUTION
# ==============================================================================

print("\n" + "-" * 100)
print("SEMANTIC FEATURE-TYPE DISTRIBUTION")
print("-" * 100)

semantic_distribution = (
    FEATURE_TYPE_DF
    .groupby(
        ["dataset_id", "semantic_type"]
    )
    .size()
    .unstack(fill_value=0)
)

display(semantic_distribution)


# ==============================================================================
# 10.14 — TARGET VALIDATION SUMMARY
# ==============================================================================

print("\n" + "-" * 100)
print("TARGET VALIDATION SUMMARY")
print("-" * 100)

target_validation_summary = (
    FEATURE_TYPE_DF[
        FEATURE_TYPE_DF["is_target"].astype(bool)
    ][
        [
            "dataset_id",
            "column",
            "pandas_dtype",
            "semantic_type",
            "n_unique",
            "n_missing",
            "missing_rate",
        ]
    ]
    .reset_index(drop=True)
)

display(target_validation_summary)


# ==============================================================================
# 10.15 — IDENTIFIER VALIDATION SUMMARY
# ==============================================================================

print("\n" + "-" * 100)
print("IDENTIFIER VALIDATION SUMMARY")
print("-" * 100)

identifier_validation_summary = (
    FEATURE_TYPE_DF[
        FEATURE_TYPE_DF["is_identifier"].astype(bool)
    ][
        [
            "dataset_id",
            "column",
            "pandas_dtype",
            "semantic_type",
            "n_unique",
            "n_missing",
            "missing_rate",
        ]
    ]
    .reset_index(drop=True)
)

if identifier_validation_summary.empty:

    print("No identifier columns are configured.")

else:

    display(identifier_validation_summary)


# ==============================================================================
# 10.16 — FINAL SECTION STATUS
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 10 COMPLETE")
print("=" * 100)

print(
    f"Datasets validated       : {len(DATASET_IDS)}"
)

print(
    f"Total feature records    : {len(FEATURE_TYPE_DF):,}"
)

print(
    f"Modeling feature records : "
    f"{int((~FEATURE_TYPE_DF['is_identifier'].astype(bool)).sum()):,}"
)

print(
    f"Identifier records       : "
    f"{int(FEATURE_TYPE_DF['is_identifier'].astype(bool).sum()):,}"
)

print(
    f"Target records           : "
    f"{int(FEATURE_TYPE_DF['is_target'].astype(bool).sum()):,}"
)

print("\n[PASS] SECTION 10 — VALIDATE FEATURE TYPES")
print("[PASS] No identifier leakage detected.")
print("[PASS] Target columns validated.")
print("[PASS] Numeric/categorical feature accounting validated.")
print("[PASS] Global feature-type validation passed.")

10. VALIDATE FEATURE TYPES
[PASS] Required Notebook 02 objects are available.
[PASS] Feature-type validation containers initialized.

----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
Rows                    : 48,842
Total columns           : 15
Modeling columns        : 15
Numeric features        : 6
Categorical features    : 9
Identifier columns      : 0
Target                  : income
Target unique values    : 2
Target missing values   : 0

[PASS] Feature-type validation completed.

----------------------------------------------------------------------------------------------------
Dataset: bank_marketing
----------------------------------------------------------------------------------------------------
Rows                    : 45,211
Total columns           : 17
Modeling columns        : 17
Numeric features 

,dataset_id,rows,total_columns,modeling_columns,numeric_columns,categorical_columns,identifier_columns,target_column,target_dtype,target_unique_values,target_missing_values
0,adult_income,48842,15,15,6,9,0,income,object,2,0
1,bank_marketing,45211,17,17,7,10,0,y,object,2,0
2,diabetes_130us,101766,50,48,11,37,2,readmitted,object,3,0



----------------------------------------------------------------------------------------------------
SEMANTIC FEATURE-TYPE DISTRIBUTION
----------------------------------------------------------------------------------------------------


semantic_type,categorical,identifier,numeric
dataset_id,,,
adult_income,9,0,6
bank_marketing,10,0,7
diabetes_130us,37,2,11



----------------------------------------------------------------------------------------------------
TARGET VALIDATION SUMMARY
----------------------------------------------------------------------------------------------------


,dataset_id,column,pandas_dtype,semantic_type,n_unique,n_missing,missing_rate
0,adult_income,income,object,categorical,2,0,0.0
1,bank_marketing,y,object,categorical,2,0,0.0
2,diabetes_130us,readmitted,object,categorical,3,0,0.0



----------------------------------------------------------------------------------------------------
IDENTIFIER VALIDATION SUMMARY
----------------------------------------------------------------------------------------------------


,dataset_id,column,pandas_dtype,semantic_type,n_unique,n_missing,missing_rate
0,diabetes_130us,encounter_id,int64,identifier,101766,0,0.0
1,diabetes_130us,patient_nbr,int64,identifier,71518,0,0.0



SECTION 10 COMPLETE
Datasets validated       : 3
Total feature records    : 82
Modeling feature records : 80
Identifier records       : 2
Target records           : 3

[PASS] SECTION 10 — VALIDATE FEATURE TYPES
[PASS] No identifier leakage detected.
[PASS] Target columns validated.
[PASS] Numeric/categorical feature accounting validated.
[PASS] Global feature-type validation passed.


In [42]:
# ==================================================================================================
# 11. VALIDATE TARGET
# ==================================================================================================

print("=" * 100)
print("11. VALIDATE TARGET")
print("=" * 100)


TARGET_VALIDATION_REPORT = []


for dataset_id in DATASET_IDS:

    df = NORMALIZED_DATASETS[dataset_id]

    target = TARGET_COLUMNS[dataset_id]

    # ----------------------------------------------------------------------------------------------
    # Existence
    # ----------------------------------------------------------------------------------------------

    if target not in df.columns:

        raise ValueError(
            f"Target '{target}' does not exist in '{dataset_id}'."
        )


    target_series = df[target]


    # ----------------------------------------------------------------------------------------------
    # Missing target
    # ----------------------------------------------------------------------------------------------

    missing_count = int(
        target_series.isna().sum()
    )

    if missing_count > 0:

        if (
            MISSING_VALUE_POLICY["target_missing_policy"]
            == "fail"
        ):

            raise ValueError(
                f"Target '{target}' in '{dataset_id}' contains "
                f"{missing_count:,} missing values."
            )


    # ----------------------------------------------------------------------------------------------
    # Unique values
    # ----------------------------------------------------------------------------------------------

    unique_values = (
        target_series
        .dropna()
        .unique()
        .tolist()
    )


    if len(unique_values) < 2:

        raise ValueError(
            f"Target '{target}' in '{dataset_id}' has fewer than "
            f"two observed classes."
        )


    TARGET_VALIDATION_REPORT.append(
        {
            "dataset_id": dataset_id,
            "target_column": target,
            "dtype": str(target_series.dtype),
            "n_rows": len(df),
            "missing_count": missing_count,
            "missing_rate": float(
                missing_count / len(df)
            ),
            "n_unique": len(unique_values),
            "target_classes": json.dumps(
                [str(x) for x in unique_values]
            ),
            "validation_status": "PASS",
        }
    )

    print()
    print(
        f"✓ {dataset_id:<20} "
        f"Target = {target!r} | "
        f"Classes = {len(unique_values)} | "
        f"Missing = {missing_count:,}"
    )


TARGET_VALIDATION_DF = pd.DataFrame(
    TARGET_VALIDATION_REPORT
)


print()
print("✓ All targets validated successfully.")

11. VALIDATE TARGET

✓ adult_income         Target = 'income' | Classes = 2 | Missing = 0

✓ bank_marketing       Target = 'y' | Classes = 2 | Missing = 0

✓ diabetes_130us       Target = 'readmitted' | Classes = 3 | Missing = 0

✓ All targets validated successfully.


In [43]:
# ==================================================================================================
# 12. CREATE ORIGINAL ROW IDS
# ==================================================================================================

print("=" * 100)
print("12. CREATE ORIGINAL ROW IDS")
print("=" * 100)


DATASETS_WITH_ROW_IDS = {}


for dataset_id in DATASET_IDS:

    df = NORMALIZED_DATASETS[dataset_id].copy(
        deep=True
    )

    # Preserve original row order explicitly.
    original_row_ids = np.arange(
        len(df),
        dtype=np.int64
    )

    # Internal provenance identifier.
    df.insert(
        0,
        "__original_row_id__",
        original_row_ids
    )

    DATASETS_WITH_ROW_IDS[dataset_id] = df

    print(
        f"✓ {dataset_id:<20} "
        f"{len(df):,} original row IDs created."
    )


print()
print("✓ Original row IDs created.")
print("✓ Original row IDs are provenance metadata only.")
print("✓ Original row IDs will not be used as model features.")

12. CREATE ORIGINAL ROW IDS
✓ adult_income         48,842 original row IDs created.
✓ bank_marketing       45,211 original row IDs created.
✓ diabetes_130us       101,766 original row IDs created.

✓ Original row IDs created.
✓ Original row IDs are provenance metadata only.
✓ Original row IDs will not be used as model features.


In [59]:
# ==================================================================================================
# 13. STRATIFIED TRAIN / VALIDATION / TEST SPLIT
# ==================================================================================================

print("=" * 100)
print("13. STRATIFIED TRAIN / VALIDATION / TEST SPLIT")
print("=" * 100)

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split


# ==================================================================================================
# 13.1 REQUIRED OBJECT VALIDATION
# ==================================================================================================

required_objects = [
    "DATASET_IDS",
    "RAW_DATASETS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 13 cannot proceed.\n"
        f"Missing required objects: {missing_objects}"
    )

print("✓ Required objects detected.")


# ==================================================================================================
# 13.2 SPLIT CONFIGURATION
# ==================================================================================================

TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15

if not np.isclose(
    TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION,
    1.0
):
    raise RuntimeError(
        "Train/validation/test fractions must sum to 1.0."
    )

# Use the canonical project seed.
if "RANDOM_SEED" in globals():
    SPLIT_RANDOM_SEED = int(RANDOM_SEED)
elif "MASTER_SEED" in globals():
    SPLIT_RANDOM_SEED = int(MASTER_SEED)
else:
    SPLIT_RANDOM_SEED = 2025

print(f"✓ Train fraction      : {TRAIN_FRACTION:.2f}")
print(f"✓ Validation fraction : {VALIDATION_FRACTION:.2f}")
print(f"✓ Test fraction       : {TEST_FRACTION:.2f}")
print(f"✓ Random seed         : {SPLIT_RANDOM_SEED}")


# ==================================================================================================
# 13.3 CANONICAL PROVENANCE COLUMN
# ==================================================================================================

PROVENANCE_COLUMN = "__original_row_id__"

print(f"✓ Provenance column   : {PROVENANCE_COLUMN}")


# ==================================================================================================
# 13.4 INITIALIZE OUTPUT CONTAINERS
# ==================================================================================================

TRAIN_DATASETS = {}
VALIDATION_DATASETS = {}
TEST_DATASETS = {}

SPLIT_MANIFESTS = {}
SPLIT_SUMMARY = []

print("✓ Split containers initialized.")


# ==================================================================================================
# 13.5 CREATE STRATIFIED SPLITS
# ==================================================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    # ----------------------------------------------------------------------------------------------
    # Load raw dataset
    # ----------------------------------------------------------------------------------------------

    if dataset_id not in RAW_DATASETS:
        raise RuntimeError(
            f"{dataset_id}: RAW_DATASETS entry not found."
        )

    df = RAW_DATASETS[dataset_id].copy()

    target = TARGET_COLUMNS[dataset_id]
    identifier_columns = IDENTIFIER_COLUMNS.get(dataset_id, [])

    print(f"Original rows            : {len(df):,}")
    print(f"Original columns         : {len(df.columns):,}")
    print(f"Target                   : {target}")
    print(f"Identifier columns       : {identifier_columns}")


    # ----------------------------------------------------------------------------------------------
    # Validate target
    # ----------------------------------------------------------------------------------------------

    if target not in df.columns:
        raise RuntimeError(
            f"{dataset_id}: target column '{target}' not found."
        )

    if df[target].isna().any():
        raise RuntimeError(
            f"{dataset_id}: target contains missing values. "
            "Stratified splitting cannot proceed safely."
        )


    # ----------------------------------------------------------------------------------------------
    # Validate identifiers
    # ----------------------------------------------------------------------------------------------

    missing_identifiers = [
        col for col in identifier_columns
        if col not in df.columns
    ]

    if missing_identifiers:
        raise RuntimeError(
            f"{dataset_id}: configured identifier columns not found: "
            f"{missing_identifiers}"
        )


    # ----------------------------------------------------------------------------------------------
    # Create deterministic original-row provenance ID
    #
    # IMPORTANT:
    # This is created BEFORE splitting so every observation retains its
    # exact relationship with the original raw dataset.
    # ----------------------------------------------------------------------------------------------

    if PROVENANCE_COLUMN in df.columns:
        raise RuntimeError(
            f"{dataset_id}: provenance column '{PROVENANCE_COLUMN}' "
            "already exists in the raw dataset."
        )

    df[PROVENANCE_COLUMN] = np.arange(
        len(df),
        dtype=np.int64
    )

    if not df[PROVENANCE_COLUMN].is_unique:
        raise RuntimeError(
            f"{dataset_id}: generated provenance IDs are not unique."
        )

    if df[PROVENANCE_COLUMN].isna().any():
        raise RuntimeError(
            f"{dataset_id}: generated provenance IDs contain missing values."
        )


    # ----------------------------------------------------------------------------------------------
    # First split:
    # 70% train
    # 30% temporary
    # ----------------------------------------------------------------------------------------------

    train_df, temp_df = train_test_split(
        df,
        test_size=(VALIDATION_FRACTION + TEST_FRACTION),
        random_state=SPLIT_RANDOM_SEED,
        stratify=df[target],
    )


    # ----------------------------------------------------------------------------------------------
    # Second split:
    # Split temporary 50/50 into validation and test.
    #
    # Because validation and test are both 15% of the original data,
    # each receives 50% of the temporary 30%.
    # ----------------------------------------------------------------------------------------------

    validation_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        random_state=SPLIT_RANDOM_SEED,
        stratify=temp_df[target],
    )


    # ----------------------------------------------------------------------------------------------
    # Reset indexes only.
    #
    # The original row provenance remains untouched.
    # ----------------------------------------------------------------------------------------------

    train_df = train_df.reset_index(drop=True)
    validation_df = validation_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)


    # ----------------------------------------------------------------------------------------------
    # Validate provenance
    # ----------------------------------------------------------------------------------------------

    for split_name, split_df in [
        ("train", train_df),
        ("validation", validation_df),
        ("test", test_df),
    ]:

        if PROVENANCE_COLUMN not in split_df.columns:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                f"provenance column '{PROVENANCE_COLUMN}' "
                "was not preserved."
            )

        if split_df[PROVENANCE_COLUMN].isna().any():
            raise RuntimeError(
                f"{dataset_id} | {split_name}: provenance contains NaN."
            )

        if not split_df[PROVENANCE_COLUMN].is_unique:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: provenance IDs "
                "are not unique."
            )


    # ----------------------------------------------------------------------------------------------
    # Validate split disjointness
    # ----------------------------------------------------------------------------------------------

    train_ids = set(train_df[PROVENANCE_COLUMN].tolist())
    validation_ids = set(validation_df[PROVENANCE_COLUMN].tolist())
    test_ids = set(test_df[PROVENANCE_COLUMN].tolist())

    if train_ids.intersection(validation_ids):
        raise RuntimeError(
            f"{dataset_id}: train and validation provenance overlap."
        )

    if train_ids.intersection(test_ids):
        raise RuntimeError(
            f"{dataset_id}: train and test provenance overlap."
        )

    if validation_ids.intersection(test_ids):
        raise RuntimeError(
            f"{dataset_id}: validation and test provenance overlap."
        )


    # ----------------------------------------------------------------------------------------------
    # Validate complete coverage
    # ----------------------------------------------------------------------------------------------

    combined_ids = (
        train_ids
        | validation_ids
        | test_ids
    )

    expected_ids = set(
        df[PROVENANCE_COLUMN].tolist()
    )

    if combined_ids != expected_ids:
        missing_ids = expected_ids - combined_ids
        extra_ids = combined_ids - expected_ids

        raise RuntimeError(
            f"{dataset_id}: split provenance coverage mismatch.\n"
            f"Missing IDs: {len(missing_ids)}\n"
            f"Extra IDs  : {len(extra_ids)}"
        )


    # ----------------------------------------------------------------------------------------------
    # Validate row counts
    # ----------------------------------------------------------------------------------------------

    total_rows = len(df)

    if (
        len(train_df)
        + len(validation_df)
        + len(test_df)
        != total_rows
    ):
        raise RuntimeError(
            f"{dataset_id}: split row counts do not sum to original rows."
        )


    # ----------------------------------------------------------------------------------------------
    # Validate target distribution
    # ----------------------------------------------------------------------------------------------

    train_target_distribution = (
        train_df[target]
        .value_counts(normalize=True, dropna=False)
        .sort_index()
    )

    validation_target_distribution = (
        validation_df[target]
        .value_counts(normalize=True, dropna=False)
        .sort_index()
    )

    test_target_distribution = (
        test_df[target]
        .value_counts(normalize=True, dropna=False)
        .sort_index()
    )

    all_target_classes = sorted(
        set(train_target_distribution.index)
        | set(validation_target_distribution.index)
        | set(test_target_distribution.index)
    )

    for class_value in all_target_classes:

        train_pct = float(
            train_target_distribution.get(class_value, 0.0)
        )

        validation_pct = float(
            validation_target_distribution.get(class_value, 0.0)
        )

        test_pct = float(
            test_target_distribution.get(class_value, 0.0)
        )

        if (
            train_pct == 0.0
            or validation_pct == 0.0
            or test_pct == 0.0
        ):
            raise RuntimeError(
                f"{dataset_id}: target class '{class_value}' "
                "is absent from at least one split."
            )


    # ----------------------------------------------------------------------------------------------
    # Store canonical split datasets
    # ----------------------------------------------------------------------------------------------

    TRAIN_DATASETS[dataset_id] = train_df
    VALIDATION_DATASETS[dataset_id] = validation_df
    TEST_DATASETS[dataset_id] = test_df


    # ----------------------------------------------------------------------------------------------
    # Create split manifest
    # ----------------------------------------------------------------------------------------------

    split_manifest = pd.concat(
        [
            train_df[[PROVENANCE_COLUMN]].assign(
                split="train"
            ),
            validation_df[[PROVENANCE_COLUMN]].assign(
                split="validation"
            ),
            test_df[[PROVENANCE_COLUMN]].assign(
                split="test"
            ),
        ],
        ignore_index=True,
    )

    if len(split_manifest) != total_rows:
        raise RuntimeError(
            f"{dataset_id}: split manifest row count mismatch."
        )

    if not split_manifest[PROVENANCE_COLUMN].is_unique:
        raise RuntimeError(
            f"{dataset_id}: split manifest provenance is not unique."
        )

    SPLIT_MANIFESTS[dataset_id] = split_manifest


    # ----------------------------------------------------------------------------------------------
    # Summary record
    # ----------------------------------------------------------------------------------------------

    SPLIT_SUMMARY.append(
        {
            "dataset_id": dataset_id,
            "original_rows": total_rows,
            "original_columns": len(df.columns) - 1,
            "train_rows": len(train_df),
            "validation_rows": len(validation_df),
            "test_rows": len(test_df),
            "train_fraction": len(train_df) / total_rows,
            "validation_fraction": len(validation_df) / total_rows,
            "test_fraction": len(test_df) / total_rows,
            "target_column": target,
            "identifier_count": len(identifier_columns),
            "provenance_column": PROVENANCE_COLUMN,
            "provenance_unique": True,
            "split_disjoint": True,
            "complete_coverage": True,
        }
    )


    # ----------------------------------------------------------------------------------------------
    # Display
    # ----------------------------------------------------------------------------------------------

    print(f"Train rows               : {len(train_df):,}")
    print(f"Validation rows          : {len(validation_df):,}")
    print(f"Test rows                : {len(test_df):,}")

    print(
        f"Train fraction           : "
        f"{len(train_df) / total_rows:.6f}"
    )

    print(
        f"Validation fraction      : "
        f"{len(validation_df) / total_rows:.6f}"
    )

    print(
        f"Test fraction            : "
        f"{len(test_df) / total_rows:.6f}"
    )

    print(
        f"Provenance column        : "
        f"{PROVENANCE_COLUMN}"
    )

    print(
        f"Provenance coverage      : "
        f"{len(combined_ids):,}/{total_rows:,}"
    )

    print("✓ Split provenance       : PASS")
    print("✓ Split disjointness     : PASS")
    print("✓ Complete coverage      : PASS")
    print("✓ Target stratification  : PASS")


# ==================================================================================================
# 13.6 SUMMARY DATAFRAME
# ==================================================================================================

SPLIT_SUMMARY_DF = pd.DataFrame(SPLIT_SUMMARY)


# ==================================================================================================
# 13.7 GLOBAL VALIDATION
# ==================================================================================================

if set(TRAIN_DATASETS.keys()) != set(DATASET_IDS):
    raise RuntimeError(
        "TRAIN_DATASETS dataset coverage mismatch."
    )

if set(VALIDATION_DATASETS.keys()) != set(DATASET_IDS):
    raise RuntimeError(
        "VALIDATION_DATASETS dataset coverage mismatch."
    )

if set(TEST_DATASETS.keys()) != set(DATASET_IDS):
    raise RuntimeError(
        "TEST_DATASETS dataset coverage mismatch."
    )

if set(SPLIT_MANIFESTS.keys()) != set(DATASET_IDS):
    raise RuntimeError(
        "SPLIT_MANIFESTS dataset coverage mismatch."
    )


# ==================================================================================================
# 13.8 FINAL OUTPUT
# ==================================================================================================

print("\n" + "=" * 100)
print("SECTION 13 COMPLETE")
print("=" * 100)

print("\nSplit Summary:")
display(SPLIT_SUMMARY_DF)

print("\nCanonical split objects created:")
print("  ✓ TRAIN_DATASETS")
print("  ✓ VALIDATION_DATASETS")
print("  ✓ TEST_DATASETS")
print("  ✓ SPLIT_MANIFESTS")
print("  ✓ SPLIT_SUMMARY_DF")

print("\nProvenance policy:")
print(f"  ✓ Canonical column: {PROVENANCE_COLUMN}")
print("  ✓ Created before splitting")
print("  ✓ Unique within each dataset")
print("  ✓ Train/validation/test mutually exclusive")
print("  ✓ Complete original-row coverage")

print("\nSTATUS: PASS")

13. STRATIFIED TRAIN / VALIDATION / TEST SPLIT
✓ Required objects detected.
✓ Train fraction      : 0.70
✓ Validation fraction : 0.15
✓ Test fraction       : 0.15
✓ Random seed         : 2025
✓ Provenance column   : __original_row_id__
✓ Split containers initialized.

----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
Original rows            : 48,842
Original columns         : 15
Target                   : income
Identifier columns       : []
Train rows               : 34,189
Validation rows          : 7,326
Test rows                : 7,327
Train fraction           : 0.699992
Validation fraction      : 0.149994
Test fraction            : 0.150014
Provenance column        : __original_row_id__
Provenance coverage      : 48,842/48,842
✓ Split provenance       : PASS
✓ Split disjointness     : PASS
✓ Complete coverage

,dataset_id,original_rows,original_columns,train_rows,validation_rows,test_rows,train_fraction,validation_fraction,test_fraction,target_column,identifier_count,provenance_column,provenance_unique,split_disjoint,complete_coverage
0,adult_income,48842,15,34189,7326,7327,0.699992,0.149994,0.150014,income,0,__original_row_id__,True,True,True
1,bank_marketing,45211,17,31647,6782,6782,0.699985,0.150008,0.150008,y,0,__original_row_id__,True,True,True
2,diabetes_130us,101766,50,71236,15265,15265,0.699998,0.150001,0.150001,readmitted,2,__original_row_id__,True,True,True



Canonical split objects created:
  ✓ TRAIN_DATASETS
  ✓ VALIDATION_DATASETS
  ✓ TEST_DATASETS
  ✓ SPLIT_MANIFESTS
  ✓ SPLIT_SUMMARY_DF

Provenance policy:
  ✓ Canonical column: __original_row_id__
  ✓ Created before splitting
  ✓ Unique within each dataset
  ✓ Train/validation/test mutually exclusive
  ✓ Complete original-row coverage

STATUS: PASS


In [60]:
# ==================================================================================================
# 14. VERIFY SPLIT INTEGRITY
# ==================================================================================================

print("=" * 100)
print("14. VERIFY SPLIT INTEGRITY")
print("=" * 100)


SPLIT_INTEGRITY_RESULTS = []


for dataset_id in DATASET_IDS:

    df = DATASETS_WITH_ROW_IDS[dataset_id]

    train_df = SPLIT_DATASETS[dataset_id]["train"]
    validation_df = SPLIT_DATASETS[dataset_id]["validation"]
    test_df = SPLIT_DATASETS[dataset_id]["test"]


    original_ids = set(
        df["__original_row_id__"]
    )

    train_ids = set(
        train_df["__original_row_id__"]
    )

    validation_ids = set(
        validation_df["__original_row_id__"]
    )

    test_ids = set(
        test_df["__original_row_id__"]
    )


    # ----------------------------------------------------------------------------------------------
    # No overlap
    # ----------------------------------------------------------------------------------------------

    train_validation_overlap = (
        train_ids & validation_ids
    )

    train_test_overlap = (
        train_ids & test_ids
    )

    validation_test_overlap = (
        validation_ids & test_ids
    )


    # ----------------------------------------------------------------------------------------------
    # Complete coverage
    # ----------------------------------------------------------------------------------------------

    combined_ids = (
        train_ids
        | validation_ids
        | test_ids
    )

    complete_coverage = (
        combined_ids == original_ids
    )


    no_overlap = (
        len(train_validation_overlap) == 0
        and len(train_test_overlap) == 0
        and len(validation_test_overlap) == 0
    )


    row_count_correct = (
        len(train_df)
        + len(validation_df)
        + len(test_df)
        == len(df)
    )


    integrity_pass = (
        no_overlap
        and complete_coverage
        and row_count_correct
    )


    SPLIT_INTEGRITY_RESULTS.append(
        {
            "dataset_id": dataset_id,
            "original_rows": len(df),
            "train_rows": len(train_df),
            "validation_rows": len(validation_df),
            "test_rows": len(test_df),
            "train_validation_overlap": len(
                train_validation_overlap
            ),
            "train_test_overlap": len(
                train_test_overlap
            ),
            "validation_test_overlap": len(
                validation_test_overlap
            ),
            "complete_coverage": complete_coverage,
            "row_count_correct": row_count_correct,
            "no_overlap": no_overlap,
            "integrity_status": (
                "PASS"
                if integrity_pass
                else "FAIL"
            ),
        }
    )


    print()
    print(
        f"{dataset_id:<20} "
        f"Integrity = "
        f"{'PASS' if integrity_pass else 'FAIL'}"
    )

    print(
        f"  Train/Validation overlap : "
        f"{len(train_validation_overlap)}"
    )

    print(
        f"  Train/Test overlap       : "
        f"{len(train_test_overlap)}"
    )

    print(
        f"  Validation/Test overlap  : "
        f"{len(validation_test_overlap)}"
    )

    print(
        f"  Complete coverage        : "
        f"{complete_coverage}"
    )


SPLIT_INTEGRITY_DF = pd.DataFrame(
    SPLIT_INTEGRITY_RESULTS
)


if not (
    SPLIT_INTEGRITY_DF["integrity_status"] == "PASS"
).all():

    raise RuntimeError(
        "One or more dataset splits failed integrity validation."
    )


print()
print("✓ All train/validation/test splits passed integrity validation.")

14. VERIFY SPLIT INTEGRITY

adult_income         Integrity = PASS
  Train/Validation overlap : 0
  Train/Test overlap       : 0
  Validation/Test overlap  : 0
  Complete coverage        : True

bank_marketing       Integrity = PASS
  Train/Validation overlap : 0
  Train/Test overlap       : 0
  Validation/Test overlap  : 0
  Complete coverage        : True

diabetes_130us       Integrity = PASS
  Train/Validation overlap : 0
  Train/Test overlap       : 0
  Validation/Test overlap  : 0
  Complete coverage        : True

✓ All train/validation/test splits passed integrity validation.


In [66]:
# ==================================================================================================
# 15. PREPARE TRAINING PREPROCESSING DATA
# ==================================================================================================

print("=" * 100)
print("15. PREPARE TRAINING PREPROCESSING DATA")
print("=" * 100)


# ==================================================================================================
# 15.1 REQUIRED OBJECT VALIDATION
# ==================================================================================================

required_objects = [
    "DATASET_IDS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
    "TRAIN_DATASETS",
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 15 cannot proceed.\n"
        f"Missing required objects: {missing_objects}"
    )

print("✓ Required objects detected.")


# ==================================================================================================
# 15.2 CANONICAL PROVENANCE COLUMN
# ==================================================================================================

PROVENANCE_COLUMN = "__original_row_id__"


# ==================================================================================================
# 15.3 INITIALIZE OUTPUT CONTAINERS
# ==================================================================================================

TRAIN_PREPROCESSING_DATA = {}
TRAIN_PREPROCESSING_COLUMNS = {}
TRAIN_PREPROCESSING_METADATA = {}

TRAIN_PREPROCESSING_SUMMARY = []

print("✓ Preprocessing containers initialized.")


# ==================================================================================================
# 15.4 BUILD TRAINING-ONLY MODELING DATA
# ==================================================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    train_df = TRAIN_DATASETS[dataset_id].copy()

    target = TARGET_COLUMNS[dataset_id]

    identifier_columns = IDENTIFIER_COLUMNS.get(
        dataset_id,
        []
    )

    # ----------------------------------------------------------------------------------------------
    # Validate provenance
    # ----------------------------------------------------------------------------------------------

    if PROVENANCE_COLUMN not in train_df.columns:
        raise RuntimeError(
            f"{dataset_id}: training dataset does not contain "
            f"required provenance column '{PROVENANCE_COLUMN}'."
        )


    # ----------------------------------------------------------------------------------------------
    # Determine canonical modeling columns
    #
    # CRITICAL:
    # Provenance is NEVER a modeling feature.
    # Explicit identifiers are NEVER modeling features.
    # Target IS retained as a modeling column.
    # ----------------------------------------------------------------------------------------------

    modeling_columns = [
        col
        for col in train_df.columns
        if col != PROVENANCE_COLUMN
        and col not in identifier_columns
    ]


    # ----------------------------------------------------------------------------------------------
    # Validate target
    # ----------------------------------------------------------------------------------------------

    if target not in modeling_columns:
        raise RuntimeError(
            f"{dataset_id}: target '{target}' is not present "
            "in canonical modeling columns."
        )


    # ----------------------------------------------------------------------------------------------
    # Validate identifier exclusion
    # ----------------------------------------------------------------------------------------------

    leaked_identifiers = [
        col
        for col in identifier_columns
        if col in modeling_columns
    ]

    if leaked_identifiers:
        raise RuntimeError(
            f"{dataset_id}: identifier leakage detected:\n"
            f"{leaked_identifiers}"
        )


    # ----------------------------------------------------------------------------------------------
    # Validate provenance exclusion
    # ----------------------------------------------------------------------------------------------

    if PROVENANCE_COLUMN in modeling_columns:
        raise RuntimeError(
            f"{dataset_id}: provenance column accidentally included "
            "in modeling columns."
        )


    # ----------------------------------------------------------------------------------------------
    # Validate duplicate columns
    # ----------------------------------------------------------------------------------------------

    if len(modeling_columns) != len(set(modeling_columns)):
        raise RuntimeError(
            f"{dataset_id}: duplicate modeling columns detected."
        )


    # ----------------------------------------------------------------------------------------------
    # Create training-only preprocessing DataFrame
    #
    # This is the ONLY data passed to the preprocessing pipeline.
    # ----------------------------------------------------------------------------------------------

    preprocessing_df = train_df[
        modeling_columns
    ].copy()


    # ----------------------------------------------------------------------------------------------
    # Determine feature types
    #
    # Use the semantic feature-type information generated earlier.
    # Target remains categorical.
    # ----------------------------------------------------------------------------------------------

    numeric_columns = []
    categorical_columns = []

    for column in modeling_columns:

        if column == target:
            categorical_columns.append(column)

        elif pd.api.types.is_bool_dtype(
            preprocessing_df[column]
        ):
            categorical_columns.append(column)

        elif pd.api.types.is_numeric_dtype(
            preprocessing_df[column]
        ):
            numeric_columns.append(column)

        else:
            categorical_columns.append(column)


    # ----------------------------------------------------------------------------------------------
    # Validate partition
    # ----------------------------------------------------------------------------------------------

    if set(numeric_columns).intersection(
        categorical_columns
    ):
        raise RuntimeError(
            f"{dataset_id}: numeric/categorical feature overlap detected."
        )

    if set(numeric_columns).union(
        categorical_columns
    ) != set(modeling_columns):
        raise RuntimeError(
            f"{dataset_id}: feature-type partition does not "
            "cover all modeling columns."
        )


    # ----------------------------------------------------------------------------------------------
    # Store canonical preprocessing data
    # ----------------------------------------------------------------------------------------------

    TRAIN_PREPROCESSING_DATA[
        dataset_id
    ] = preprocessing_df


    # ----------------------------------------------------------------------------------------------
    # Canonical preprocessing schema
    #
    # IMPORTANT:
    # `all_columns` means MODELING columns only.
    # It must NEVER contain provenance or explicit identifiers.
    # ----------------------------------------------------------------------------------------------

    TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ] = {
        "all_columns": modeling_columns,
        "numeric_columns": numeric_columns,
        "categorical_columns": categorical_columns,
        "target_column": target,
        "identifier_columns": identifier_columns,
        "provenance_column": PROVENANCE_COLUMN,
        "transformed_columns": [],
    }


    # ----------------------------------------------------------------------------------------------
    # Metadata
    # ----------------------------------------------------------------------------------------------

    TRAIN_PREPROCESSING_METADATA[
        dataset_id
    ] = {
        "dataset_id": dataset_id,
        "training_rows": int(len(preprocessing_df)),
        "input_columns": modeling_columns,
        "numeric_columns": numeric_columns,
        "categorical_columns": categorical_columns,
        "target_column": target,
        "identifier_columns": identifier_columns,
        "provenance_column": PROVENANCE_COLUMN,
        "provenance_excluded_from_modeling": True,
        "identifiers_excluded_from_modeling": True,
        "fit_dataset": "train_only",
    }


    # ----------------------------------------------------------------------------------------------
    # Summary
    # ----------------------------------------------------------------------------------------------

    TRAIN_PREPROCESSING_SUMMARY.append(
        {
            "dataset_id": dataset_id,
            "training_rows": len(preprocessing_df),
            "modeling_columns": len(modeling_columns),
            "numeric_columns": len(numeric_columns),
            "categorical_columns": len(categorical_columns),
            "target_column": target,
            "identifier_count": len(identifier_columns),
            "provenance_excluded": (
                PROVENANCE_COLUMN not in modeling_columns
            ),
        }
    )


    # ----------------------------------------------------------------------------------------------
    # Display
    # ----------------------------------------------------------------------------------------------

    print(f"Training rows             : {len(preprocessing_df):,}")
    print(f"Total modeling columns    : {len(modeling_columns):,}")
    print(f"Numeric columns           : {len(numeric_columns):,}")
    print(f"Categorical columns       : {len(categorical_columns):,}")
    print(f"Target                    : {target}")
    print(f"Identifiers excluded      : {identifier_columns}")
    print(
        f"Provenance excluded       : "
        f"{PROVENANCE_COLUMN not in modeling_columns}"
    )

    print("✓ Modeling schema          : PASS")
    print("✓ Provenance exclusion     : PASS")
    print("✓ Identifier exclusion     : PASS")
    print("✓ Feature-type partition   : PASS")


# ==================================================================================================
# 15.5 SUMMARY DATAFRAME
# ==================================================================================================

TRAIN_PREPROCESSING_SUMMARY_DF = pd.DataFrame(
    TRAIN_PREPROCESSING_SUMMARY
)


# ==================================================================================================
# 15.6 GLOBAL SCHEMA VALIDATION
# ==================================================================================================

for dataset_id in DATASET_IDS:

    schema = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]

    all_columns = schema["all_columns"]

    numeric_columns = schema["numeric_columns"]
    categorical_columns = schema["categorical_columns"]

    target = schema["target_column"]

    identifier_columns = schema["identifier_columns"]


    # No provenance
    if PROVENANCE_COLUMN in all_columns:
        raise RuntimeError(
            f"{dataset_id}: provenance leaked into modeling schema."
        )


    # No identifiers
    leaked_identifiers = [
        col
        for col in identifier_columns
        if col in all_columns
    ]

    if leaked_identifiers:
        raise RuntimeError(
            f"{dataset_id}: identifiers leaked into modeling schema: "
            f"{leaked_identifiers}"
        )


    # Target retained
    if target not in all_columns:
        raise RuntimeError(
            f"{dataset_id}: target missing from modeling schema."
        )


    # Complete feature partition
    if set(numeric_columns).union(
        categorical_columns
    ) != set(all_columns):
        raise RuntimeError(
            f"{dataset_id}: feature-type partition incomplete."
        )


    # No overlap
    if set(numeric_columns).intersection(
        categorical_columns
    ):
        raise RuntimeError(
            f"{dataset_id}: numeric/categorical overlap detected."
        )


    # Training preprocessing data must exactly match schema
    if list(
        TRAIN_PREPROCESSING_DATA[dataset_id].columns
    ) != all_columns:
        raise RuntimeError(
            f"{dataset_id}: preprocessing DataFrame does not "
            "match canonical modeling schema."
        )


print("\n✓ Global preprocessing schema validation : PASS")


# ==================================================================================================
# 15.7 FINAL SUMMARY
# ==================================================================================================

print("\n" + "=" * 100)
print("SECTION 15 COMPLETE")
print("=" * 100)

display(
    TRAIN_PREPROCESSING_SUMMARY_DF
)

print("\nCanonical objects:")
print("  ✓ TRAIN_PREPROCESSING_DATA")
print("  ✓ TRAIN_PREPROCESSING_COLUMNS")
print("  ✓ TRAIN_PREPROCESSING_METADATA")
print("  ✓ TRAIN_PREPROCESSING_SUMMARY_DF")

print("\nSchema policy:")
print("  ✓ Target retained")
print("  ✓ Explicit identifiers excluded")
print("  ✓ Provenance excluded from modeling")
print("  ✓ Provenance retained in split datasets")
print("  ✓ Preprocessing data contains modeling columns only")
print("  ✓ Training data only")

print("\nSTATUS: PASS")

15. PREPARE TRAINING PREPROCESSING DATA
✓ Required objects detected.
✓ Preprocessing containers initialized.

----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
Training rows             : 34,189
Total modeling columns    : 15
Numeric columns           : 6
Categorical columns       : 9
Target                    : income
Identifiers excluded      : []
Provenance excluded       : True
✓ Modeling schema          : PASS
✓ Provenance exclusion     : PASS
✓ Identifier exclusion     : PASS
✓ Feature-type partition   : PASS

----------------------------------------------------------------------------------------------------
Dataset: bank_marketing
----------------------------------------------------------------------------------------------------
Training rows             : 31,647
Total modeling columns    : 17
Numeric colu

,dataset_id,training_rows,modeling_columns,numeric_columns,categorical_columns,target_column,identifier_count,provenance_excluded
0,adult_income,34189,15,6,9,income,0,True
1,bank_marketing,31647,17,7,10,y,0,True
2,diabetes_130us,71236,48,11,37,readmitted,2,True



Canonical objects:
  ✓ TRAIN_PREPROCESSING_DATA
  ✓ TRAIN_PREPROCESSING_COLUMNS
  ✓ TRAIN_PREPROCESSING_METADATA
  ✓ TRAIN_PREPROCESSING_SUMMARY_DF

Schema policy:
  ✓ Target retained
  ✓ Explicit identifiers excluded
  ✓ Provenance excluded from modeling
  ✓ Provenance retained in split datasets
  ✓ Preprocessing data contains modeling columns only
  ✓ Training data only

STATUS: PASS


In [67]:
# ==============================================================================
# SECTION 16 — FIT TRAINING-ONLY PREPROCESSOR
# ==============================================================================

print("=" * 100)
print("16. FIT TRAINING-ONLY PREPROCESSOR")
print("=" * 100)

import os
import json
import pickle
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# ==============================================================================
# 16.1 — REQUIRED OBJECT VALIDATION
# ==============================================================================

REQUIRED_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_DATASETS",
    "TRAIN_PREPROCESSING_DATA",
    "TRAIN_PREPROCESSING_COLUMNS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
]

_missing_objects = [
    obj
    for obj in REQUIRED_OBJECTS
    if obj not in globals()
]

if _missing_objects:
    raise RuntimeError(
        "Required objects for Section 16 are missing:\n"
        f"{_missing_objects}\n\n"
        "Execute Sections 1–15 before running Section 16."
    )

print("[PASS] Required Section 16 objects are available.")


# ==============================================================================
# 16.2 — INITIALIZE PREPROCESSOR CONTAINERS
# ==============================================================================

TRAIN_PREPROCESSORS = {}
PREPROCESSOR_METADATA = {}

print(
    "[PASS] Preprocessor containers initialized."
)


# ==============================================================================
# 16.3 — ONE-HOT ENCODER COMPATIBILITY
# ==============================================================================

# ------------------------------------------------------------------------------
# scikit-learn versions differ in whether the parameter is named:
#
#     sparse_output=False
#
# or:
#
#     sparse=False
#
# Detect the installed version dynamically so the notebook remains
# Colab-compatible.
# ------------------------------------------------------------------------------

try:

    import inspect

    _OHE_PARAMETERS = inspect.signature(
        OneHotEncoder
    ).parameters

    if "sparse_output" in _OHE_PARAMETERS:

        OHE_CONFIG = {
            "handle_unknown": "ignore",
            "sparse_output": False,
            "dtype": np.float32,
        }

        OHE_SPARSE_PARAMETER = "sparse_output"

    else:

        OHE_CONFIG = {
            "handle_unknown": "ignore",
            "sparse": False,
            "dtype": np.float32,
        }

        OHE_SPARSE_PARAMETER = "sparse"

except Exception as exc:

    raise RuntimeError(
        "Unable to determine the installed scikit-learn "
        f"OneHotEncoder interface: {exc}"
    )

print(
    f"[INFO] OneHotEncoder configuration: "
    f"{OHE_SPARSE_PARAMETER}=False, "
    f"handle_unknown='ignore'"
)


# ==============================================================================
# 16.4 — DATASET-WISE TRAINING-ONLY FITTING
# ==============================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    # ==============================================================================
    # 16.4.1 — RETRIEVE TRAINING DATA
    # ==============================================================================

    if dataset_id not in TRAIN_DATASETS:

        raise RuntimeError(
            f"[FAIL] Training dataset '{dataset_id}' "
            "is not available."
        )

    train_df = TRAIN_DATASETS[dataset_id].copy()

    if not isinstance(train_df, pd.DataFrame):

        raise TypeError(
            f"[FAIL] TRAIN_DATASETS['{dataset_id}'] "
            "is not a pandas DataFrame."
        )

    # ==============================================================================
    # 16.4.2 — RETRIEVE PREPROCESSING CONFIGURATION
    # ==============================================================================

    if dataset_id not in TRAIN_PREPROCESSING_COLUMNS:

        raise RuntimeError(
            f"[FAIL] Training preprocessing configuration for "
            f"'{dataset_id}' is unavailable.\n"
            "Run Section 15 first."
        )

    preprocessing_config = (
        TRAIN_PREPROCESSING_COLUMNS[dataset_id]
    )

    # ------------------------------------------------------------------------------
    # IMPORTANT:
    #
    # Section 15 uses:
    #
    #     all_columns
    #     numeric_columns
    #     categorical_columns
    #     target_column
    #     identifier_columns
    #
    # Section 16 consumes exactly those fields.
    # ------------------------------------------------------------------------------

    model_columns = list(
        preprocessing_config["all_columns"]
    )

    numeric_columns = list(
        preprocessing_config["numeric_columns"]
    )

    categorical_columns = list(
        preprocessing_config["categorical_columns"]
    )

    target_column = preprocessing_config[
        "target_column"
    ]

    identifier_columns = list(
        preprocessing_config["identifier_columns"]
    )

    # ==============================================================================
    # 16.4.3 — VALIDATE CONFIGURATION
    # ==============================================================================

    if target_column not in model_columns:

        raise RuntimeError(
            f"[FAIL] Target '{target_column}' is missing "
            f"from model columns for '{dataset_id}'."
        )

    leaked_identifiers = [
        column
        for column in identifier_columns
        if column in model_columns
    ]

    if leaked_identifiers:

        raise RuntimeError(
            f"[FAIL] Identifier leakage detected in model columns "
            f"for '{dataset_id}': {leaked_identifiers}"
        )

    missing_model_columns = [
        column
        for column in model_columns
        if column not in train_df.columns
    ]

    if missing_model_columns:

        raise RuntimeError(
            f"[FAIL] Model columns missing from training data "
            f"for '{dataset_id}': {missing_model_columns}"
        )

    # ==============================================================================
    # 16.4.4 — SELECT TRAINING-ONLY MODEL DATA
    # ==============================================================================

    X_train = train_df[
        model_columns
    ].copy()

    # ------------------------------------------------------------------------------
    # Target is included in the synthetic-data representation.
    #
    # Therefore the target is processed as a categorical feature rather than
    # being removed at this stage.
    # ------------------------------------------------------------------------------

    # ==============================================================================
    # 16.4.5 — VERIFY NUMERIC FEATURES
    # ==============================================================================

    numeric_missing_from_data = [
        column
        for column in numeric_columns
        if column not in X_train.columns
    ]

    if numeric_missing_from_data:

        raise RuntimeError(
            f"[FAIL] Numeric columns missing from training data "
            f"for '{dataset_id}': {numeric_missing_from_data}"
        )

    # ==============================================================================
    # 16.4.6 — VERIFY CATEGORICAL FEATURES
    # ==============================================================================

    categorical_missing_from_data = [
        column
        for column in categorical_columns
        if column not in X_train.columns
    ]

    if categorical_missing_from_data:

        raise RuntimeError(
            f"[FAIL] Categorical columns missing from training data "
            f"for '{dataset_id}': {categorical_missing_from_data}"
        )

    # ==============================================================================
    # 16.4.7 — VERIFY FEATURE ACCOUNTING
    # ==============================================================================

    if (
        len(numeric_columns)
        +
        len(categorical_columns)
        !=
        len(model_columns)
    ):

        raise RuntimeError(
            f"[FAIL] Numeric/categorical feature accounting mismatch "
            f"for '{dataset_id}'.\n"
            f"Model columns      : {len(model_columns)}\n"
            f"Numeric columns    : {len(numeric_columns)}\n"
            f"Categorical columns: {len(categorical_columns)}"
        )

    # ==============================================================================
    # 16.4.8 — VERIFY NO OVERLAP
    # ==============================================================================

    feature_overlap = (
        set(numeric_columns)
        &
        set(categorical_columns)
    )

    if feature_overlap:

        raise RuntimeError(
            f"[FAIL] Feature-type overlap detected for "
            f"'{dataset_id}': {sorted(feature_overlap)}"
        )

    # ==============================================================================
    # 16.4.9 — CREATE NUMERIC PIPELINE
    # ==============================================================================

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                ),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    # ==============================================================================
    # 16.4.10 — CREATE CATEGORICAL PIPELINE
    # ==============================================================================

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                ),
            ),
            (
                "encoder",
                OneHotEncoder(
                    **OHE_CONFIG
                ),
            ),
        ]
    )

    # ==============================================================================
    # 16.4.11 — CREATE COLUMN TRANSFORMER
    # ==============================================================================

    transformers = []

    # --------------------------------------------------------------------------
    # Numeric transformer
    # --------------------------------------------------------------------------

    if numeric_columns:

        transformers.append(
            (
                "numeric",
                numeric_pipeline,
                numeric_columns,
            )
        )

    # --------------------------------------------------------------------------
    # Categorical transformer
    # --------------------------------------------------------------------------

    if categorical_columns:

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_columns,
            )
        )

    if not transformers:

        raise RuntimeError(
            f"[FAIL] No preprocessing transformers were created "
            f"for '{dataset_id}'."
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )

    # ==============================================================================
    # 16.4.12 — FIT PREPROCESSOR USING TRAINING DATA ONLY
    # ==============================================================================

    print(
        f"Fitting preprocessor on {len(X_train):,} training rows..."
    )

    with warnings.catch_warnings():

        warnings.simplefilter("ignore")

        preprocessor.fit(
            X_train
        )

    print(
        "[PASS] Preprocessor fitted using training data only."
    )

    # ==============================================================================
    # 16.4.13 — DETERMINE OUTPUT FEATURE COUNT
    # ==============================================================================

    try:

        transformed_feature_names = (
            preprocessor
            .get_feature_names_out()
            .tolist()
        )

    except Exception as exc:

        raise RuntimeError(
            f"[FAIL] Unable to obtain transformed feature names "
            f"for '{dataset_id}': {exc}"
        )

    transformed_feature_count = len(
        transformed_feature_names
    )

    if transformed_feature_count == 0:

        raise RuntimeError(
            f"[FAIL] Preprocessor produced zero transformed features "
            f"for '{dataset_id}'."
        )

    # ==============================================================================
    # 16.4.14 — STORE PREPROCESSOR
    # ==============================================================================

    TRAIN_PREPROCESSORS[
        dataset_id
    ] = preprocessor

    # ==============================================================================
    # 16.4.15 — STORE METADATA
    # ==============================================================================

    TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]["transformed_columns"] = (
        transformed_feature_names
    )

    PREPROCESSOR_METADATA[
        dataset_id
    ] = {

        "dataset_id":
            dataset_id,

        "training_rows":
            int(len(X_train)),

        "input_columns":
            int(len(model_columns)),

        "numeric_input_columns":
            int(len(numeric_columns)),

        "categorical_input_columns":
            int(len(categorical_columns)),

        "identifier_columns_excluded":
            int(len(identifier_columns)),

        "target_column":
            target_column,

        "transformed_columns":
            int(transformed_feature_count),

        "numeric_imputation":
            "median",

        "numeric_scaling":
            "StandardScaler",

        "categorical_imputation":
            "most_frequent",

        "categorical_encoding":
            "OneHotEncoder",

        "handle_unknown":
            "ignore",

        "fit_dataset":
            "training_only",
    }

    # ==============================================================================
    # 16.4.16 — VALIDATE FITTED PREPROCESSOR
    # ==============================================================================

    if not hasattr(
        preprocessor,
        "transformers_"
    ):

        raise RuntimeError(
            f"[FAIL] Preprocessor for '{dataset_id}' "
            "does not appear to be fitted."
        )

    # ==============================================================================
    # 16.4.17 — TEST TRANSFORMATION ON TRAINING DATA
    # ==============================================================================

    with warnings.catch_warnings():

        warnings.simplefilter("ignore")

        X_train_encoded = (
            preprocessor.transform(
                X_train
            )
        )

    # --------------------------------------------------------------------------
    # Convert to dense ndarray if necessary
    # --------------------------------------------------------------------------

    if hasattr(
        X_train_encoded,
        "toarray"
    ):

        X_train_encoded = (
            X_train_encoded.toarray()
        )

    X_train_encoded = np.asarray(
        X_train_encoded,
        dtype=np.float32
    )

    # --------------------------------------------------------------------------
    # Verify transformed shape
    # --------------------------------------------------------------------------

    if X_train_encoded.shape[0] != len(X_train):

        raise RuntimeError(
            f"[FAIL] Transformed row count mismatch for "
            f"'{dataset_id}'."
        )

    if X_train_encoded.shape[1] != transformed_feature_count:

        raise RuntimeError(
            f"[FAIL] Transformed feature count mismatch for "
            f"'{dataset_id}'."
        )

    # --------------------------------------------------------------------------
    # Verify finite values
    # --------------------------------------------------------------------------

    if not np.isfinite(
        X_train_encoded
    ).all():

        raise RuntimeError(
            f"[FAIL] Non-finite values detected in transformed "
            f"training data for '{dataset_id}'."
        )

    print(
        f"Transformed training shape: "
        f"{X_train_encoded.shape[0]:,} × "
        f"{X_train_encoded.shape[1]:,}"
    )

    # ==============================================================================
    # 16.4.18 — DISPLAY DATASET SUMMARY
    # ==============================================================================

    print(
        f"Input columns             : "
        f"{len(model_columns)}"
    )

    print(
        f"Numeric columns           : "
        f"{len(numeric_columns)}"
    )

    print(
        f"Categorical columns       : "
        f"{len(categorical_columns)}"
    )

    print(
        f"Identifiers excluded      : "
        f"{len(identifier_columns)}"
    )

    print(
        f"Target                    : "
        f"{target_column}"
    )

    print(
        f"Encoded output columns    : "
        f"{transformed_feature_count:,}"
    )

    print(
        f"Encoded dtype             : "
        f"{X_train_encoded.dtype}"
    )

    print(
        f"Encoded memory            : "
        f"{X_train_encoded.nbytes / (1024 ** 2):.2f} MB"
    )

    print(
        "\n[PASS] Dataset preprocessor validated."
    )


# ==============================================================================
# 16.5 — GLOBAL PREPROCESSOR VALIDATION
# ==============================================================================

print("\n" + "=" * 100)
print("GLOBAL PREPROCESSOR VALIDATION")
print("=" * 100)

# --------------------------------------------------------------------------
# All datasets have preprocessors
# --------------------------------------------------------------------------

assert set(
    TRAIN_PREPROCESSORS.keys()
) == set(DATASET_IDS), (
    "Not all configured datasets have fitted preprocessors."
)

# --------------------------------------------------------------------------
# All preprocessors must be fitted
# --------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    preprocessor = (
        TRAIN_PREPROCESSORS[dataset_id]
    )

    assert hasattr(
        preprocessor,
        "transformers_"
    ), (
        f"Preprocessor for {dataset_id} is not fitted."
    )

# --------------------------------------------------------------------------
# Metadata coverage
# --------------------------------------------------------------------------

assert set(
    PREPROCESSOR_METADATA.keys()
) == set(DATASET_IDS), (
    "Preprocessor metadata does not cover all datasets."
)

# --------------------------------------------------------------------------
# No identifiers in model columns
# --------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    identifiers = IDENTIFIER_COLUMNS.get(
        dataset_id,
        []
    )

    model_columns = (
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]["all_columns"]
    )

    leaked = [
        column
        for column in identifiers
        if column in model_columns
    ]

    assert not leaked, (
        f"Identifier leakage detected for {dataset_id}: {leaked}"
    )

print(
    "[PASS] All training-only preprocessors validated."
)


# ==============================================================================
# 16.6 — CREATE PREPROCESSOR SUMMARY DATAFRAME
# ==============================================================================

PREPROCESSOR_SUMMARY_DF = pd.DataFrame(
    PREPROCESSOR_METADATA.values()
)


# ==============================================================================
# 16.7 — DISPLAY SUMMARY
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 16 PREPROCESSOR SUMMARY")
print("=" * 100)

display(
    PREPROCESSOR_SUMMARY_DF[
        [
            "dataset_id",
            "training_rows",
            "input_columns",
            "numeric_input_columns",
            "categorical_input_columns",
            "identifier_columns_excluded",
            "target_column",
            "transformed_columns",
            "numeric_imputation",
            "numeric_scaling",
            "categorical_imputation",
            "categorical_encoding",
            "handle_unknown",
            "fit_dataset",
        ]
    ]
)


# ==============================================================================
# 16.8 — FINAL STATUS
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 16 COMPLETE")
print("=" * 100)

print(
    f"Datasets processed         : {len(DATASET_IDS)}"
)

print(
    f"Fitted preprocessors       : "
    f"{len(TRAIN_PREPROCESSORS)}"
)

print(
    "\n[PASS] SECTION 16 — FIT TRAINING-ONLY PREPROCESSOR"
)

print(
    "[PASS] Preprocessors fitted using training data only."
)

print(
    "[PASS] Numeric median imputation configured."
)

print(
    "[PASS] Numeric StandardScaler configured."
)

print(
    "[PASS] Categorical most-frequent imputation configured."
)

print(
    "[PASS] One-hot encoding configured with unknown-category handling."
)

print(
    "[PASS] Identifier columns excluded from modeling."
)

print(
    "[PASS] Target column retained as categorical."
)

print(
    "[PASS] Training transformations validated."
)

16. FIT TRAINING-ONLY PREPROCESSOR
[PASS] Required Section 16 objects are available.
[PASS] Preprocessor containers initialized.
[INFO] OneHotEncoder configuration: sparse_output=False, handle_unknown='ignore'

----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
Fitting preprocessor on 34,189 training rows...
[PASS] Preprocessor fitted using training data only.
Transformed training shape: 34,189 × 107
Input columns             : 15
Numeric columns           : 6
Categorical columns       : 9
Identifiers excluded      : 0
Target                    : income
Encoded output columns    : 107
Encoded dtype             : float32
Encoded memory            : 13.96 MB

[PASS] Dataset preprocessor validated.

----------------------------------------------------------------------------------------------------
Dataset: bank_market

,dataset_id,training_rows,input_columns,numeric_input_columns,categorical_input_columns,identifier_columns_excluded,target_column,transformed_columns,numeric_imputation,numeric_scaling,categorical_imputation,categorical_encoding,handle_unknown,fit_dataset
0,adult_income,34189,15,6,9,0,income,107,median,StandardScaler,most_frequent,OneHotEncoder,ignore,training_only
1,bank_marketing,31647,17,7,10,0,y,53,median,StandardScaler,most_frequent,OneHotEncoder,ignore,training_only
2,diabetes_130us,71236,48,11,37,2,readmitted,2339,median,StandardScaler,most_frequent,OneHotEncoder,ignore,training_only



SECTION 16 COMPLETE
Datasets processed         : 3
Fitted preprocessors       : 3

[PASS] SECTION 16 — FIT TRAINING-ONLY PREPROCESSOR
[PASS] Preprocessors fitted using training data only.
[PASS] Numeric median imputation configured.
[PASS] Numeric StandardScaler configured.
[PASS] Categorical most-frequent imputation configured.
[PASS] One-hot encoding configured with unknown-category handling.
[PASS] Identifier columns excluded from modeling.
[PASS] Target column retained as categorical.
[PASS] Training transformations validated.


In [68]:
# ==============================================================================
# SECTION 17 — TRANSFORM TRAIN / VALIDATION / TEST
# ==============================================================================
#
# PURPOSE
# -------
# Transform TRAIN / VALIDATION / TEST datasets using the SAME preprocessor
# fitted exclusively on TRAIN in Section 16.
#
# IMPORTANT DATA-PROTECTION RULE
# ------------------------------
# __original_row_id is an AUDIT / REPRODUCIBILITY column.
#
# It must NOT be passed to the sklearn preprocessor.
#
# Explicit identifiers such as:
#   encounter_id
#   patient_nbr
#
# are also excluded from modeling data according to the canonical
# IDENTIFIER_COLUMNS policy.
#
# Therefore:
#
#   Split dataframe
#          |
#          +--> __original_row_id      -> audit only
#          |
#          +--> identifier columns    -> excluded from modeling
#          |
#          +--> canonical all_columns -> passed to preprocessor
#
# NO preprocessing is fitted in this section.
#
# INPUTS
# ------
# DATASET_IDS
# TRAIN_DATASETS
# VALIDATION_DATASETS
# TEST_DATASETS
# TRAIN_PREPROCESSORS
# TRAIN_PREPROCESSING_COLUMNS
#
# OUTPUTS
# -------
# TRANSFORMED_TRAIN_DATASETS
# TRANSFORMED_VALIDATION_DATASETS
# TRANSFORMED_TEST_DATASETS
# TRANSFORMED_DATASET_SUMMARY
# TRANSFORMED_DATASET_SUMMARY_DF
#
# ==============================================================================

print("=" * 100)
print("SECTION 17 — TRANSFORM TRAIN / VALIDATION / TEST")
print("=" * 100)


# ==============================================================================
# 17.1 — REQUIRED OBJECT CHECKS
# ==============================================================================

required_objects = [
    "DATASET_IDS",
    "TRAIN_DATASETS",
    "VALIDATION_DATASETS",
    "TEST_DATASETS",
    "TRAIN_PREPROCESSORS",
    "TRAIN_PREPROCESSING_COLUMNS",
]

missing_objects = [
    obj
    for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise RuntimeError(
        "SECTION 17 CANNOT START.\n"
        "The following required objects are missing:\n"
        + "\n".join(f"  - {x}" for x in missing_objects)
        + "\n\n"
        "Run Sections 13, 15, and 16 successfully before Section 17."
    )

print("✓ Required Section 17 objects detected.")


# ==============================================================================
# 17.2 — IDENTIFIER POLICY
# ==============================================================================

if "IDENTIFIER_COLUMNS" not in globals():

    IDENTIFIER_COLUMNS = {
        dataset_id: list(
            TRAIN_PREPROCESSING_COLUMNS[
                dataset_id
            ].get("identifier_columns", [])
        )
        for dataset_id in DATASET_IDS
    }

print("✓ Identifier policy available.")


# ==============================================================================
# 17.3 — RESET OUTPUT CONTAINERS
# ==============================================================================
#
# This is important because the previous run successfully transformed
# adult_income and bank_marketing before failing on diabetes_130us.
#
# We reset everything so that Section 17 always produces one clean,
# internally consistent result set.
# ------------------------------------------------------------------------------

TRANSFORMED_TRAIN_DATASETS = {}
TRANSFORMED_VALIDATION_DATASETS = {}
TRANSFORMED_TEST_DATASETS = {}

TRANSFORMED_DATASET_SUMMARY = []

print("✓ Transformation output containers initialized.")


# ==============================================================================
# 17.4 — TRANSFORMATION HELPER
# ==============================================================================

def transform_dataset_split(
    dataset_id,
    split_name,
    dataframe,
    preprocessor,
    expected_input_columns,
    expected_output_columns,
):
    """
    Transform a dataset split using a preprocessor fitted ONLY on TRAIN.

    The input dataframe may contain audit columns such as:
        __original_row_id

    It may also contain explicit identifiers.

    Only expected_input_columns are supplied to the preprocessor.
    """

    if dataframe is None:
        raise ValueError(
            f"{dataset_id} | {split_name}: dataframe is None."
        )

    if preprocessor is None:
        raise ValueError(
            f"{dataset_id} | {split_name}: preprocessor is None."
        )

    # --------------------------------------------------------------------------
    # Validate that every canonical modeling column exists
    # --------------------------------------------------------------------------

    missing_columns = [
        col
        for col in expected_input_columns
        if col not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{dataset_id} | {split_name}: required modeling columns "
            f"are missing:\n{missing_columns}"
        )

    # --------------------------------------------------------------------------
    # Identify additional non-modeling columns
    # --------------------------------------------------------------------------

    extra_columns = [
        col
        for col in dataframe.columns
        if col not in expected_input_columns
    ]

    # These columns are allowed.
    #
    # Examples:
    #   __original_row_id
    #   encounter_id
    #   patient_nbr
    #
    # They will NOT be passed to the preprocessor.
    # --------------------------------------------------------------------------

    # --------------------------------------------------------------------------
    # Build the exact modeling dataframe
    # --------------------------------------------------------------------------

    modeling_df = dataframe.loc[
        :,
        expected_input_columns
    ].copy()

    # --------------------------------------------------------------------------
    # Confirm exact modeling schema
    # --------------------------------------------------------------------------

    if list(modeling_df.columns) != list(expected_input_columns):
        raise RuntimeError(
            f"{dataset_id} | {split_name}: "
            "internal modeling-column ordering mismatch."
        )

    # --------------------------------------------------------------------------
    # Validate row count
    # --------------------------------------------------------------------------

    input_rows = len(dataframe)

    if input_rows == 0:
        raise ValueError(
            f"{dataset_id} | {split_name}: dataframe contains zero rows."
        )

    # --------------------------------------------------------------------------
    # Transform ONLY canonical modeling columns
    # --------------------------------------------------------------------------

    transformed = preprocessor.transform(
        modeling_df
    )

    # --------------------------------------------------------------------------
    # Convert to RAM-efficient float32
    # --------------------------------------------------------------------------

    transformed = np.asarray(
        transformed,
        dtype=np.float32
    )

    # --------------------------------------------------------------------------
    # Validate dimensionality
    # --------------------------------------------------------------------------

    if transformed.ndim != 2:
        raise ValueError(
            f"{dataset_id} | {split_name}: transformed output must be 2-D.\n"
            f"Observed shape: {transformed.shape}"
        )

    output_rows, output_columns = transformed.shape

    # --------------------------------------------------------------------------
    # Row preservation
    # --------------------------------------------------------------------------

    if output_rows != input_rows:
        raise ValueError(
            f"{dataset_id} | {split_name}: row count changed during "
            f"transformation.\n"
            f"Input rows : {input_rows}\n"
            f"Output rows: {output_rows}"
        )

    # --------------------------------------------------------------------------
    # Feature dimensionality
    # --------------------------------------------------------------------------

    if output_columns != expected_output_columns:
        raise ValueError(
            f"{dataset_id} | {split_name}: transformed feature count mismatch.\n"
            f"Expected: {expected_output_columns}\n"
            f"Actual  : {output_columns}"
        )

    # --------------------------------------------------------------------------
    # Finite-value validation
    # --------------------------------------------------------------------------

    if not np.isfinite(transformed).all():
        raise ValueError(
            f"{dataset_id} | {split_name}: transformed data contains "
            "NaN or infinite values."
        )

    # --------------------------------------------------------------------------
    # Memory
    # --------------------------------------------------------------------------

    memory_mb = transformed.nbytes / (1024 ** 2)

    return (
        transformed,
        memory_mb,
        extra_columns,
    )


# ==============================================================================
# 17.5 — TRANSFORM ALL DATASETS
# ==============================================================================

for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    # ==========================================================================
    # Retrieve training-fitted preprocessor
    # ==========================================================================

    if dataset_id not in TRAIN_PREPROCESSORS:
        raise KeyError(
            f"Training preprocessor not found for dataset: {dataset_id}"
        )

    preprocessor = TRAIN_PREPROCESSORS[dataset_id]

    # ==========================================================================
    # Retrieve CANONICAL preprocessing schema
    # ==========================================================================

    if dataset_id not in TRAIN_PREPROCESSING_COLUMNS:
        raise KeyError(
            f"TRAIN_PREPROCESSING_COLUMNS does not contain "
            f"dataset '{dataset_id}'."
        )

    preprocessing_schema = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]

    # --------------------------------------------------------------------------
    # Canonical modeling columns
    # --------------------------------------------------------------------------

    model_columns = list(
        preprocessing_schema["all_columns"]
    )

    # --------------------------------------------------------------------------
    # Canonical transformed columns
    # --------------------------------------------------------------------------

    transformed_feature_names = list(
        preprocessing_schema["transformed_columns"]
    )

    expected_output_columns = len(
        transformed_feature_names
    )

    # --------------------------------------------------------------------------
    # Display schema
    # --------------------------------------------------------------------------

    print(
        f"Canonical modeling columns : {len(model_columns)}"
    )

    print(
        f"Output encoded features    : {expected_output_columns}"
    )

    # ==========================================================================
    # Retrieve split datasets
    # ==========================================================================

    train_df = TRAIN_DATASETS[dataset_id]
    validation_df = VALIDATION_DATASETS[dataset_id]
    test_df = TEST_DATASETS[dataset_id]

    print(
        f"Train rows                 : {len(train_df):,}"
    )

    print(
        f"Validation rows            : {len(validation_df):,}"
    )

    print(
        f"Test rows                  : {len(test_df):,}"
    )

    # ==========================================================================
    # 17.5.1 — VALIDATE REQUIRED MODELING COLUMNS
    # ==========================================================================

    for split_name, split_df in [
        ("Train", train_df),
        ("Validation", validation_df),
        ("Test", test_df),
    ]:

        missing_columns = [
            col
            for col in model_columns
            if col not in split_df.columns
        ]

        if missing_columns:
            raise ValueError(
                f"{dataset_id} | {split_name}: "
                "required modeling columns missing:\n"
                f"{missing_columns}"
            )

    print(
        "✓ Required modeling columns verified for all three splits."
    )

    # ==========================================================================
    # 17.5.2 — REPORT NON-MODELING COLUMNS
    # ==========================================================================

    for split_name, split_df in [
        ("Train", train_df),
        ("Validation", validation_df),
        ("Test", test_df),
    ]:

        non_model_columns = [
            col
            for col in split_df.columns
            if col not in model_columns
        ]

        if non_model_columns:
            print(
                f"  {split_name} non-modeling columns excluded: "
                f"{non_model_columns}"
            )

    # ==========================================================================
    # 17.5.3 — TRANSFORM TRAIN
    # ==========================================================================

    print()
    print("Transforming TRAIN...")

    (
        transformed_train,
        train_memory_mb,
        train_extra_columns,
    ) = transform_dataset_split(
        dataset_id=dataset_id,
        split_name="Train",
        dataframe=train_df,
        preprocessor=preprocessor,
        expected_input_columns=model_columns,
        expected_output_columns=expected_output_columns,
    )

    TRANSFORMED_TRAIN_DATASETS[
        dataset_id
    ] = transformed_train

    print(
        f"✓ Train transformed: "
        f"{transformed_train.shape} | "
        f"{train_memory_mb:.2f} MB"
    )

    # ==========================================================================
    # 17.5.4 — TRANSFORM VALIDATION
    # ==========================================================================

    print("Transforming VALIDATION...")

    (
        transformed_validation,
        validation_memory_mb,
        validation_extra_columns,
    ) = transform_dataset_split(
        dataset_id=dataset_id,
        split_name="Validation",
        dataframe=validation_df,
        preprocessor=preprocessor,
        expected_input_columns=model_columns,
        expected_output_columns=expected_output_columns,
    )

    TRANSFORMED_VALIDATION_DATASETS[
        dataset_id
    ] = transformed_validation

    print(
        f"✓ Validation transformed: "
        f"{transformed_validation.shape} | "
        f"{validation_memory_mb:.2f} MB"
    )

    # ==========================================================================
    # 17.5.5 — TRANSFORM TEST
    # ==========================================================================

    print("Transforming TEST...")

    (
        transformed_test,
        test_memory_mb,
        test_extra_columns,
    ) = transform_dataset_split(
        dataset_id=dataset_id,
        split_name="Test",
        dataframe=test_df,
        preprocessor=preprocessor,
        expected_input_columns=model_columns,
        expected_output_columns=expected_output_columns,
    )

    TRANSFORMED_TEST_DATASETS[
        dataset_id
    ] = transformed_test

    print(
        f"✓ Test transformed: "
        f"{transformed_test.shape} | "
        f"{test_memory_mb:.2f} MB"
    )

    # ==========================================================================
    # 17.5.6 — DATASET-LEVEL VALIDATION
    # ==========================================================================

    if (
        transformed_train.shape[1]
        != transformed_validation.shape[1]
        or
        transformed_train.shape[1]
        != transformed_test.shape[1]
    ):
        raise ValueError(
            f"{dataset_id}: train/validation/test transformed "
            "feature counts are inconsistent."
        )

    # --------------------------------------------------------------------------
    # Row preservation
    # --------------------------------------------------------------------------

    if transformed_train.shape[0] != len(train_df):
        raise ValueError(
            f"{dataset_id}: train row count was not preserved."
        )

    if transformed_validation.shape[0] != len(validation_df):
        raise ValueError(
            f"{dataset_id}: validation row count was not preserved."
        )

    if transformed_test.shape[0] != len(test_df):
        raise ValueError(
            f"{dataset_id}: test row count was not preserved."
        )

    # --------------------------------------------------------------------------
    # Finite-value validation
    # --------------------------------------------------------------------------

    for split_name, array in [
        ("Train", transformed_train),
        ("Validation", transformed_validation),
        ("Test", transformed_test),
    ]:

        if not np.isfinite(array).all():
            raise ValueError(
                f"{dataset_id} | {split_name}: "
                "non-finite values detected."
            )

    # --------------------------------------------------------------------------
    # Calculate memory
    # --------------------------------------------------------------------------

    total_memory_mb = (
        train_memory_mb
        + validation_memory_mb
        + test_memory_mb
    )

    # --------------------------------------------------------------------------
    # Summary
    # --------------------------------------------------------------------------

    TRANSFORMED_DATASET_SUMMARY.append({
        "dataset_id": dataset_id,
        "train_rows": len(train_df),
        "validation_rows": len(validation_df),
        "test_rows": len(test_df),
        "input_modeling_features": len(model_columns),
        "transformed_features": expected_output_columns,
        "train_memory_mb": train_memory_mb,
        "validation_memory_mb": validation_memory_mb,
        "test_memory_mb": test_memory_mb,
        "total_memory_mb": total_memory_mb,
        "dtype": str(transformed_train.dtype),
        "train_non_model_columns_excluded": len(
            train_extra_columns
        ),
        "validation_non_model_columns_excluded": len(
            validation_extra_columns
        ),
        "test_non_model_columns_excluded": len(
            test_extra_columns
        ),
        "status": "SUCCESS",
    })

    print()
    print(f"✓ {dataset_id} transformation PASSED.")
    print(
        f"  Total transformed memory: "
        f"{total_memory_mb:.2f} MB"
    )


# ==============================================================================
# 17.6 — CREATE SUMMARY DATAFRAME
# ==============================================================================

TRANSFORMED_DATASET_SUMMARY_DF = pd.DataFrame(
    TRANSFORMED_DATASET_SUMMARY
)

print()
print("=" * 100)
print("SECTION 17 SUMMARY")
print("=" * 100)

display(
    TRANSFORMED_DATASET_SUMMARY_DF
)


# ==============================================================================
# 17.7 — GLOBAL DATASET COVERAGE
# ==============================================================================

assert set(
    TRANSFORMED_TRAIN_DATASETS.keys()
) == set(DATASET_IDS), (
    "Transformed train dataset coverage mismatch."
)

assert set(
    TRANSFORMED_VALIDATION_DATASETS.keys()
) == set(DATASET_IDS), (
    "Transformed validation dataset coverage mismatch."
)

assert set(
    TRANSFORMED_TEST_DATASETS.keys()
) == set(DATASET_IDS), (
    "Transformed test dataset coverage mismatch."
)

print(
    "✓ All datasets transformed."
)


# ==============================================================================
# 17.8 — FEATURE DIMENSIONALITY VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    train_array = TRANSFORMED_TRAIN_DATASETS[
        dataset_id
    ]

    validation_array = TRANSFORMED_VALIDATION_DATASETS[
        dataset_id
    ]

    test_array = TRANSFORMED_TEST_DATASETS[
        dataset_id
    ]

    expected_features = len(
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]["transformed_columns"]
    )

    assert train_array.shape[1] == expected_features

    assert validation_array.shape[1] == expected_features

    assert test_array.shape[1] == expected_features

    assert train_array.shape[1] == validation_array.shape[1]

    assert train_array.shape[1] == test_array.shape[1]

print(
    "✓ Transformed feature dimensionality is consistent."
)


# ==============================================================================
# 17.9 — ROW PRESERVATION VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    assert (
        TRANSFORMED_TRAIN_DATASETS[
            dataset_id
        ].shape[0]
        ==
        len(TRAIN_DATASETS[dataset_id])
    )

    assert (
        TRANSFORMED_VALIDATION_DATASETS[
            dataset_id
        ].shape[0]
        ==
        len(VALIDATION_DATASETS[dataset_id])
    )

    assert (
        TRANSFORMED_TEST_DATASETS[
            dataset_id
        ].shape[0]
        ==
        len(TEST_DATASETS[dataset_id])
    )

print(
    "✓ Row counts preserved for all datasets and splits."
)


# ==============================================================================
# 17.10 — FINITE-VALUE VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    for array in [
        TRANSFORMED_TRAIN_DATASETS[dataset_id],
        TRANSFORMED_VALIDATION_DATASETS[dataset_id],
        TRANSFORMED_TEST_DATASETS[dataset_id],
    ]:

        assert np.isfinite(array).all()

print(
    "✓ No NaN or infinite values detected."
)


# ==============================================================================
# 17.11 — DTYPE VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    for array in [
        TRANSFORMED_TRAIN_DATASETS[dataset_id],
        TRANSFORMED_VALIDATION_DATASETS[dataset_id],
        TRANSFORMED_TEST_DATASETS[dataset_id],
    ]:

        assert array.dtype == np.float32

print(
    "✓ All transformed arrays are float32."
)


# ==============================================================================
# 17.12 — TRAINING-ONLY FIT VALIDATION
# ==============================================================================
#
# Section 17 must NEVER fit the preprocessor.
#
# We verify that every preprocessor is already fitted and is the same object
# that Section 16 produced.
# ------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    preprocessor = TRAIN_PREPROCESSORS[
        dataset_id
    ]

    if not hasattr(
        preprocessor,
        "transformers_"
    ):
        raise RuntimeError(
            f"{dataset_id}: preprocessor does not appear to be fitted."
        )

print(
    "✓ All preprocessors are already fitted before transformation."
)

print(
    "✓ No preprocessing refitting performed in Section 17."
)


# ==============================================================================
# 17.13 — FINAL PASS
# ==============================================================================

print()
print("=" * 100)
print("SECTION 17 — PASS")
print("=" * 100)

print()
print("Training-fitted preprocessing was successfully applied to:")

print(
    "  ✓ TRAIN"
)

print(
    "  ✓ VALIDATION"
)

print(
    "  ✓ TEST"
)

print()
print("Methodological confirmation:")

print(
    "  ✓ Preprocessors were fitted only on TRAIN."
)

print(
    "  ✓ Validation was transformed without refitting."
)

print(
    "  ✓ Test was transformed without refitting."
)

print(
    "  ✓ Explicit identifiers were not passed to the preprocessor."
)

print(
    "  ✓ __original_row_id was not passed to the preprocessor."
)

print(
    "  ✓ Canonical modeling columns were used."
)

print(
    "  ✓ Row counts were preserved."
)

print(
    "  ✓ Output feature dimensionality was preserved."
)

print(
    "  ✓ No NaN or infinite values remain."
)

print(
    "  ✓ All transformed arrays are float32."
)

print()
print(
    "SECTION 17 COMPLETED SUCCESSFULLY."
)

SECTION 17 — TRANSFORM TRAIN / VALIDATION / TEST
✓ Required Section 17 objects detected.
✓ Identifier policy available.
✓ Transformation output containers initialized.

----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
Canonical modeling columns : 15
Output encoded features    : 107
Train rows                 : 34,189
Validation rows            : 7,326
Test rows                  : 7,327
✓ Required modeling columns verified for all three splits.
  Train non-modeling columns excluded: ['__original_row_id__']
  Validation non-modeling columns excluded: ['__original_row_id__']
  Test non-modeling columns excluded: ['__original_row_id__']

Transforming TRAIN...
✓ Train transformed: (34189, 107) | 13.96 MB
Transforming VALIDATION...
✓ Validation transformed: (7326, 107) | 2.99 MB
Transforming TEST...
✓ Test transformed: 

,dataset_id,train_rows,validation_rows,test_rows,input_modeling_features,transformed_features,train_memory_mb,validation_memory_mb,test_memory_mb,total_memory_mb,dtype,train_non_model_columns_excluded,validation_non_model_columns_excluded,test_non_model_columns_excluded,status
0,adult_income,34189,7326,7327,15,107,13.955013,2.990273,2.990681,19.935966,float32,1,1,1,SUCCESS
1,bank_marketing,31647,6782,6782,17,53,6.398357,1.371178,1.371178,9.140713,float32,1,1,1,SUCCESS
2,diabetes_130us,71236,15265,15265,48,2339,635.608688,136.203136,136.203136,908.014961,float32,3,3,3,SUCCESS


✓ All datasets transformed.
✓ Transformed feature dimensionality is consistent.
✓ Row counts preserved for all datasets and splits.
✓ No NaN or infinite values detected.
✓ All transformed arrays are float32.
✓ All preprocessors are already fitted before transformation.
✓ No preprocessing refitting performed in Section 17.

SECTION 17 — PASS

Training-fitted preprocessing was successfully applied to:
  ✓ TRAIN
  ✓ VALIDATION
  ✓ TEST

Methodological confirmation:
  ✓ Preprocessors were fitted only on TRAIN.
  ✓ Validation was transformed without refitting.
  ✓ Test was transformed without refitting.
  ✓ Explicit identifiers were not passed to the preprocessor.
  ✓ __original_row_id was not passed to the preprocessor.
  ✓ Canonical modeling columns were used.
  ✓ Row counts were preserved.
  ✓ Output feature dimensionality was preserved.
  ✓ No NaN or infinite values remain.
  ✓ All transformed arrays are float32.

SECTION 17 COMPLETED SUCCESSFULLY.


In [69]:
# ==================================================================================================
# 18. CREATE NATIVE PROCESSED DATASETS
# ==================================================================================================

print("=" * 100)
print("18. CREATE NATIVE PROCESSED DATASETS")
print("=" * 100)


# ==================================================================================================
# 18.1 REQUIRED OBJECT VALIDATION
# ==================================================================================================

required_objects = [
    "DATASET_IDS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
    "TRAIN_DATASETS",
    "VALIDATION_DATASETS",
    "TEST_DATASETS",
    "TRAIN_PREPROCESSING_COLUMNS",
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 18 cannot proceed.\n"
        f"Missing required objects: {missing_objects}"
    )

print("✓ Required objects detected.")


# ==================================================================================================
# 18.2 CANONICAL PROVENANCE COLUMN
# ==================================================================================================

PROVENANCE_COLUMN = "__original_row_id__"


# ==================================================================================================
# 18.3 INITIALIZE NATIVE DATASET CONTAINER
# ==================================================================================================

NATIVE_FINAL_DATASETS = {}

print("✓ Native dataset container initialized.")


# ==================================================================================================
# 18.4 CREATE NATIVE DATASETS
# ==================================================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    target = TARGET_COLUMNS[dataset_id]

    identifier_columns = IDENTIFIER_COLUMNS.get(
        dataset_id,
        []
    )

    # ----------------------------------------------------------------------------------------------
    # Obtain canonical modeling schema
    # ----------------------------------------------------------------------------------------------

    if dataset_id not in TRAIN_PREPROCESSING_COLUMNS:
        raise RuntimeError(
            f"{dataset_id}: preprocessing schema not found in "
            "TRAIN_PREPROCESSING_COLUMNS."
        )

    schema = TRAIN_PREPROCESSING_COLUMNS[dataset_id]

    model_columns = list(
        schema["all_columns"]
    )

    print(f"Target                     : {target}")
    print(f"Identifier columns         : {identifier_columns}")
    print(f"Modeling columns           : {len(model_columns)}")


    # ----------------------------------------------------------------------------------------------
    # Validate modeling schema
    # ----------------------------------------------------------------------------------------------

    if target not in model_columns:
        raise RuntimeError(
            f"{dataset_id}: target '{target}' is missing from "
            "the canonical modeling schema."
        )

    identifier_overlap = set(
        identifier_columns
    ).intersection(model_columns)

    if identifier_overlap:
        raise RuntimeError(
            f"{dataset_id}: identifier columns appear in the "
            f"modeling schema: {sorted(identifier_overlap)}"
        )

    if len(model_columns) != len(set(model_columns)):
        raise RuntimeError(
            f"{dataset_id}: duplicate columns found in modeling schema."
        )


    # ----------------------------------------------------------------------------------------------
    # Get canonical split DataFrames
    # ----------------------------------------------------------------------------------------------

    split_sources = {
        "train": TRAIN_DATASETS[dataset_id],
        "validation": VALIDATION_DATASETS[dataset_id],
        "test": TEST_DATASETS[dataset_id],
    }

    NATIVE_FINAL_DATASETS[dataset_id] = {}


    # ----------------------------------------------------------------------------------------------
    # Process each split
    # ----------------------------------------------------------------------------------------------

    for split_name, original_split in split_sources.items():

        print(f"\n  Split: {split_name}")


        # ------------------------------------------------------------------------------------------
        # Validate provenance
        # ------------------------------------------------------------------------------------------

        if PROVENANCE_COLUMN not in original_split.columns:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                f"provenance column '{PROVENANCE_COLUMN}' "
                "was not found."
            )

        provenance = original_split[
            PROVENANCE_COLUMN
        ].copy()

        if provenance.isna().any():
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "provenance contains missing values."
            )

        if not provenance.is_unique:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "provenance IDs are not unique."
            )


        # ------------------------------------------------------------------------------------------
        # Validate all modeling columns exist
        # ------------------------------------------------------------------------------------------

        missing_model_columns = [
            col for col in model_columns
            if col not in original_split.columns
        ]

        if missing_model_columns:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "required modeling columns are missing:\n"
                f"{missing_model_columns}"
            )


        # ------------------------------------------------------------------------------------------
        # Construct native modeling data
        #
        # IMPORTANT:
        # Only the canonical modeling columns are selected here.
        #
        # Explicit identifiers are therefore excluded for diabetes_130us.
        #
        # Provenance is then added exactly once.
        # ------------------------------------------------------------------------------------------

        native_features = original_split[
            model_columns
        ].copy()


        # ------------------------------------------------------------------------------------------
        # Verify provenance is NOT already present
        # ------------------------------------------------------------------------------------------

        if PROVENANCE_COLUMN in native_features.columns:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                f"provenance column '{PROVENANCE_COLUMN}' "
                "unexpectedly appears inside the modeling schema."
            )


        # ------------------------------------------------------------------------------------------
        # Add provenance exactly once
        # ------------------------------------------------------------------------------------------

        native_features.insert(
            0,
            PROVENANCE_COLUMN,
            provenance.to_numpy(copy=True)
        )


        # ------------------------------------------------------------------------------------------
        # Validate target retention
        # ------------------------------------------------------------------------------------------

        if target not in native_features.columns:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                f"target '{target}' was not retained."
            )


        # ------------------------------------------------------------------------------------------
        # Validate identifier exclusion
        # ------------------------------------------------------------------------------------------

        identifier_leaks = [
            col
            for col in identifier_columns
            if col in native_features.columns
        ]

        if identifier_leaks:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "identifier leakage detected:\n"
                f"{identifier_leaks}"
            )


        # ------------------------------------------------------------------------------------------
        # Validate duplicate columns
        # ------------------------------------------------------------------------------------------

        if len(native_features.columns) != len(
            set(native_features.columns)
        ):
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "duplicate columns detected."
            )


        # ------------------------------------------------------------------------------------------
        # Validate row count
        # ------------------------------------------------------------------------------------------

        if len(native_features) != len(original_split):
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "row count changed during native dataset construction."
            )


        # ------------------------------------------------------------------------------------------
        # Validate provenance alignment
        # ------------------------------------------------------------------------------------------

        if not np.array_equal(
            native_features[PROVENANCE_COLUMN].to_numpy(),
            original_split[PROVENANCE_COLUMN].to_numpy()
        ):
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "provenance alignment changed."
            )


        # ------------------------------------------------------------------------------------------
        # Validate target alignment
        # ------------------------------------------------------------------------------------------

        if not np.array_equal(
            native_features[target].to_numpy(),
            original_split[target].to_numpy()
        ):
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "target values changed during native dataset construction."
            )


        # ------------------------------------------------------------------------------------------
        # Store
        # ------------------------------------------------------------------------------------------

        NATIVE_FINAL_DATASETS[dataset_id][
            split_name
        ] = native_features


        # ------------------------------------------------------------------------------------------
        # Display split result
        # ------------------------------------------------------------------------------------------

        print(
            f"    Rows                     : "
            f"{len(native_features):,}"
        )

        print(
            f"    Modeling columns        : "
            f"{len(model_columns):,}"
        )

        print(
            f"    Native columns          : "
            f"{len(native_features.columns):,}"
        )

        print(
            f"    Provenance              : "
            f"{PROVENANCE_COLUMN}"
        )

        print(
            f"    Target retained        : "
            f"PASS"
        )

        print(
            f"    Identifier exclusion   : "
            f"PASS"
        )

        print(
            f"    Row alignment          : "
            f"PASS"
        )


# ==================================================================================================
# 18.5 GLOBAL NATIVE DATASET VALIDATION
# ==================================================================================================

print("\n" + "=" * 100)
print("18.5 GLOBAL NATIVE DATASET VALIDATION")
print("=" * 100)


for dataset_id in DATASET_IDS:

    for split_name in [
        "train",
        "validation",
        "test",
    ]:

        native_df = NATIVE_FINAL_DATASETS[
            dataset_id
        ][split_name]

        target = TARGET_COLUMNS[dataset_id]

        identifier_columns = IDENTIFIER_COLUMNS.get(
            dataset_id,
            []
        )

        model_columns = TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]["all_columns"]


        # Expected native structure:
        # provenance + modeling columns

        expected_column_count = len(model_columns) + 1

        if len(native_df.columns) != expected_column_count:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                f"expected {expected_column_count} native columns, "
                f"found {len(native_df.columns)}."
            )


        expected_columns = [
            PROVENANCE_COLUMN
        ] + model_columns

        if list(native_df.columns) != expected_columns:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "native column order/schema mismatch."
            )


        # Target
        if target not in native_df.columns:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "target missing from native dataset."
            )


        # Identifiers
        leaked_identifiers = [
            col
            for col in identifier_columns
            if col in native_df.columns
        ]

        if leaked_identifiers:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                f"identifier leakage: {leaked_identifiers}"
            )


        # Provenance
        if PROVENANCE_COLUMN not in native_df.columns:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "provenance column missing."
            )

        if not native_df[
            PROVENANCE_COLUMN
        ].is_unique:
            raise RuntimeError(
                f"{dataset_id} | {split_name}: "
                "provenance is not unique."
            )


print("✓ Native dataset schema validation : PASS")
print("✓ Target retention                : PASS")
print("✓ Identifier exclusion             : PASS")
print("✓ Provenance validation             : PASS")
print("✓ Column-order validation           : PASS")


# ==================================================================================================
# 18.6 FINAL SUMMARY
# ==================================================================================================

print("\n" + "=" * 100)
print("SECTION 18 COMPLETE")
print("=" * 100)

for dataset_id in DATASET_IDS:

    print(f"\nDataset: {dataset_id}")

    for split_name in [
        "train",
        "validation",
        "test",
    ]:

        native_df = NATIVE_FINAL_DATASETS[
            dataset_id
        ][split_name]

        print(
            f"  {split_name:10s} : "
            f"{native_df.shape[0]:,} rows × "
            f"{native_df.shape[1]:,} columns"
        )

print("\nCanonical output:")
print("  ✓ NATIVE_FINAL_DATASETS")

print("\nNative dataset policy:")
print("  ✓ Original/native feature values preserved")
print("  ✓ Target retained")
print("  ✓ Explicit identifiers excluded")
print("  ✓ Provenance retained for auditability")
print("  ✓ No encoding")
print("  ✓ No scaling")
print("  ✓ No refitting")
print("  ✓ No transformation of native values")

print("\nSTATUS: PASS")

18. CREATE NATIVE PROCESSED DATASETS
✓ Required objects detected.
✓ Native dataset container initialized.

----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
Target                     : income
Identifier columns         : []
Modeling columns           : 15

  Split: train
    Rows                     : 34,189
    Modeling columns        : 15
    Native columns          : 16
    Provenance              : __original_row_id__
    Target retained        : PASS
    Identifier exclusion   : PASS
    Row alignment          : PASS

  Split: validation
    Rows                     : 7,326
    Modeling columns        : 15
    Native columns          : 16
    Provenance              : __original_row_id__
    Target retained        : PASS
    Identifier exclusion   : PASS
    Row alignment          : PASS

  Split: test
    Ro

In [111]:
# ==============================================================================
# SECTION 19 — CREATE FINAL ENCODED DATASETS
# ==============================================================================
#
# Canonical structure:
#
# ENCODED_FINAL_DATASETS[dataset_id]["train"]
# ENCODED_FINAL_DATASETS[dataset_id]["validation"]
# ENCODED_FINAL_DATASETS[dataset_id]["test"]
#
# Encoded datasets contain ONLY transformed model features.
#
# They must NOT contain:
#   - __original_row_id__
#   - explicit identifiers
#   - manually appended raw target
#
# Target is already represented through the fitted categorical preprocessing
# pipeline because target is part of TRAIN_PREPROCESSING_COLUMNS["all_columns"].
# ==============================================================================

print("=" * 100)
print("19. CREATE FINAL ENCODED DATASETS")
print("=" * 100)


# ==============================================================================
# 19.1 REQUIRED OBJECTS
# ==============================================================================

required_objects = [
    "DATASET_IDS",
    "TRAIN_PREPROCESSING_COLUMNS",
    "TRANSFORMED_TRAIN_DATASETS",
    "TRANSFORMED_VALIDATION_DATASETS",
    "TRANSFORMED_TEST_DATASETS",
    "TRAIN_DATASETS",
    "VALIDATION_DATASETS",
    "TEST_DATASETS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 19 cannot start. Missing required objects:\n"
        + "\n".join(f"  - {name}" for name in missing_objects)
    )


# ==============================================================================
# 19.2 RESET CANONICAL DATASET-FIRST STRUCTURE
# ==============================================================================

ENCODED_FINAL_DATASETS = {}

ENCODED_DATASET_SUMMARY = []


# ==============================================================================
# 19.3 BUILD ENCODED DATASETS
# ==============================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"DATASET : {dataset_id}")
    print("-" * 100)

    schema = TRAIN_PREPROCESSING_COLUMNS[dataset_id]

    transformed_columns = list(
        schema["transformed_columns"]
    )

    expected_feature_count = len(
        transformed_columns
    )

    target_column = TARGET_COLUMNS[dataset_id]

    identifier_columns = IDENTIFIER_COLUMNS.get(
        dataset_id,
        []
    )

    provenance_column = schema.get(
        "provenance_column",
        "__original_row_id__"
    )

    # --------------------------------------------------------------------------
    # IMPORTANT:
    # Dataset-first canonical structure
    # --------------------------------------------------------------------------

    ENCODED_FINAL_DATASETS[dataset_id] = {}

    split_arrays = {
        "train": TRANSFORMED_TRAIN_DATASETS[dataset_id],
        "validation": TRANSFORMED_VALIDATION_DATASETS[dataset_id],
        "test": TRANSFORMED_TEST_DATASETS[dataset_id],
    }

    original_split_data = {
        "train": TRAIN_DATASETS[dataset_id],
        "validation": VALIDATION_DATASETS[dataset_id],
        "test": TEST_DATASETS[dataset_id],
    }

    # --------------------------------------------------------------------------
    # Process each split
    # --------------------------------------------------------------------------

    for split_name, transformed_array in split_arrays.items():

        original_df = original_split_data[split_name]

        if not isinstance(
            transformed_array,
            np.ndarray
        ):
            raise TypeError(
                f"{dataset_id} / {split_name}: "
                f"expected NumPy array, got "
                f"{type(transformed_array).__name__}"
            )

        if transformed_array.ndim != 2:
            raise ValueError(
                f"{dataset_id} / {split_name}: "
                f"expected 2D array, got ndim="
                f"{transformed_array.ndim}"
            )

        # ----------------------------------------------------------------------
        # Validate transformed dimensions
        # ----------------------------------------------------------------------

        if transformed_array.shape[1] != expected_feature_count:

            raise ValueError(
                f"{dataset_id} / {split_name}: transformed feature count "
                f"{transformed_array.shape[1]} != expected "
                f"{expected_feature_count}"
            )

        if transformed_array.shape[0] != len(original_df):

            raise ValueError(
                f"{dataset_id} / {split_name}: row count "
                f"{transformed_array.shape[0]} != original split "
                f"row count {len(original_df)}"
            )

        # ----------------------------------------------------------------------
        # Convert to float32 without unnecessary duplication
        # ----------------------------------------------------------------------

        encoded_array = np.asarray(
            transformed_array,
            dtype=np.float32
        )

        # ----------------------------------------------------------------------
        # Finite-value validation
        # ----------------------------------------------------------------------

        chunk_size = 10000

        finite_pass = True

        for start in range(
            0,
            encoded_array.shape[0],
            chunk_size
        ):

            stop = min(
                start + chunk_size,
                encoded_array.shape[0]
            )

            if not np.isfinite(
                encoded_array[start:stop]
            ).all():

                finite_pass = False

                raise ValueError(
                    f"{dataset_id} / {split_name}: "
                    f"non-finite transformed values detected "
                    f"in rows {start}:{stop}"
                )

        # ----------------------------------------------------------------------
        # CRITICAL:
        #
        # Store ONLY model-transformed features.
        #
        # DO NOT append:
        #   provenance
        #   identifiers
        #   raw target
        #
        # Provenance remains in NATIVE_FINAL_DATASETS.
        # ----------------------------------------------------------------------

        ENCODED_FINAL_DATASETS[
            dataset_id
        ][split_name] = encoded_array

        # ----------------------------------------------------------------------
        # Structural validation
        # ----------------------------------------------------------------------

        stored_array = ENCODED_FINAL_DATASETS[
            dataset_id
        ][split_name]

        if stored_array.shape != (
            len(original_df),
            expected_feature_count
        ):

            raise AssertionError(
                f"{dataset_id} / {split_name}: "
                f"stored encoded shape mismatch."
            )

        # ----------------------------------------------------------------------
        # Record summary
        # ----------------------------------------------------------------------

        ENCODED_DATASET_SUMMARY.append({

            "dataset_id": dataset_id,

            "split": split_name,

            "rows": len(original_df),

            "encoded_features": expected_feature_count,

            "dtype": str(
                stored_array.dtype
            ),

            "target_column": target_column,

            "target_in_modeling_schema": (
                target_column in schema["all_columns"]
            ),

            "identifier_columns_excluded": (
                len(identifier_columns) == 0
                or all(
                    identifier not in schema["all_columns"]
                    for identifier in identifier_columns
                )
            ),

            "provenance_excluded": (
                provenance_column
                not in schema["all_columns"]
            ),

            "raw_target_manually_appended": False,

            "finite_values": finite_pass,

            "status": "PASS",
        })

        print(
            f"  {split_name:<12}: "
            f"{stored_array.shape} | "
            f"dtype={stored_array.dtype}"
        )


# ==============================================================================
# 19.4 BUILD SUMMARY DATAFRAME
# ==============================================================================

ENCODED_DATASET_SUMMARY_DF = pd.DataFrame(
    ENCODED_DATASET_SUMMARY
)


# ==============================================================================
# 19.5 FINAL STRUCTURE VALIDATION
# ==============================================================================

required_splits = [
    "train",
    "validation",
    "test",
]

encoded_structure_pass = True

for dataset_id in DATASET_IDS:

    if dataset_id not in ENCODED_FINAL_DATASETS:

        encoded_structure_pass = False

        raise AssertionError(
            f"Missing dataset key: {dataset_id}"
        )

    dataset_container = (
        ENCODED_FINAL_DATASETS[dataset_id]
    )

    if list(dataset_container.keys()) != required_splits:

        encoded_structure_pass = False

        raise AssertionError(
            f"{dataset_id}: incorrect split structure. "
            f"Found {list(dataset_container.keys())}; "
            f"expected {required_splits}"
        )


# ==============================================================================
# 19.6 VERIFY NO PROVENANCE / IDENTIFIER LEAKAGE
# ==============================================================================

for dataset_id in DATASET_IDS:

    schema = TRAIN_PREPROCESSING_COLUMNS[dataset_id]

    provenance_column = schema.get(
        "provenance_column",
        "__original_row_id__"
    )

    identifier_columns = IDENTIFIER_COLUMNS.get(
        dataset_id,
        []
    )

    # Encoded datasets are NumPy arrays.
    # Therefore raw metadata columns cannot be present.

    for split_name in required_splits:

        encoded_array = (
            ENCODED_FINAL_DATASETS[
                dataset_id
            ][split_name]
        )

        if not isinstance(
            encoded_array,
            np.ndarray
        ):

            raise AssertionError(
                f"{dataset_id} / {split_name}: "
                "encoded dataset must be NumPy ndarray."
            )

        # The encoded array must have exactly the transformed
        # feature count and nothing else.

        if encoded_array.shape[1] != len(
            schema["transformed_columns"]
        ):

            raise AssertionError(
                f"{dataset_id} / {split_name}: "
                "encoded array contains unexpected columns."
            )


# ==============================================================================
# 19.7 CROSS-SPLIT ROW VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    expected_rows = {
        "train": len(
            TRAIN_DATASETS[dataset_id]
        ),
        "validation": len(
            VALIDATION_DATASETS[dataset_id]
        ),
        "test": len(
            TEST_DATASETS[dataset_id]
        ),
    }

    for split_name in required_splits:

        actual_rows = len(
            ENCODED_FINAL_DATASETS[
                dataset_id
            ][split_name]
        )

        if actual_rows != expected_rows[split_name]:

            raise AssertionError(
                f"{dataset_id} / {split_name}: "
                f"encoded rows={actual_rows}, "
                f"expected={expected_rows[split_name]}"
            )


# ==============================================================================
# 19.8 DISPLAY SUMMARY
# ==============================================================================

print("\n" + "=" * 100)
print("ENCODED DATASET SUMMARY")
print("=" * 100)

for dataset_id in DATASET_IDS:

    print(f"\n{dataset_id}")

    for split_name in required_splits:

        array = (
            ENCODED_FINAL_DATASETS[
                dataset_id
            ][split_name]
        )

        print(
            f"  {split_name:<12}: "
            f"{array.shape} | "
            f"dtype={array.dtype}"
        )


# ==============================================================================
# 19.9 FINAL SECTION 19 STATUS
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 19 STATUS: PASS")
print("=" * 100)

print(
    "\n✓ Dataset-first encoded structure confirmed."
)

print(
    "✓ Only transformed model features are stored."
)

print(
    "✓ Provenance is NOT included in encoded datasets."
)

print(
    "✓ Explicit identifiers are NOT included in encoded datasets."
)

print(
    "✓ Raw target was NOT manually appended."
)

print(
    "✓ Target remains represented through the fitted preprocessing pipeline."
)

19. CREATE FINAL ENCODED DATASETS

----------------------------------------------------------------------------------------------------
DATASET : adult_income
----------------------------------------------------------------------------------------------------
  train       : (34189, 107) | dtype=float32
  validation  : (7326, 107) | dtype=float32
  test        : (7327, 107) | dtype=float32

----------------------------------------------------------------------------------------------------
DATASET : bank_marketing
----------------------------------------------------------------------------------------------------
  train       : (31647, 53) | dtype=float32
  validation  : (6782, 53) | dtype=float32
  test        : (6782, 53) | dtype=float32

----------------------------------------------------------------------------------------------------
DATASET : diabetes_130us
----------------------------------------------------------------------------------------------------
  train       : (7123

In [112]:
# ==============================================================================
# SECTION 20 — PERSIST PREPROCESSORS, SCHEMAS & METADATA
# ==============================================================================
#
# Canonical artifacts:
#
# preprocessors/
#   adult_income/
#       train_fitted_preprocessor.joblib
#   bank_marketing/
#       train_fitted_preprocessor.joblib
#   diabetes_130us/
#       train_fitted_preprocessor.joblib
#
# schemas/
#   adult_income/
#       preprocessing_schema.json
#   bank_marketing/
#       preprocessing_schema.json
#   diabetes_130us/
#       preprocessing_schema.json
#
# schemas/metadata/
#   adult_income_preprocessing_metadata.json
#   bank_marketing_preprocessing_metadata.json
#   diabetes_130us_preprocessing_metadata.json
# ==============================================================================

import os
import json
import joblib
from pathlib import Path
from datetime import datetime, timezone

print("=" * 100)
print("20. PERSIST PREPROCESSORS, SCHEMAS & METADATA")
print("=" * 100)


# ==============================================================================
# 20.1 REQUIRED OBJECTS
# ==============================================================================

required_objects = [
    "DATASET_IDS",
    "TRAIN_PREPROCESSORS",
    "TRAIN_PREPROCESSING_COLUMNS",
    "TRAIN_PREPROCESSING_METADATA",
    "NB02_DIRECTORIES",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "Section 20 cannot start. Missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in missing_objects
        )
    )


# ==============================================================================
# 20.2 CANONICAL ROOTS
# ==============================================================================

preprocessors_root = Path(
    NB02_DIRECTORIES["preprocessors"]
)

schemas_root = Path(
    NB02_DIRECTORIES["schemas"]
)

metadata_root = (
    schemas_root / "metadata"
)

preprocessors_root.mkdir(
    parents=True,
    exist_ok=True
)

schemas_root.mkdir(
    parents=True,
    exist_ok=True
)

metadata_root.mkdir(
    parents=True,
    exist_ok=True
)


# ==============================================================================
# 20.3 RESET MANIFEST
# ==============================================================================

PREPROCESSOR_MANIFEST_RECORDS = []


# ==============================================================================
# 20.4 SAVE EACH DATASET
# ==============================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"PERSISTING : {dataset_id}")
    print("-" * 100)

    preprocessor = (
        TRAIN_PREPROCESSORS[dataset_id]
    )

    schema = (
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]
    )

    metadata = (
        TRAIN_PREPROCESSING_METADATA[
            dataset_id
        ]
    )

    # --------------------------------------------------------------------------
    # Dataset-specific directories
    # --------------------------------------------------------------------------

    dataset_preprocessor_root = (
        preprocessors_root / dataset_id
    )

    dataset_schema_root = (
        schemas_root / dataset_id
    )

    dataset_preprocessor_root.mkdir(
        parents=True,
        exist_ok=True
    )

    dataset_schema_root.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------------------------
    # Preprocessor artifact
    # --------------------------------------------------------------------------

    preprocessor_path = (
        dataset_preprocessor_root
        / "train_fitted_preprocessor.joblib"
    )

    joblib.dump(
        preprocessor,
        preprocessor_path
    )

    # --------------------------------------------------------------------------
    # Transformation feature names
    # --------------------------------------------------------------------------

    try:

        transformed_feature_names = list(
            preprocessor.get_feature_names_out()
        )

    except Exception as exc:

        raise RuntimeError(
            f"{dataset_id}: unable to obtain transformed "
            f"feature names: {exc}"
        )

    # --------------------------------------------------------------------------
    # Canonical schema
    # --------------------------------------------------------------------------

    canonical_schema = {

        "dataset_id": dataset_id,

        "schema_version": "2.0",

        "fit_policy": "train_only",

        "modeling_schema": {

            "all_columns": list(
                schema["all_columns"]
            ),

            "numeric_columns": list(
                schema["numeric_columns"]
            ),

            "categorical_columns": list(
                schema["categorical_columns"]
            ),

            "identifier_columns_excluded": list(
                schema["identifier_columns"]
            ),

            "target_column": schema[
                "target_column"
            ],

            "provenance_column": schema.get(
                "provenance_column",
                "__original_row_id__"
            ),
        },

        "transformation_schema": {

            "transformed_feature_count": len(
                transformed_feature_names
            ),

            "transformed_feature_names": (
                transformed_feature_names
            ),
        },

        "preprocessing": {

            "numeric": {

                "imputation": "median",

                "scaling": "standard_scaler",
            },

            "categorical": {

                "imputation": "most_frequent",

                "encoding": "one_hot",

                "handle_unknown": "ignore",
            },

            "column_transformer": {

                "remainder": "drop",

                "verbose_feature_names_out": False,
            },
        },

        "training_rows": int(
            len(
                TRAIN_PREPROCESSING_DATA[
                    dataset_id
                ]
            )
        )
        if "TRAIN_PREPROCESSING_DATA" in globals()
        else int(
            metadata.get(
                "training_rows",
                0
            )
        ),

        "creation_timestamp_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
    }

    # --------------------------------------------------------------------------
    # Save schema
    # --------------------------------------------------------------------------

    schema_path = (
        dataset_schema_root
        / "preprocessing_schema.json"
    )

    with open(
        schema_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            canonical_schema,
            f,
            indent=2
        )

    # --------------------------------------------------------------------------
    # Canonical metadata
    #
    # IMPORTANT:
    # This filename is the exact filename Section 24 verifies.
    # --------------------------------------------------------------------------

    canonical_metadata = dict(
        metadata
    )

    canonical_metadata.update({

        "dataset_id": dataset_id,

        "fit_dataset": "train_only",

        "fit_policy": "train_only",

        "preprocessor_artifact": str(
            preprocessor_path
        ),

        "schema_artifact": str(
            schema_path
        ),

        "transformed_feature_count": len(
            transformed_feature_names
        ),

        "transformed_feature_names": (
            transformed_feature_names
        ),

        "creation_timestamp_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
    })

    metadata_path = (
        metadata_root
        / f"{dataset_id}_preprocessing_metadata.json"
    )

    with open(
        metadata_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            canonical_metadata,
            f,
            indent=2
        )

    # --------------------------------------------------------------------------
    # Immediate artifact validation
    # --------------------------------------------------------------------------

    if not preprocessor_path.exists():
        raise RuntimeError(
            f"{dataset_id}: preprocessor artifact not created."
        )

    if not schema_path.exists():
        raise RuntimeError(
            f"{dataset_id}: schema artifact not created."
        )

    if not metadata_path.exists():
        raise RuntimeError(
            f"{dataset_id}: metadata artifact not created."
        )

    # --------------------------------------------------------------------------
    # Manifest
    # --------------------------------------------------------------------------

    PREPROCESSOR_MANIFEST_RECORDS.append({

        "dataset_id": dataset_id,

        "preprocessor_path": str(
            preprocessor_path
        ),

        "schema_path": str(
            schema_path
        ),

        "metadata_path": str(
            metadata_path
        ),

        "fit_policy": "train_only",

        "transformed_feature_count": len(
            transformed_feature_names
        ),

        "preprocessor_exists": True,

        "schema_exists": True,

        "metadata_exists": True,

        "status": "PASS",
    })

    print(
        f"  Preprocessor : {preprocessor_path}"
    )

    print(
        f"  Schema       : {schema_path}"
    )

    print(
        f"  Metadata     : {metadata_path}"
    )


# ==============================================================================
# 20.5 SAVE MANIFEST
# ==============================================================================

PREPROCESSOR_MANIFEST_DF = pd.DataFrame(
    PREPROCESSOR_MANIFEST_RECORDS
)

manifest_path = (
    schemas_root
    / "preprocessor_manifest.csv"
)

PREPROCESSOR_MANIFEST_DF.to_csv(
    manifest_path,
    index=False
)


# ==============================================================================
# 20.6 FINAL VERIFICATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    preprocessor_path = (
        preprocessors_root
        / dataset_id
        / "train_fitted_preprocessor.joblib"
    )

    schema_path = (
        schemas_root
        / dataset_id
        / "preprocessing_schema.json"
    )

    metadata_path = (
        metadata_root
        / f"{dataset_id}_preprocessing_metadata.json"
    )

    assert preprocessor_path.exists()
    assert schema_path.exists()
    assert metadata_path.exists()


print("\n" + "=" * 100)
print("SECTION 20 STATUS: PASS")
print("=" * 100)

print(
    "\n✓ Train-fitted preprocessors persisted."
)

print(
    "✓ Canonical preprocessing schemas persisted."
)

print(
    "✓ Canonical preprocessing metadata persisted."
)

print(
    "✓ All metadata paths validated."
)

20. PERSIST PREPROCESSORS, SCHEMAS & METADATA

----------------------------------------------------------------------------------------------------
PERSISTING : adult_income
----------------------------------------------------------------------------------------------------
  Preprocessor : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/preprocessors/adult_income/train_fitted_preprocessor.joblib
  Schema       : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/adult_income/preprocessing_schema.json
  Metadata     : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/metadata/adult_income_preprocessing_metadata.json

----------------------------------------------------------------------------------------------------
PERSISTING : bank_marketing
----------------------------------------------------------------------------------------------------
  Preprocessor : /content/drive/MyDrive/SPP_GAN_Research/data/processed/noteb

In [128]:
# ==============================================================================
# SECTION 21 — FEATURE MAPPING
# ==============================================================================
#
# Purpose:
#   Build an explicit mapping between:
#       1. Original modeling features
#       2. Numeric/categorical source features
#       3. Transformed encoded features
#
# Critical policy:
#   - Uses ONLY the fitted TRAIN preprocessor.
#   - Does NOT refit anything.
#   - Identifiers are excluded.
#   - __original_row_id__ is excluded.
#   - Target remains represented according to the modeling schema.
#   - Mapping is derived from the actual fitted ColumnTransformer.
#   - Transformer names are NOT assumed.
#
# Output:
#   feature_mapping.csv
#   feature_mapping_summary.csv
#   feature_mapping_metadata.json
#
# ==============================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd

print("=" * 100)
print("21. FEATURE MAPPING")
print("=" * 100)


# ==============================================================================
# 21.1 REQUIRED OBJECTS
# ==============================================================================

REQUIRED_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_PREPROCESSORS",
    "TRAIN_PREPROCESSING_COLUMNS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
    "NB02_DIRECTORIES",
]

missing_objects = [
    name for name in REQUIRED_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 21 cannot run.\n"
        f"Missing required objects: {missing_objects}"
    )

print("Required objects : PASS")


# ==============================================================================
# 21.2 FEATURE MAPPING ROOT
# ==============================================================================

FEATURE_MAPPING_ROOT = Path(
    NB02_DIRECTORIES["feature_mapping"]
)

FEATURE_MAPPING_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("\nFeature mapping root:")
print(f"  {FEATURE_MAPPING_ROOT}")


# ==============================================================================
# 21.3 RESET OUTPUT OBJECTS
# ==============================================================================

FEATURE_MAPPING_RECORDS = []
FEATURE_MAPPING_SUMMARY_RECORDS = []


# ==============================================================================
# 21.4 HELPER — LOCATE TRANSFORMER BY STRUCTURE
# ==============================================================================

def _find_pipeline_step(pipeline, step_type=None, step_name_contains=None):
    """
    Locate a step inside an sklearn Pipeline without assuming
    the user's exact step names.
    """

    if pipeline is None:
        return None, None

    # Pipeline-like object
    if hasattr(pipeline, "steps"):

        for name, step in pipeline.steps:

            if step_type is not None:
                try:
                    if isinstance(step, step_type):
                        return name, step
                except TypeError:
                    pass

            if (
                step_name_contains is not None
                and step_name_contains.lower() in str(name).lower()
            ):
                return name, step

    return None, None


def _find_categorical_encoder(column_transformer):
    """
    Find the fitted OneHotEncoder structurally.

    Does NOT assume that the ColumnTransformer transformer
    is named 'categorical', 'cat', etc.
    """

    # First inspect ColumnTransformer transformers_
    if not hasattr(column_transformer, "transformers_"):
        return None, None, None, None

    for transformer_name, transformer, columns in column_transformer.transformers_:

        if transformer_name == "remainder":
            continue

        if transformer == "drop":
            continue

        if transformer == "passthrough":
            continue

        # Direct OneHotEncoder
        if transformer.__class__.__name__ == "OneHotEncoder":
            return (
                transformer_name,
                transformer,
                columns,
                transformer,
            )

        # Pipeline containing OneHotEncoder
        if hasattr(transformer, "steps"):

            for step_name, step in transformer.steps:

                if step.__class__.__name__ == "OneHotEncoder":

                    return (
                        transformer_name,
                        transformer,
                        columns,
                        step,
                    )

    return None, None, None, None


def _find_numeric_transformer(column_transformer):
    """
    Find the numeric transformer structurally.
    """

    if not hasattr(column_transformer, "transformers_"):
        return None, None, None

    for transformer_name, transformer, columns in column_transformer.transformers_:

        if transformer_name == "remainder":
            continue

        if transformer == "drop":
            continue

        if transformer == "passthrough":
            continue

        # StandardScaler directly
        if transformer.__class__.__name__ == "StandardScaler":
            return transformer_name, transformer, columns

        # Pipeline containing StandardScaler
        if hasattr(transformer, "steps"):

            for step_name, step in transformer.steps:

                if step.__class__.__name__ == "StandardScaler":
                    return transformer_name, transformer, columns

    return None, None, None


# ==============================================================================
# 21.5 BUILD MAPPING DATASET BY DATASET
# ==============================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"BUILDING FEATURE MAPPING : {dataset_id}")
    print("-" * 100)

    # --------------------------------------------------------------------------
    # Retrieve schema
    # --------------------------------------------------------------------------

    schema = TRAIN_PREPROCESSING_COLUMNS[dataset_id]

    modeling_columns = list(
        schema["all_columns"]
    )

    numeric_columns = list(
        schema["numeric_columns"]
    )

    categorical_columns = list(
        schema["categorical_columns"]
    )

    target_column = TARGET_COLUMNS[dataset_id]

    identifier_columns = list(
        IDENTIFIER_COLUMNS.get(dataset_id, [])
    )

    provenance_column = schema.get(
        "provenance_column",
        "__original_row_id__"
    )

    expected_transformed_columns = list(
        schema.get("transformed_columns", [])
    )

    # --------------------------------------------------------------------------
    # Basic counts
    # --------------------------------------------------------------------------

    print(f"  Modeling features    : {len(modeling_columns)}")
    print(f"  Numeric features     : {len(numeric_columns)}")
    print(f"  Categorical features : {len(categorical_columns)}")
    print(f"  Transformed features : {len(expected_transformed_columns)}")

    # --------------------------------------------------------------------------
    # Validate modeling schema
    # --------------------------------------------------------------------------

    if provenance_column in modeling_columns:
        raise RuntimeError(
            f"{dataset_id}: provenance column appears in modeling schema."
        )

    identifier_overlap = set(modeling_columns).intersection(
        set(identifier_columns)
    )

    if identifier_overlap:
        raise RuntimeError(
            f"{dataset_id}: identifier columns appear in modeling schema: "
            f"{sorted(identifier_overlap)}"
        )

    if target_column not in modeling_columns:
        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' is missing "
            f"from modeling columns."
        )

    if not set(numeric_columns).issubset(set(modeling_columns)):
        raise RuntimeError(
            f"{dataset_id}: numeric columns are not a subset "
            f"of modeling columns."
        )

    if not set(categorical_columns).issubset(set(modeling_columns)):
        raise RuntimeError(
            f"{dataset_id}: categorical columns are not a subset "
            f"of modeling columns."
        )

    if set(numeric_columns).intersection(set(categorical_columns)):
        raise RuntimeError(
            f"{dataset_id}: numeric/categorical feature overlap detected."
        )

    # --------------------------------------------------------------------------
    # Retrieve fitted training-only preprocessor
    # --------------------------------------------------------------------------

    preprocessor = TRAIN_PREPROCESSORS[dataset_id]

    if preprocessor is None:
        raise RuntimeError(
            f"{dataset_id}: TRAIN_PREPROCESSORS entry is None."
        )

    if not hasattr(preprocessor, "transformers_"):
        raise RuntimeError(
            f"{dataset_id}: fitted ColumnTransformer not detected."
        )

    # --------------------------------------------------------------------------
    # Get actual transformed feature names
    # --------------------------------------------------------------------------

    actual_transformed_columns = list(
        preprocessor.get_feature_names_out()
    )

    if not actual_transformed_columns:
        raise RuntimeError(
            f"{dataset_id}: fitted preprocessor returned no "
            f"transformed feature names."
        )

    # If Section 16 schema did not retain the names, use fitted names.
    if not expected_transformed_columns:
        expected_transformed_columns = actual_transformed_columns

    if len(actual_transformed_columns) != len(expected_transformed_columns):

        raise RuntimeError(
            f"{dataset_id}: transformed feature count mismatch.\n"
            f"Schema : {len(expected_transformed_columns)}\n"
            f"Actual : {len(actual_transformed_columns)}"
        )

    # --------------------------------------------------------------------------
    # Locate transformers structurally
    # --------------------------------------------------------------------------

    categorical_transformer_name, categorical_transformer, \
        categorical_source_columns, categorical_encoder = \
        _find_categorical_encoder(preprocessor)

    if categorical_encoder is None:

        # Diagnostic information before failing
        print("\n  ColumnTransformer transformers_:")

        for name, transformer, columns in preprocessor.transformers_:

            print(
                f"    - name={name!r}, "
                f"type={type(transformer).__name__}, "
                f"columns={len(columns) if hasattr(columns, '__len__') else columns}"
            )

            if hasattr(transformer, "steps"):
                print(
                    f"      pipeline steps: "
                    f"{[step_name for step_name, _ in transformer.steps]}"
                )

        raise RuntimeError(
            f"{dataset_id}: fitted OneHotEncoder could not be located "
            f"structurally inside the ColumnTransformer."
        )

    print(
        f"  Categorical transformer : "
        f"{categorical_transformer_name}"
    )

    print(
        f"  Categorical encoder     : "
        f"{type(categorical_encoder).__name__}"
    )

    # --------------------------------------------------------------------------
    # Locate numeric transformer
    # --------------------------------------------------------------------------

    numeric_transformer_name, numeric_transformer, \
        numeric_source_columns = \
        _find_numeric_transformer(preprocessor)

    if numeric_transformer is None:

        raise RuntimeError(
            f"{dataset_id}: fitted numeric transformer could not "
            f"be located structurally."
        )

    print(
        f"  Numeric transformer     : "
        f"{numeric_transformer_name}"
    )

    # ==============================================================================
    # 21.6 BUILD NUMERIC FEATURE MAPPING
    # ==============================================================================

    numeric_position_map = {}

    for position, transformed_name in enumerate(actual_transformed_columns):

        if position >= len(actual_transformed_columns):
            continue

        # Numeric transformer produces one output per numeric source feature.
        # Match by original feature name.
        for source_column in numeric_columns:

            if transformed_name == source_column:

                numeric_position_map[source_column] = {
                    "position": position,
                    "transformed_name": transformed_name
                }

    # Some sklearn versions/pipelines may return prefixed names.
    # Add a robust fallback based on the numeric source ordering.
    unmatched_numeric = [
        col for col in numeric_columns
        if col not in numeric_position_map
    ]

    if unmatched_numeric:

        numeric_positions = []

        # Locate numeric transformer output region using transformer's
        # output feature names if available.
        try:
            numeric_feature_names = list(
                numeric_transformer.get_feature_names_out(
                    numeric_source_columns
                )
            )
        except Exception:
            numeric_feature_names = list(
                numeric_source_columns
            )

        for name in numeric_feature_names:

            matches = [
                idx
                for idx, actual_name
                in enumerate(actual_transformed_columns)
                if actual_name == name
            ]

            if matches:
                numeric_positions.append(
                    (name, matches[0])
                )

        for source_name, position in numeric_positions:

            # Match source feature directly where possible
            source_match = None

            if source_name in numeric_columns:
                source_match = source_name

            elif source_name.startswith(
                "num__"
            ):
                candidate = source_name.replace(
                    "num__",
                    "",
                    1
                )

                if candidate in numeric_columns:
                    source_match = candidate

            if source_match is not None:

                numeric_position_map[source_match] = {
                    "position": position,
                    "transformed_name":
                        actual_transformed_columns[position]
                }

    # ==============================================================================
    # 21.7 BUILD CATEGORICAL FEATURE MAPPING
    # ==============================================================================

    categorical_position_map = {}

    categories = getattr(
        categorical_encoder,
        "categories_",
        None
    )

    if categories is None:
        raise RuntimeError(
            f"{dataset_id}: fitted OneHotEncoder has no categories_."
        )

    encoder_input_columns = list(
        categorical_source_columns
    )

    if len(encoder_input_columns) != len(categories):

        raise RuntimeError(
            f"{dataset_id}: OneHotEncoder input/category mismatch.\n"
            f"Input columns : {len(encoder_input_columns)}\n"
            f"Categories    : {len(categories)}"
        )

    # Get encoder output names using the actual fitted encoder.
    try:
        encoder_feature_names = list(
            categorical_encoder.get_feature_names_out(
                encoder_input_columns
            )
        )
    except Exception:
        encoder_feature_names = []

    # Find each encoder output in the global transformed feature list.
    for local_position, transformed_name in enumerate(
        encoder_feature_names
    ):

        matches = [
            idx
            for idx, actual_name
            in enumerate(actual_transformed_columns)
            if actual_name == transformed_name
        ]

        if not matches:

            # Handle possible ColumnTransformer prefixes.
            matches = [
                idx
                for idx, actual_name
                in enumerate(actual_transformed_columns)
                if str(actual_name).endswith(
                    str(transformed_name)
                )
            ]

        if not matches:
            raise RuntimeError(
                f"{dataset_id}: transformed categorical feature "
                f"'{transformed_name}' could not be located."
            )

        global_position = matches[0]

        # Determine source feature.
        source_feature = None

        for source_column in encoder_input_columns:

            prefix = f"{source_column}_"

            if (
                transformed_name.startswith(prefix)
                or transformed_name.startswith(
                    f"{source_column}="
                )
            ):
                source_feature = source_column
                break

        # Robust fallback:
        # use category-output grouping from the encoder.
        if source_feature is None:

            cumulative = 0

            for source_column, source_categories in zip(
                encoder_input_columns,
                categories
            ):

                category_count = len(
                    source_categories
                )

                if (
                    cumulative
                    <= local_position
                    <
                    cumulative + category_count
                ):
                    source_feature = source_column
                    break

                cumulative += category_count

        if source_feature is None:
            raise RuntimeError(
                f"{dataset_id}: could not determine source categorical "
                f"feature for '{transformed_name}'."
            )

        category_value = None

        # Derive category value from encoder category metadata
        # where possible.
        source_index = encoder_input_columns.index(
            source_feature
        )

        source_categories = categories[source_index]

        # Determine local category position.
        local_output_start = 0

        for previous_index in range(source_index):
            previous_categories = categories[
                previous_index
            ]

            # Account for dropped categories.
            drop_idx = getattr(
                categorical_encoder,
                "drop_idx_",
                None
            )

            if (
                drop_idx is not None
                and drop_idx[previous_index] is not None
            ):
                local_output_start += (
                    len(previous_categories) - 1
                )
            else:
                local_output_start += len(
                    previous_categories
                )

        local_category_position = (
            local_position - local_output_start
        )

        drop_idx = getattr(
            categorical_encoder,
            "drop_idx_",
            None
        )

        available_categories = list(
            source_categories
        )

        if (
            drop_idx is not None
            and drop_idx[source_index] is not None
        ):
            available_categories = [
                value
                for idx, value
                in enumerate(source_categories)
                if idx != drop_idx[source_index]
            ]

        if (
            0 <= local_category_position
            < len(available_categories)
        ):
            category_value = available_categories[
                local_category_position
            ]

        categorical_position_map.setdefault(
            source_feature,
            []
        ).append(
            {
                "position": global_position,
                "transformed_name":
                    actual_transformed_columns[
                        global_position
                    ],
                "category": category_value
            }
        )

    # ==============================================================================
    # 21.8 CREATE NUMERIC MAPPING RECORDS
    # ==============================================================================

    for source_feature in numeric_columns:

        if source_feature not in numeric_position_map:

            raise RuntimeError(
                f"{dataset_id}: numeric feature "
                f"'{source_feature}' is not represented "
                f"in transformed output."
            )

        item = numeric_position_map[
            source_feature
        ]

        FEATURE_MAPPING_RECORDS.append(
            {
                "dataset_id": dataset_id,
                "source_feature": source_feature,
                "source_feature_type": "numeric",
                "source_category": None,
                "transformed_feature":
                    item["transformed_name"],
                "transformed_position":
                    int(item["position"]),
                "is_target":
                    bool(source_feature == target_column),
                "is_identifier": False,
                "is_provenance": False,
                "encoding": "numeric_scaled",
            }
        )

    # ==============================================================================
    # 21.9 CREATE CATEGORICAL MAPPING RECORDS
    # ==============================================================================

    for source_feature in categorical_columns:

        if source_feature not in categorical_position_map:

            raise RuntimeError(
                f"{dataset_id}: categorical feature "
                f"'{source_feature}' is not represented "
                f"in transformed output."
            )

        for item in categorical_position_map[
            source_feature
        ]:

            FEATURE_MAPPING_RECORDS.append(
                {
                    "dataset_id": dataset_id,
                    "source_feature": source_feature,
                    "source_feature_type": "categorical",
                    "source_category": item["category"],
                    "transformed_feature":
                        item["transformed_name"],
                    "transformed_position":
                        int(item["position"]),
                    "is_target":
                        bool(source_feature == target_column),
                    "is_identifier": False,
                    "is_provenance": False,
                    "encoding": "one_hot",
                }
            )

    # ==============================================================================
    # 21.10 DATASET-LEVEL VALIDATION
    # ==============================================================================

    dataset_mapping = [
        record
        for record in FEATURE_MAPPING_RECORDS
        if record["dataset_id"] == dataset_id
    ]

    mapping_positions = [
        record["transformed_position"]
        for record in dataset_mapping
    ]

    # Every transformed position must occur exactly once.
    if len(mapping_positions) != len(
        set(mapping_positions)
    ):
        duplicated_positions = sorted(
            pd.Series(mapping_positions)[
                pd.Series(mapping_positions).duplicated(
                    keep=False
                )
            ].unique()
        )

        raise RuntimeError(
            f"{dataset_id}: duplicate transformed positions detected: "
            f"{duplicated_positions}"
        )

    expected_positions = set(
        range(len(actual_transformed_columns))
    )

    actual_positions = set(
        mapping_positions
    )

    if actual_positions != expected_positions:

        missing_positions = sorted(
            expected_positions - actual_positions
        )

        extra_positions = sorted(
            actual_positions - expected_positions
        )

        raise RuntimeError(
            f"{dataset_id}: transformed position coverage mismatch.\n"
            f"Missing positions : {missing_positions[:20]}\n"
            f"Extra positions   : {extra_positions[:20]}"
        )

    # Every modeling feature must be represented.
    mapped_source_features = set(
        record["source_feature"]
        for record in dataset_mapping
    )

    if mapped_source_features != set(
        modeling_columns
    ):

        missing_features = sorted(
            set(modeling_columns)
            - mapped_source_features
        )

        extra_features = sorted(
            mapped_source_features
            - set(modeling_columns)
        )

        raise RuntimeError(
            f"{dataset_id}: source feature coverage mismatch.\n"
            f"Missing : {missing_features}\n"
            f"Extra   : {extra_features}"
        )

    # Identifiers and provenance must never appear.
    forbidden = (
        set(identifier_columns)
        | {provenance_column}
    )

    forbidden_found = (
        mapped_source_features
        & forbidden
    )

    if forbidden_found:
        raise RuntimeError(
            f"{dataset_id}: forbidden metadata features "
            f"found in mapping: {sorted(forbidden_found)}"
        )

    # Target must be represented.
    target_records = [
        record
        for record in dataset_mapping
        if record["source_feature"] == target_column
    ]

    if not target_records:
        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' "
            f"is not represented in feature mapping."
        )

    # Transformed count must match.
    if len(dataset_mapping) != len(
        actual_transformed_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: mapping row count mismatch.\n"
            f"Mapping rows : {len(dataset_mapping)}\n"
            f"Transformed  : {len(actual_transformed_columns)}"
        )

    # --------------------------------------------------------------------------
    # Dataset summary
    # --------------------------------------------------------------------------

    FEATURE_MAPPING_SUMMARY_RECORDS.append(
        {
            "dataset_id": dataset_id,
            "modeling_feature_count":
                len(modeling_columns),
            "numeric_feature_count":
                len(numeric_columns),
            "categorical_feature_count":
                len(categorical_columns),
            "transformed_feature_count":
                len(actual_transformed_columns),
            "mapping_row_count":
                len(dataset_mapping),
            "target_column":
                target_column,
            "target_mapped":
                True,
            "identifier_count_excluded":
                len(identifier_columns),
            "provenance_excluded":
                True,
            "position_coverage":
                True,
            "source_feature_coverage":
                True,
            "status":
                "PASS",
        }
    )

    print(
        f"  Mapping rows          : "
        f"{len(dataset_mapping)}"
    )

    print(
        f"  Position coverage     : PASS"
    )

    print(
        f"  Source feature coverage: PASS"
    )

    print(
        f"  Target mapping        : PASS"
    )

    print(
        f"  Metadata exclusion    : PASS"
    )


# ==============================================================================
# 21.11 CREATE DATAFRAMES
# ==============================================================================

FEATURE_MAPPING_DF = pd.DataFrame(
    FEATURE_MAPPING_RECORDS
)

FEATURE_MAPPING_SUMMARY_DF = pd.DataFrame(
    FEATURE_MAPPING_SUMMARY_RECORDS
)

if FEATURE_MAPPING_DF.empty:
    raise RuntimeError(
        "FEATURE_MAPPING_DF is empty."
    )

if FEATURE_MAPPING_SUMMARY_DF.empty:
    raise RuntimeError(
        "FEATURE_MAPPING_SUMMARY_DF is empty."
    )


# ==============================================================================
# 21.12 GLOBAL VALIDATION
# ==============================================================================

expected_dataset_count = len(
    DATASET_IDS
)

actual_dataset_count = (
    FEATURE_MAPPING_DF["dataset_id"]
    .nunique()
)

if actual_dataset_count != expected_dataset_count:

    raise RuntimeError(
        "Global dataset coverage mismatch.\n"
        f"Expected : {expected_dataset_count}\n"
        f"Actual   : {actual_dataset_count}"
    )

for dataset_id in DATASET_IDS:

    expected_count = len(
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]["transformed_columns"]
    )

    actual_count = int(
        (
            FEATURE_MAPPING_DF[
                "dataset_id"
            ] == dataset_id
        ).sum()
    )

    if actual_count != expected_count:

        raise RuntimeError(
            f"{dataset_id}: final mapping count mismatch.\n"
            f"Expected : {expected_count}\n"
            f"Actual   : {actual_count}"
        )


# ==============================================================================
# 21.13 SORT MAPPING
# ==============================================================================

FEATURE_MAPPING_DF = (
    FEATURE_MAPPING_DF
    .sort_values(
        by=[
            "dataset_id",
            "transformed_position"
        ]
    )
    .reset_index(drop=True)
)

FEATURE_MAPPING_SUMMARY_DF = (
    FEATURE_MAPPING_SUMMARY_DF
    .sort_values(
        by=["dataset_id"]
    )
    .reset_index(drop=True)
)


# ==============================================================================
# 21.14 SAVE FEATURE MAPPING
# ==============================================================================

FEATURE_MAPPING_CSV = (
    FEATURE_MAPPING_ROOT
    / "feature_mapping.csv"
)

FEATURE_MAPPING_SUMMARY_CSV = (
    FEATURE_MAPPING_ROOT
    / "feature_mapping_summary.csv"
)

FEATURE_MAPPING_METADATA_JSON = (
    FEATURE_MAPPING_ROOT
    / "feature_mapping_metadata.json"
)

FEATURE_MAPPING_DF.to_csv(
    FEATURE_MAPPING_CSV,
    index=False
)

FEATURE_MAPPING_SUMMARY_DF.to_csv(
    FEATURE_MAPPING_SUMMARY_CSV,
    index=False
)


# ==============================================================================
# 21.15 SAVE METADATA
# ==============================================================================

FEATURE_MAPPING_METADATA = {
    "artifact": "feature_mapping",
    "schema_version": "1.0",
    "creation_timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
    "dataset_ids":
        list(DATASET_IDS),
    "mapping_policy": {
        "fit_policy": "train_only",
        "preprocessor_source":
            "TRAIN_PREPROCESSORS",
        "identifier_columns_excluded":
            True,
        "provenance_column_excluded":
            True,
        "target_retained_in_modeling_schema":
            True,
        "numeric_encoding":
            "median_imputation_plus_standard_scaling",
        "categorical_encoding":
            "most_frequent_imputation_plus_one_hot_encoding",
        "transformer_names_inferred":
            True,
        "transformer_names_hardcoded":
            False,
    },
    "artifacts": {
        "feature_mapping_csv":
            str(FEATURE_MAPPING_CSV),
        "feature_mapping_summary_csv":
            str(FEATURE_MAPPING_SUMMARY_CSV),
        "feature_mapping_metadata_json":
            str(FEATURE_MAPPING_METADATA_JSON),
    },
    "dataset_summary":
        FEATURE_MAPPING_SUMMARY_DF.to_dict(
            orient="records"
        ),
}

with open(
    FEATURE_MAPPING_METADATA_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        FEATURE_MAPPING_METADATA,
        f,
        indent=2,
        ensure_ascii=False
    )


# ==============================================================================
# 21.16 ARTIFACT VALIDATION
# ==============================================================================

required_artifacts = {
    "feature_mapping":
        FEATURE_MAPPING_CSV,
    "feature_mapping_summary":
        FEATURE_MAPPING_SUMMARY_CSV,
    "feature_mapping_metadata":
        FEATURE_MAPPING_METADATA_JSON,
}

artifact_status = {}

for artifact_name, artifact_path in required_artifacts.items():

    exists = artifact_path.exists()

    artifact_status[
        artifact_name
    ] = exists

    print(
        f"  {artifact_name:30s}: "
        f"{'FOUND' if exists else 'MISSING'}"
    )

if not all(
    artifact_status.values()
):

    missing_artifacts = [
        name
        for name, exists
        in artifact_status.items()
        if not exists
    ]

    raise RuntimeError(
        "Section 21 artifact persistence failed.\n"
        f"Missing artifacts: {missing_artifacts}"
    )


# ==============================================================================
# 21.17 FINAL SECTION 21 VALIDATION
# ==============================================================================

print("\n" + "-" * 100)
print("SECTION 21 VALIDATION")
print("-" * 100)

print(
    f"Datasets mapped       : "
    f"{FEATURE_MAPPING_DF['dataset_id'].nunique()}"
)

print(
    f"Mapping rows          : "
    f"{len(FEATURE_MAPPING_DF)}"
)

print(
    f"Summary rows          : "
    f"{len(FEATURE_MAPPING_SUMMARY_DF)}"
)

print(
    f"Artifacts persisted   : "
    f"{all(artifact_status.values())}"
)

print(
    f"\nFEATURE MAPPING CSV:")
print(
    f"  {FEATURE_MAPPING_CSV}"
)

print(
    f"\nFEATURE MAPPING SUMMARY:")
print(
    f"  {FEATURE_MAPPING_SUMMARY_CSV}"
)

print(
    f"\nFEATURE MAPPING METADATA:")
print(
    f"  {FEATURE_MAPPING_METADATA_JSON}"
)

print("\n" + "=" * 100)
print("SECTION 21 STATUS: PASS")
print("=" * 100)

21. FEATURE MAPPING
Required objects : PASS

Feature mapping root:
  /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/feature_mapping

----------------------------------------------------------------------------------------------------
BUILDING FEATURE MAPPING : adult_income
----------------------------------------------------------------------------------------------------
  Modeling features    : 15
  Numeric features     : 6
  Categorical features : 9
  Transformed features : 107
  Categorical transformer : categorical
  Categorical encoder     : OneHotEncoder
  Numeric transformer     : numeric
  Mapping rows          : 107
  Position coverage     : PASS
  Source feature coverage: PASS
  Target mapping        : PASS
  Metadata exclusion    : PASS

----------------------------------------------------------------------------------------------------
BUILDING FEATURE MAPPING : bank_marketing
-----------------------------------------------------------------------------

In [129]:
# ==================================================================================================
# 22. SAVE SPLIT MANIFESTS AND VALIDATION REPORTS
# ==================================================================================================

from pathlib import Path
import json
import pandas as pd
from datetime import datetime

print("=" * 100)
print("22. SAVE SPLIT MANIFESTS AND VALIDATION REPORTS")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 22.0 Validate required canonical objects
# --------------------------------------------------------------------------------------------------

REQUIRED_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_DATASETS",
    "VALIDATION_DATASETS",
    "TEST_DATASETS",
    "SPLIT_MANIFESTS",
    "SPLIT_SUMMARY_DF",
    "TRANSFORMED_DATASET_SUMMARY_DF",
    "TRAIN_PREPROCESSING_COLUMNS",
    "TRAIN_PREPROCESSING_METADATA",
]

missing_objects = [
    obj for obj in REQUIRED_OBJECTS
    if obj not in globals()
]

if missing_objects:
    raise RuntimeError(
        "SECTION 22 CANNOT START.\n"
        f"Missing required canonical objects: {missing_objects}\n"
        "Run the required preceding sections before executing Section 22."
    )


# --------------------------------------------------------------------------------------------------
# 22.1 Resolve manifest directory
# --------------------------------------------------------------------------------------------------

if "manifests" not in NB02_DIRECTORIES:
    raise RuntimeError(
        "NB02_DIRECTORIES does not contain the required 'manifests' directory."
    )

MANIFEST_ROOT = Path(NB02_DIRECTORIES["manifests"])
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

print()
print(f"Manifest directory:")
print(f"  {MANIFEST_ROOT}")


# --------------------------------------------------------------------------------------------------
# 22.2 Build canonical combined split manifest
# --------------------------------------------------------------------------------------------------

manifest_records = []

for dataset_id in DATASET_IDS:

    if dataset_id not in SPLIT_MANIFESTS:
        raise RuntimeError(
            f"{dataset_id}: SPLIT_MANIFESTS entry not found."
        )

    manifest = SPLIT_MANIFESTS[dataset_id]

    # Expected structure from Section 13:
    # dataset_id / split / original_row_id

    if isinstance(manifest, pd.DataFrame):

        manifest_df = manifest.copy()

        # Normalize likely column names without changing the underlying architecture.
        rename_map = {}

        if "__original_row_id__" not in manifest_df.columns:

            for candidate in [
                "original_row_id",
                "row_id",
                "original_id",
            ]:
                if candidate in manifest_df.columns:
                    rename_map[candidate] = "__original_row_id__"
                    break

        if "dataset_id" not in manifest_df.columns:
            manifest_df["dataset_id"] = dataset_id

        if rename_map:
            manifest_df = manifest_df.rename(columns=rename_map)

        manifest_records.append(manifest_df)

    elif isinstance(manifest, dict):

        for split_name, split_ids in manifest.items():

            if isinstance(split_ids, (list, tuple, pd.Series)):

                for row_id in split_ids:

                    manifest_records.append(
                        {
                            "dataset_id": dataset_id,
                            "split": split_name,
                            "__original_row_id__": row_id,
                        }
                    )

            else:
                raise RuntimeError(
                    f"{dataset_id}: unsupported SPLIT_MANIFESTS structure "
                    f"for split '{split_name}'."
                )

    else:
        raise RuntimeError(
            f"{dataset_id}: unsupported SPLIT_MANIFESTS type: "
            f"{type(manifest).__name__}"
        )


if not manifest_records:
    raise RuntimeError(
        "No split manifest records were generated."
    )


if all(isinstance(x, pd.DataFrame) for x in manifest_records):
    SPLIT_MANIFEST_FINAL_DF = pd.concat(
        manifest_records,
        ignore_index=True
    )
else:
    SPLIT_MANIFEST_FINAL_DF = pd.DataFrame(manifest_records)


# --------------------------------------------------------------------------------------------------
# 22.3 Normalize and validate split manifest columns
# --------------------------------------------------------------------------------------------------

required_manifest_columns = {
    "dataset_id",
    "split",
    "__original_row_id__",
}

missing_manifest_columns = (
    required_manifest_columns
    - set(SPLIT_MANIFEST_FINAL_DF.columns)
)

if missing_manifest_columns:
    raise RuntimeError(
        "Split manifest is missing required columns: "
        f"{sorted(missing_manifest_columns)}"
    )


SPLIT_MANIFEST_FINAL_DF = (
    SPLIT_MANIFEST_FINAL_DF[
        [
            "dataset_id",
            "split",
            "__original_row_id__",
        ]
    ]
    .copy()
)


SPLIT_MANIFEST_FINAL_DF["__original_row_id__"] = (
    pd.to_numeric(
        SPLIT_MANIFEST_FINAL_DF["__original_row_id__"],
        errors="raise"
    )
    .astype("int64")
)


# --------------------------------------------------------------------------------------------------
# 22.4 Validate split labels
# --------------------------------------------------------------------------------------------------

EXPECTED_SPLITS = {
    "train",
    "validation",
    "test",
}

observed_splits = set(
    SPLIT_MANIFEST_FINAL_DF["split"].astype(str).unique()
)

unexpected_splits = observed_splits - EXPECTED_SPLITS

if unexpected_splits:
    raise RuntimeError(
        f"Unexpected split labels found: {sorted(unexpected_splits)}"
    )


# --------------------------------------------------------------------------------------------------
# 22.5 Validate dataset coverage
# --------------------------------------------------------------------------------------------------

observed_datasets = set(
    SPLIT_MANIFEST_FINAL_DF["dataset_id"].astype(str).unique()
)

expected_datasets = set(DATASET_IDS)

if observed_datasets != expected_datasets:
    raise RuntimeError(
        "Dataset coverage mismatch in split manifest.\n"
        f"Expected: {sorted(expected_datasets)}\n"
        f"Observed: {sorted(observed_datasets)}"
    )


# --------------------------------------------------------------------------------------------------
# 22.6 Validate provenance uniqueness
# --------------------------------------------------------------------------------------------------

duplicate_provenance = (
    SPLIT_MANIFEST_FINAL_DF
    .duplicated(
        subset=[
            "dataset_id",
            "__original_row_id__",
        ],
        keep=False
    )
)

if duplicate_provenance.any():

    duplicate_rows = int(duplicate_provenance.sum())

    raise RuntimeError(
        "Split manifest contains duplicated original-row provenance.\n"
        f"Duplicate rows: {duplicate_rows}"
    )


# --------------------------------------------------------------------------------------------------
# 22.7 Validate train/validation/test disjointness
# --------------------------------------------------------------------------------------------------

integrity_records = []

for dataset_id in DATASET_IDS:

    dataset_manifest = SPLIT_MANIFEST_FINAL_DF[
        SPLIT_MANIFEST_FINAL_DF["dataset_id"] == dataset_id
    ]

    split_sets = {}

    for split_name in EXPECTED_SPLITS:

        ids = set(
            dataset_manifest.loc[
                dataset_manifest["split"] == split_name,
                "__original_row_id__"
            ].tolist()
        )

        split_sets[split_name] = ids

    train_ids = split_sets["train"]
    validation_ids = split_sets["validation"]
    test_ids = split_sets["test"]

    train_val_overlap = train_ids & validation_ids
    train_test_overlap = train_ids & test_ids
    validation_test_overlap = validation_ids & test_ids

    passed = (
        len(train_val_overlap) == 0
        and len(train_test_overlap) == 0
        and len(validation_test_overlap) == 0
    )

    integrity_records.append(
        {
            "dataset_id": dataset_id,
            "train_rows": len(train_ids),
            "validation_rows": len(validation_ids),
            "test_rows": len(test_ids),
            "train_validation_overlap": len(train_val_overlap),
            "train_test_overlap": len(train_test_overlap),
            "validation_test_overlap": len(validation_test_overlap),
            "integrity_status": "PASS" if passed else "FAIL",
        }
    )

    if not passed:
        raise RuntimeError(
            f"{dataset_id}: split provenance overlap detected."
        )


SPLIT_INTEGRITY_FINAL_DF = pd.DataFrame(integrity_records)


# --------------------------------------------------------------------------------------------------
# 22.8 Validate against canonical split datasets
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    expected_counts = {
        "train": len(TRAIN_DATASETS[dataset_id]),
        "validation": len(VALIDATION_DATASETS[dataset_id]),
        "test": len(TEST_DATASETS[dataset_id]),
    }

    for split_name, expected_count in expected_counts.items():

        actual_count = int(
            (
                SPLIT_MANIFEST_FINAL_DF["dataset_id"].eq(dataset_id)
                &
                SPLIT_MANIFEST_FINAL_DF["split"].eq(split_name)
            ).sum()
        )

        if actual_count != expected_count:

            raise RuntimeError(
                f"{dataset_id} | {split_name}: manifest row count mismatch.\n"
                f"Expected: {expected_count}\n"
                f"Observed: {actual_count}"
            )


# --------------------------------------------------------------------------------------------------
# 22.9 Use canonical Section 13 split summary
# --------------------------------------------------------------------------------------------------

if not isinstance(SPLIT_SUMMARY_DF, pd.DataFrame):
    raise RuntimeError(
        "SPLIT_SUMMARY_DF is not a pandas DataFrame."
    )

SPLIT_SUMMARY_FINAL_DF = SPLIT_SUMMARY_DF.copy()


# --------------------------------------------------------------------------------------------------
# 22.10 Use canonical Section 17 transformation summary
# --------------------------------------------------------------------------------------------------

if not isinstance(
    TRANSFORMED_DATASET_SUMMARY_DF,
    pd.DataFrame
):
    raise RuntimeError(
        "TRANSFORMED_DATASET_SUMMARY_DF is not a pandas DataFrame."
    )

TRANSFORMATION_FINAL_DF = (
    TRANSFORMED_DATASET_SUMMARY_DF.copy()
)


# --------------------------------------------------------------------------------------------------
# 22.11 Build preprocessing schema report
# --------------------------------------------------------------------------------------------------

preprocessing_records = []

for dataset_id in DATASET_IDS:

    schema = TRAIN_PREPROCESSING_COLUMNS[dataset_id]
    metadata = TRAIN_PREPROCESSING_METADATA[dataset_id]

    preprocessing_records.append(
        {
            "dataset_id": dataset_id,
            "modeling_columns": len(schema["all_columns"]),
            "numeric_columns": len(schema["numeric_columns"]),
            "categorical_columns": len(schema["categorical_columns"]),
            "target_column": schema["target_column"],
            "identifier_columns": len(
                schema["identifier_columns"]
            ),
            "transformed_features": len(
                schema["transformed_columns"]
            ),
            "fit_dataset": metadata.get(
                "fit_dataset",
                "train"
            ),
        }
    )

PREPROCESSING_SCHEMA_SUMMARY_DF = pd.DataFrame(
    preprocessing_records
)


# --------------------------------------------------------------------------------------------------
# 22.12 Save split manifest
# --------------------------------------------------------------------------------------------------

split_manifest_path = (
    MANIFEST_ROOT / "split_manifest.csv"
)

SPLIT_MANIFEST_FINAL_DF.to_csv(
    split_manifest_path,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 22.13 Save split integrity
# --------------------------------------------------------------------------------------------------

split_integrity_path = (
    MANIFEST_ROOT / "split_integrity.csv"
)

SPLIT_INTEGRITY_FINAL_DF.to_csv(
    split_integrity_path,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 22.14 Save split summary
# --------------------------------------------------------------------------------------------------

split_summary_path = (
    MANIFEST_ROOT / "split_summary.csv"
)

SPLIT_SUMMARY_FINAL_DF.to_csv(
    split_summary_path,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 22.15 Save transformation report
# --------------------------------------------------------------------------------------------------

transformation_path = (
    MANIFEST_ROOT / "transformation_report.csv"
)

TRANSFORMATION_FINAL_DF.to_csv(
    transformation_path,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 22.16 Save preprocessing schema summary
# --------------------------------------------------------------------------------------------------

preprocessing_summary_path = (
    MANIFEST_ROOT / "preprocessing_schema_summary.csv"
)

PREPROCESSING_SCHEMA_SUMMARY_DF.to_csv(
    preprocessing_summary_path,
    index=False
)


# --------------------------------------------------------------------------------------------------
# 22.17 Save Section 22 metadata
# --------------------------------------------------------------------------------------------------

section_22_metadata = {
    "section": "22. SAVE SPLIT MANIFESTS AND VALIDATION REPORTS",
    "timestamp": datetime.now().isoformat(),
    "datasets": list(DATASET_IDS),
    "total_manifest_rows": int(
        len(SPLIT_MANIFEST_FINAL_DF)
    ),
    "manifest_columns": list(
        SPLIT_MANIFEST_FINAL_DF.columns
    ),
    "split_integrity_status": "PASS",
    "provenance_unique_within_dataset": True,
    "train_validation_disjoint": True,
    "train_test_disjoint": True,
    "validation_test_disjoint": True,
    "transformation_report_source": (
        "TRANSFORMED_DATASET_SUMMARY_DF"
    ),
    "preprocessing_fit_policy": "TRAIN_ONLY",
}

section_22_metadata_path = (
    MANIFEST_ROOT / "section_22_metadata.json"
)

with open(
    section_22_metadata_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        section_22_metadata,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------------------------------------------------------
# 22.18 Verify saved artifacts
# --------------------------------------------------------------------------------------------------

saved_artifacts = {
    "split_manifest": split_manifest_path,
    "split_integrity": split_integrity_path,
    "split_summary": split_summary_path,
    "transformation_report": transformation_path,
    "preprocessing_schema_summary": preprocessing_summary_path,
    "section_22_metadata": section_22_metadata_path,
}

missing_files = [
    str(path)
    for path in saved_artifacts.values()
    if not Path(path).exists()
]

if missing_files:
    raise RuntimeError(
        "Section 22 artifact verification failed.\n"
        f"Missing files: {missing_files}"
    )


# --------------------------------------------------------------------------------------------------
# 22.19 Final Section 22 verification
# --------------------------------------------------------------------------------------------------

if not (
    SPLIT_INTEGRITY_FINAL_DF["integrity_status"]
    .eq("PASS")
    .all()
):
    raise RuntimeError(
        "Section 22 split integrity verification failed."
    )


print()
print("=" * 100)
print("SECTION 22 SUMMARY")
print("=" * 100)

print(
    f"Datasets                    : {len(DATASET_IDS)}"
)

print(
    f"Split manifest rows         : "
    f"{len(SPLIT_MANIFEST_FINAL_DF):,}"
)

print(
    f"Split integrity records     : "
    f"{len(SPLIT_INTEGRITY_FINAL_DF):,}"
)

print(
    f"Transformation records      : "
    f"{len(TRANSFORMATION_FINAL_DF):,}"
)

print()
print("Saved artifacts:")

for name, path in saved_artifacts.items():
    print(f"  ✓ {name}: {path}")

print()
print("Provenance uniqueness       : PASS")
print("Train/Validation disjoint   : PASS")
print("Train/Test disjoint         : PASS")
print("Validation/Test disjoint    : PASS")
print("Manifest row-count checks   : PASS")
print("Artifact existence checks   : PASS")

print()
print("SECTION 22 STATUS: PASS")
print("=" * 100)

22. SAVE SPLIT MANIFESTS AND VALIDATION REPORTS

Manifest directory:
  /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/notebook_02_manifests

SECTION 22 SUMMARY
Datasets                    : 3
Split manifest rows         : 195,819
Split integrity records     : 3
Transformation records      : 3

Saved artifacts:
  ✓ split_manifest: /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/notebook_02_manifests/split_manifest.csv
  ✓ split_integrity: /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/notebook_02_manifests/split_integrity.csv
  ✓ split_summary: /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/notebook_02_manifests/split_summary.csv
  ✓ transformation_report: /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/notebook_02_manifests/transformation_report.csv
  ✓ preprocessing_schema_summary: /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/notebook_02_manifests/preprocessing_schema_summary.csv
  ✓ sectio

In [130]:
# ==============================================================================
# SECTION 23 — RELOAD & VALIDATE ARTIFACTS
# ==============================================================================
# Purpose:
#   Reload and validate all persisted Notebook 02 preprocessing artifacts.
#
# IMPORTANT:
#   TRANSFORMED_TRAIN_DATASETS / VALIDATION / TEST may be NumPy arrays.
#   Therefore this section NEVER assumes a `.columns` attribute for transformed
#   datasets.
#
#   Canonical feature names are validated against:
#       TRAIN_PREPROCESSING_COLUMNS
#       reloaded ColumnTransformer.get_feature_names_out()
#
#   This section does NOT modify any preprocessing architecture.
# ==============================================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.compose import ColumnTransformer


print("=" * 100)
print("23. RELOAD & VALIDATE ARTIFACTS")
print("=" * 100)


# ==============================================================================
# 23.1 — REQUIRED CANONICAL OBJECTS
# ==============================================================================

_REQUIRED_OBJECTS = [
    "DATASET_IDS",
    "TRAIN_DATASETS",
    "VALIDATION_DATASETS",
    "TEST_DATASETS",
    "TRAIN_PREPROCESSORS",
    "TRAIN_PREPROCESSING_COLUMNS",
    "TRAIN_PREPROCESSING_METADATA",
    "TRANSFORMED_TRAIN_DATASETS",
    "TRANSFORMED_VALIDATION_DATASETS",
    "TRANSFORMED_TEST_DATASETS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",
    "NB02_DIRECTORIES",
]

_missing_objects = [
    name for name in _REQUIRED_OBJECTS
    if name not in globals()
]

if _missing_objects:
    raise RuntimeError(
        "SECTION 23 cannot run because required canonical objects are missing:\n"
        + "\n".join(f"  - {x}" for x in _missing_objects)
    )

print("Required canonical objects: PASS")


# ==============================================================================
# 23.2 — ARTIFACT ROOTS
# ==============================================================================

PREPROCESSOR_ROOT = Path(
    NB02_DIRECTORIES["preprocessors"]
)

SCHEMA_ROOT = Path(
    NB02_DIRECTORIES["schemas"]
)

METADATA_ROOT = (
    SCHEMA_ROOT / "metadata"
)

print(f"Preprocessor root : {PREPROCESSOR_ROOT}")
print(f"Schema root       : {SCHEMA_ROOT}")
print(f"Metadata root     : {METADATA_ROOT}")


# ==============================================================================
# 23.3 — CONSTANTS / HELPERS
# ==============================================================================

PROVENANCE_COLUMN = "__original_row_id__"


def safe_list(value):
    """
    Convert common serialized/in-memory sequence types to Python list.
    """
    if value is None:
        return []

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, pd.Index):
        return value.tolist()

    if isinstance(value, tuple):
        return list(value)

    if isinstance(value, list):
        return value

    return [value]


def first_existing(mapping, keys, default=None):
    """
    Return the first available key from a dictionary.
    """
    if not isinstance(mapping, dict):
        return default

    for key in keys:
        if key in mapping:
            return mapping[key]

    return default


def same_sequence(expected, actual):
    """
    Exact ordered sequence comparison.
    """
    return list(expected) == list(actual)


def is_fitted_column_transformer(preprocessor):
    """
    Validate that the reloaded object is fitted.
    """
    if not isinstance(
        preprocessor,
        ColumnTransformer
    ):
        return False

    if not hasattr(
        preprocessor,
        "transformers_"
    ):
        return False

    try:
        names = preprocessor.get_feature_names_out()
        return len(names) > 0
    except Exception:
        return False


def resolve_fit_scope(metadata, schema):
    """
    Resolve persisted train-only fitting declaration.
    """

    candidates = []

    if isinstance(metadata, dict):

        candidates.extend([
            metadata.get("fit_dataset"),
            metadata.get("fit_scope"),
            metadata.get("fit_policy"),
        ])

    if isinstance(schema, dict):

        candidates.extend([
            schema.get("fit_dataset"),
            schema.get("fit_scope"),
            schema.get("fit_policy"),
        ])

    for value in candidates:

        if isinstance(value, str):

            value_lower = value.lower()

            if (
                "train_only" in value_lower
                or "train-only" in value_lower
                or value_lower == "train"
            ):
                return value

        elif isinstance(value, dict):

            text = str(value).lower()

            if (
                "train_only" in text
                or "train-only" in text
            ):
                return text

    return None


def array_shape(value):
    """
    Safely obtain shape from NumPy array / DataFrame.
    """
    if hasattr(value, "shape"):
        return tuple(value.shape)

    return None


# ==============================================================================
# 23.4 — VALIDATION STORAGE
# ==============================================================================

validation_records = []

failed_datasets = []


# ==============================================================================
# 23.5 — DATASET-BY-DATASET VALIDATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    # --------------------------------------------------------------------------
    # Artifact paths
    # --------------------------------------------------------------------------

    preprocessor_path = (
        PREPROCESSOR_ROOT
        / dataset_id
        / "train_fitted_preprocessor.joblib"
    )

    schema_path = (
        SCHEMA_ROOT
        / dataset_id
        / "preprocessing_schema.json"
    )

    metadata_path = (
        METADATA_ROOT
        / dataset_id
        / "preprocessing_metadata.json"
    )

    preprocessor_exists = (
        preprocessor_path.exists()
    )

    schema_exists = (
        schema_path.exists()
    )

    metadata_exists = (
        metadata_path.exists()
    )

    print(
        "Preprocessor artifact: "
        + ("FOUND" if preprocessor_exists else "MISSING")
    )

    print(
        "Schema artifact      : "
        + ("FOUND" if schema_exists else "MISSING")
    )

    print(
        "Metadata artifact    : "
        + ("FOUND" if metadata_exists else "MISSING")
    )


    # --------------------------------------------------------------------------
    # Initialize flags
    # --------------------------------------------------------------------------

    preprocessor_type_valid = False
    preprocessor_fitted_valid = False

    schema_dataset_valid = False
    metadata_dataset_valid = False

    modeling_schema_valid = False

    modeling_columns_match = False
    numeric_schema_match = False
    categorical_schema_match = False
    target_schema_match = False
    identifier_schema_match = False
    provenance_schema_match = False

    transformed_schema_match = False

    input_feature_count_match = False
    transformed_feature_count_match = False
    feature_names_match = False
    fitted_input_schema_match = False

    train_only_fit_valid = False

    identifier_exclusion_valid = False
    provenance_exclusion_valid = False

    # --------------------------------------------------------------------------
    # Reload artifacts
    # --------------------------------------------------------------------------

    reloaded_preprocessor = None
    reloaded_schema = None
    reloaded_metadata = None

    if preprocessor_exists:

        try:

            reloaded_preprocessor = joblib.load(
                preprocessor_path
            )

        except Exception as exc:

            print(
                f"Preprocessor reload: FAIL — {exc}"
            )

    if schema_exists:

        try:

            with open(
                schema_path,
                "r",
                encoding="utf-8"
            ) as f:

                reloaded_schema = json.load(f)

        except Exception as exc:

            print(
                f"Schema reload: FAIL — {exc}"
            )

    if metadata_exists:

        try:

            with open(
                metadata_path,
                "r",
                encoding="utf-8"
            ) as f:

                reloaded_metadata = json.load(f)

        except Exception as exc:

            print(
                f"Metadata reload: FAIL — {exc}"
            )


    # ==============================================================================
    # 23.6 — PREPROCESSOR VALIDATION
    # ==============================================================================

    preprocessor_type_valid = isinstance(
        reloaded_preprocessor,
        ColumnTransformer
    )

    print(
        f"Preprocessor type        : "
        f"{'PASS' if preprocessor_type_valid else 'FAIL'}"
    )


    preprocessor_fitted_valid = (
        is_fitted_column_transformer(
            reloaded_preprocessor
        )
    )

    print(
        f"Preprocessor fitted      : "
        f"{'PASS' if preprocessor_fitted_valid else 'FAIL'}"
    )


    # ==============================================================================
    # 23.7 — BASIC SCHEMA / METADATA VALIDATION
    # ==============================================================================

    if isinstance(
        reloaded_schema,
        dict
    ):

        schema_dataset_valid = (
            reloaded_schema.get("dataset_id")
            == dataset_id
        )

    if isinstance(
        reloaded_metadata,
        dict
    ):

        metadata_dataset_valid = (
            reloaded_metadata.get("dataset_id")
            == dataset_id
        )

    print(
        f"Schema                   : "
        f"{'PASS' if schema_dataset_valid else 'FAIL'}"
    )

    print(
        f"Metadata                 : "
        f"{'PASS' if metadata_dataset_valid else 'FAIL'}"
    )


    # ==============================================================================
    # 23.8 — CANONICAL IN-MEMORY SCHEMA
    # ==============================================================================

    expected_schema = (
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]
    )

    expected_modeling_columns = list(
        expected_schema["all_columns"]
    )

    expected_numeric_columns = list(
        expected_schema["numeric_columns"]
    )

    expected_categorical_columns = list(
        expected_schema["categorical_columns"]
    )

    expected_target_column = (
        expected_schema["target_column"]
    )

    expected_identifier_columns = list(
        expected_schema["identifier_columns"]
    )

    expected_provenance_column = (
        expected_schema["provenance_column"]
    )

    expected_transformed_columns = list(
        expected_schema["transformed_columns"]
    )

    expected_input_count = len(
        expected_modeling_columns
    )

    expected_transformed_count = len(
        expected_transformed_columns
    )


    # ==============================================================================
    # 23.9 — SERIALIZED SCHEMA STRUCTURE
    # ==============================================================================

    serialized_modeling_schema = {}
    serialized_transformation_schema = {}
    serialized_preprocessing = {}

    if isinstance(
        reloaded_schema,
        dict
    ):

        serialized_modeling_schema = (
            reloaded_schema.get(
                "modeling_schema",
                {}
            )
        )

        serialized_transformation_schema = (
            reloaded_schema.get(
                "transformation_schema",
                {}
            )
        )

        serialized_preprocessing = (
            reloaded_schema.get(
                "preprocessing",
                {}
            )
        )

    print("\nSchema structure:")

    if isinstance(
        reloaded_schema,
        dict
    ):

        print(
            "  Top-level keys:",
            list(
                reloaded_schema.keys()
            )
        )

        print(
            "  modeling_schema keys:",
            list(
                serialized_modeling_schema.keys()
            )
            if isinstance(
                serialized_modeling_schema,
                dict
            )
            else []
        )

        print(
            "  transformation_schema keys:",
            list(
                serialized_transformation_schema.keys()
            )
            if isinstance(
                serialized_transformation_schema,
                dict
            )
            else []
        )

        print(
            "  preprocessing keys:",
            list(
                serialized_preprocessing.keys()
            )
            if isinstance(
                serialized_preprocessing,
                dict
            )
            else []
        )


    # ==============================================================================
    # 23.10 — SERIALIZED MODELING SCHEMA
    # ==============================================================================

    saved_modeling_columns = first_existing(
        serialized_modeling_schema,
        [
            "all_columns",
            "modeling_columns",
            "columns",
            "features",
        ],
        default=[],
    )

    saved_numeric_columns = first_existing(
        serialized_modeling_schema,
        [
            "numeric_columns",
            "numeric_features",
        ],
        default=[],
    )

    saved_categorical_columns = first_existing(
        serialized_modeling_schema,
        [
            "categorical_columns",
            "categorical_features",
        ],
        default=[],
    )

    saved_identifier_columns = first_existing(
        serialized_modeling_schema,
        [
            "identifier_columns_excluded",
            "identifier_columns",
            "identifiers",
        ],
        default=[],
    )

    saved_target_column = first_existing(
        serialized_modeling_schema,
        [
            "target_column",
            "target",
        ],
        default=None,
    )

    saved_provenance_column = first_existing(
        serialized_modeling_schema,
        [
            "provenance_column",
            "provenance",
        ],
        default=None,
    )


    saved_modeling_columns = safe_list(
        saved_modeling_columns
    )

    saved_numeric_columns = safe_list(
        saved_numeric_columns
    )

    saved_categorical_columns = safe_list(
        saved_categorical_columns
    )

    saved_identifier_columns = safe_list(
        saved_identifier_columns
    )


    # ==============================================================================
    # 23.11 — MODELING SCHEMA COMPARISON
    # ==============================================================================

    modeling_columns_match = same_sequence(
        expected_modeling_columns,
        saved_modeling_columns
    )

    numeric_schema_match = same_sequence(
        expected_numeric_columns,
        saved_numeric_columns
    )

    categorical_schema_match = same_sequence(
        expected_categorical_columns,
        saved_categorical_columns
    )

    target_schema_match = (
        expected_target_column
        == saved_target_column
    )

    identifier_schema_match = same_sequence(
        expected_identifier_columns,
        saved_identifier_columns
    )

    provenance_schema_match = (
        expected_provenance_column
        == saved_provenance_column
    )

    modeling_schema_valid = all([
        modeling_columns_match,
        numeric_schema_match,
        categorical_schema_match,
        target_schema_match,
        identifier_schema_match,
        provenance_schema_match,
    ])


    # ==============================================================================
    # 23.12 — TRANSFORMED SCHEMA COMPARISON
    # ==============================================================================

    saved_transformed_count = first_existing(
        serialized_transformation_schema,
        [
            "transformed_feature_count",
            "feature_count",
            "transformed_count",
        ],
        default=0,
    )

    saved_transformed_columns = first_existing(
        serialized_transformation_schema,
        [
            "transformed_feature_names",
            "transformed_columns",
            "feature_names",
        ],
        default=[],
    )

    try:

        saved_transformed_count = int(
            saved_transformed_count
        )

    except Exception:

        saved_transformed_count = 0

    saved_transformed_columns = safe_list(
        saved_transformed_columns
    )

    transformed_schema_match = (
        saved_transformed_count
        == expected_transformed_count
        and
        same_sequence(
            expected_transformed_columns,
            saved_transformed_columns
        )
    )


    # ==============================================================================
    # 23.13 — PREPROCESSOR INPUT FEATURE COUNT
    # ==============================================================================

    actual_input_feature_count = None

    fitted_input_columns = []

    if reloaded_preprocessor is not None:

        try:

            fitted_input_columns = list(
                reloaded_preprocessor.feature_names_in_
            )

            actual_input_feature_count = len(
                fitted_input_columns
            )

        except Exception:

            fitted_input_columns = []
            actual_input_feature_count = None

    input_feature_count_match = (
        actual_input_feature_count
        == expected_input_count
    )

    print(
        f"Input feature count      : "
        f"{'PASS' if input_feature_count_match else 'FAIL'} "
        f"({actual_input_feature_count}/"
        f"{expected_input_count})"
    )


    # ==============================================================================
    # 23.14 — PREPROCESSOR TRANSFORMED FEATURE COUNT
    # ==============================================================================

    actual_transformed_names = []
    actual_transformed_count = None

    if reloaded_preprocessor is not None:

        try:

            actual_transformed_names = list(
                reloaded_preprocessor
                .get_feature_names_out()
            )

            actual_transformed_count = len(
                actual_transformed_names
            )

        except Exception:

            actual_transformed_names = []
            actual_transformed_count = None

    transformed_feature_count_match = (
        actual_transformed_count
        == expected_transformed_count
    )

    print(
        f"Transformed feature count: "
        f"{'PASS' if transformed_feature_count_match else 'FAIL'} "
        f"({actual_transformed_count}/"
        f"{expected_transformed_count})"
    )


    # ==============================================================================
    # 23.15 — FEATURE NAME VALIDATION
    # ==============================================================================

    feature_names_match = (
        actual_transformed_names
        == expected_transformed_columns
    )

    print(
        f"Feature names            : "
        f"{'PASS' if feature_names_match else 'FAIL'}"
    )


    # ==============================================================================
    # 23.16 — FITTED INPUT SCHEMA VALIDATION
    # ==============================================================================

    fitted_input_schema_match = (
        fitted_input_columns
        == expected_modeling_columns
    )

    print(
        f"Fitted input schema      : "
        f"{'PASS' if fitted_input_schema_match else 'FAIL'}"
    )


    # ==============================================================================
    # 23.17 — TRAIN-ONLY FITTING VALIDATION
    # ==============================================================================

    fit_scope = resolve_fit_scope(
        reloaded_metadata,
        reloaded_schema
    )

    train_only_fit_valid = (
        fit_scope is not None
        and
        "train" in str(
            fit_scope
        ).lower()
    )

    print(
        f"Train-only fitting       : "
        f"{'PASS' if train_only_fit_valid else 'FAIL'}"
    )


    # ==============================================================================
    # 23.18 — IDENTIFIER EXCLUSION
    # ==============================================================================
    # IMPORTANT:
    #   Transformed datasets may be NumPy arrays.
    #   Therefore do NOT access `.columns`.
    #
    #   Identifier exclusion is validated through:
    #       1. canonical modeling schema
    #       2. fitted input schema
    #       3. transformed feature names
    # ==============================================================================

    identifiers = list(
        IDENTIFIER_COLUMNS[dataset_id]
    )

    # Identifiers must not be modeling inputs.
    identifiers_absent_from_modeling = all(
        identifier not in expected_modeling_columns
        for identifier in identifiers
    )

    # Identifiers must not be fitted preprocessor inputs.
    identifiers_absent_from_fitted_input = all(
        identifier not in fitted_input_columns
        for identifier in identifiers
    )

    # Identifiers must not appear as raw transformed feature names.
    identifiers_absent_from_transformed = all(
        identifier not in actual_transformed_names
        for identifier in identifiers
    )

    identifier_exclusion_valid = all([
        identifiers_absent_from_modeling,
        identifiers_absent_from_fitted_input,
        identifiers_absent_from_transformed,
    ])

    print(
        f"Identifier exclusion     : "
        f"{'PASS' if identifier_exclusion_valid else 'FAIL'}"
    )


    # ==============================================================================
    # 23.19 — PROVENANCE EXCLUSION
    # ==============================================================================

    provenance_absent_from_modeling = (
        PROVENANCE_COLUMN
        not in expected_modeling_columns
    )

    provenance_absent_from_fitted_input = (
        PROVENANCE_COLUMN
        not in fitted_input_columns
    )

    provenance_absent_from_transformed = (
        PROVENANCE_COLUMN
        not in actual_transformed_names
    )

    provenance_exclusion_valid = all([
        provenance_absent_from_modeling,
        provenance_absent_from_fitted_input,
        provenance_absent_from_transformed,
    ])

    print(
        f"Provenance exclusion     : "
        f"{'PASS' if provenance_exclusion_valid else 'FAIL'}"
    )


    # ==============================================================================
    # 23.20 — TRANSFORMED DATASET ARRAY VALIDATION
    # ==============================================================================
    # This is the corrected portion.
    #
    # TRANSFORMED_*_DATASETS may be NumPy arrays, so validation uses `.shape`
    # rather than `.columns`.
    # ==============================================================================

    train_transformed = (
        TRANSFORMED_TRAIN_DATASETS[
            dataset_id
        ]
    )

    validation_transformed = (
        TRANSFORMED_VALIDATION_DATASETS[
            dataset_id
        ]
    )

    test_transformed = (
        TRANSFORMED_TEST_DATASETS[
            dataset_id
        ]
    )

    train_shape = array_shape(
        train_transformed
    )

    validation_shape = array_shape(
        validation_transformed
    )

    test_shape = array_shape(
        test_transformed
    )

    expected_train_rows = len(
        TRAIN_DATASETS[dataset_id]
    )

    expected_validation_rows = len(
        VALIDATION_DATASETS[dataset_id]
    )

    expected_test_rows = len(
        TEST_DATASETS[dataset_id]
    )

    transformed_arrays_valid = all([
        train_shape is not None,
        validation_shape is not None,
        test_shape is not None,

        train_shape == (
            expected_train_rows,
            expected_transformed_count,
        ),

        validation_shape == (
            expected_validation_rows,
            expected_transformed_count,
        ),

        test_shape == (
            expected_test_rows,
            expected_transformed_count,
        ),
    ])

    print(
        "\nTransformed dataset arrays:"
    )

    print(
        f"  Train      : "
        f"{'PASS' if train_shape == (expected_train_rows, expected_transformed_count) else 'FAIL'} "
        f"{train_shape}"
    )

    print(
        f"  Validation : "
        f"{'PASS' if validation_shape == (expected_validation_rows, expected_transformed_count) else 'FAIL'} "
        f"{validation_shape}"
    )

    print(
        f"  Test       : "
        f"{'PASS' if test_shape == (expected_test_rows, expected_transformed_count) else 'FAIL'} "
        f"{test_shape}"
    )


    # --------------------------------------------------------------------------
    # The canonical transformed feature-count validation is strengthened by
    # checking the actual NumPy array shape.
    # --------------------------------------------------------------------------

    transformed_feature_count_match = (
        transformed_feature_count_match
        and
        transformed_arrays_valid
    )


    # ==============================================================================
    # 23.21 — SERIALIZED SCHEMA DIAGNOSTICS
    # ==============================================================================

    if not modeling_schema_valid:

        print(
            "\nModeling schema diagnostics:"
        )

        print(
            f"  Expected modeling count: "
            f"{len(expected_modeling_columns)}"
        )

        print(
            f"  Saved modeling count   : "
            f"{len(saved_modeling_columns)}"
        )

        if not modeling_columns_match:

            print(
                "  Modeling columns: FAIL"
            )

        if not numeric_schema_match:

            print(
                "  Numeric columns: FAIL"
            )

        if not categorical_schema_match:

            print(
                "  Categorical columns: FAIL"
            )

        if not target_schema_match:

            print(
                "  Target column: FAIL"
            )

        if not identifier_schema_match:

            print(
                "  Identifier columns: FAIL"
            )

        if not provenance_schema_match:

            print(
                "  Provenance column: FAIL"
            )


    if not transformed_schema_match:

        print(
            "\nTransformed schema diagnostics:"
        )

        print(
            f"  Expected count: "
            f"{expected_transformed_count}"
        )

        print(
            f"  Saved count   : "
            f"{saved_transformed_count}"
        )

        print(
            f"  Expected names: "
            f"{len(expected_transformed_columns)}"
        )

        print(
            f"  Saved names   : "
            f"{len(saved_transformed_columns)}"
        )

        if (
            expected_transformed_columns
            != saved_transformed_columns
        ):

            preview_count = 10

            print(
                "  Expected first names:",
                expected_transformed_columns[
                    :preview_count
                ]
            )

            print(
                "  Saved first names   :",
                saved_transformed_columns[
                    :preview_count
                ]
            )


    # ==============================================================================
    # 23.22 — FINAL DATASET STATUS
    # ==============================================================================

    dataset_status = all([
        preprocessor_exists,
        schema_exists,
        metadata_exists,

        preprocessor_type_valid,
        preprocessor_fitted_valid,

        schema_dataset_valid,
        metadata_dataset_valid,

        modeling_schema_valid,
        transformed_schema_match,

        input_feature_count_match,
        transformed_feature_count_match,

        feature_names_match,
        fitted_input_schema_match,

        train_only_fit_valid,

        identifier_exclusion_valid,
        provenance_exclusion_valid,

        transformed_arrays_valid,
    ])

    print(
        f"\nDataset status           : "
        f"{'PASS' if dataset_status else 'FAIL'}"
    )

    if not dataset_status:

        failed_datasets.append(
            dataset_id
        )


    # ==============================================================================
    # 23.23 — VALIDATION RECORD
    # ==============================================================================

    validation_records.append({

        "dataset_id": dataset_id,

        "preprocessor_exists":
            preprocessor_exists,

        "schema_exists":
            schema_exists,

        "metadata_exists":
            metadata_exists,

        "preprocessor_type_valid":
            preprocessor_type_valid,

        "preprocessor_fitted_valid":
            preprocessor_fitted_valid,

        "schema_dataset_valid":
            schema_dataset_valid,

        "metadata_dataset_valid":
            metadata_dataset_valid,

        "modeling_schema_valid":
            modeling_schema_valid,

        "modeling_columns_match":
            modeling_columns_match,

        "numeric_schema_match":
            numeric_schema_match,

        "categorical_schema_match":
            categorical_schema_match,

        "target_schema_match":
            target_schema_match,

        "identifier_schema_match":
            identifier_schema_match,

        "provenance_schema_match":
            provenance_schema_match,

        "transformed_schema_match":
            transformed_schema_match,

        "input_feature_count_match":
            input_feature_count_match,

        "transformed_feature_count_match":
            transformed_feature_count_match,

        "feature_names_match":
            feature_names_match,

        "fitted_input_schema_match":
            fitted_input_schema_match,

        "train_only_fit_valid":
            train_only_fit_valid,

        "identifier_exclusion_valid":
            identifier_exclusion_valid,

        "provenance_exclusion_valid":
            provenance_exclusion_valid,

        "transformed_arrays_valid":
            transformed_arrays_valid,

        "expected_input_feature_count":
            expected_input_count,

        "actual_input_feature_count":
            actual_input_feature_count,

        "expected_transformed_feature_count":
            expected_transformed_count,

        "actual_transformed_feature_count":
            actual_transformed_count,

        "serialized_transformed_feature_count":
            saved_transformed_count,

        "train_shape":
            str(train_shape),

        "validation_shape":
            str(validation_shape),

        "test_shape":
            str(test_shape),

        "fit_scope":
            str(fit_scope)
            if fit_scope is not None
            else None,

        "status":
            "PASS"
            if dataset_status
            else "FAIL",
    })


# ==============================================================================
# 23.24 — SAVE VALIDATION REPORT
# ==============================================================================

RELOAD_VALIDATION_DF = pd.DataFrame(
    validation_records
)

reload_validation_path = (
    SCHEMA_ROOT
    / "reload_validation.csv"
)

RELOAD_VALIDATION_DF.to_csv(
    reload_validation_path,
    index=False
)


# ==============================================================================
# 23.25 — FINAL SUMMARY
# ==============================================================================

print("\n" + "=" * 100)
print("SECTION 23 VALIDATION SUMMARY")
print("=" * 100)

print(
    RELOAD_VALIDATION_DF.to_string(
        index=False
    )
)

print(
    f"\nReload validation report saved:"
    f"\n{reload_validation_path}"
)


# ==============================================================================
# 23.26 — SECTION GATE
# ==============================================================================

if failed_datasets:

    print(
        "\n" + "=" * 100
    )

    print(
        "SECTION 23 VALIDATION FAILURE"
    )

    print(
        "=" * 100
    )

    print(
        f"Failed datasets: {failed_datasets}"
    )

    raise RuntimeError(
        "One or more saved preprocessing artifacts "
        "failed reload validation.\n"
        f"Failed datasets: {failed_datasets}"
    )


print(
    "\n" + "=" * 100
)

print(
    "SECTION 23 STATUS: PASS"
)

print(
    "=" * 100
)

print(
    "All Notebook 02 preprocessing artifacts "
    "successfully reloaded and validated."
)

print(
    "\nValidated:"
    "\n  ✓ Training-only fitted preprocessors"
    "\n  ✓ Serialized modeling schemas"
    "\n  ✓ Numeric/categorical schemas"
    "\n  ✓ Target retention"
    "\n  ✓ Identifier exclusion"
    "\n  ✓ Provenance exclusion"
    "\n  ✓ Serialized transformed feature schema"
    "\n  ✓ Reloaded transformed feature names"
    "\n  ✓ Input feature counts"
    "\n  ✓ Transformed feature counts"
    "\n  ✓ Fitted input schema"
    "\n  ✓ Train-only fitting policy"
    "\n  ✓ Train/validation/test transformed array shapes"
    "\n  ✓ Persisted artifact integrity"
)

23. RELOAD & VALIDATE ARTIFACTS
Required canonical objects: PASS
Preprocessor root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/preprocessors
Schema root       : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas
Metadata root     : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/metadata

----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
Preprocessor artifact: FOUND
Schema artifact      : FOUND
Metadata artifact    : FOUND
Preprocessor type        : PASS
Preprocessor fitted      : PASS
Schema                   : PASS
Metadata                 : PASS

Schema structure:
  Top-level keys: ['dataset_id', 'schema_version', 'fit_policy', 'modeling_schema', 'transformation_schema', 'preprocessing', 'training_rows', 'creation_timestamp_utc']
  model

In [131]:
# ==================================================================================================
# SECTION 24 — FINAL INTEGRITY VERIFICATION
# ==================================================================================================
#
# Purpose:
#   Perform the final publication-grade integrity audit for Notebook 02.
#
# Canonical architecture validated:
#   DATASET_IDS
#   RAW_DATASETS
#   TARGET_COLUMNS
#   IDENTIFIER_COLUMNS
#   TRAIN_DATASETS
#   VALIDATION_DATASETS
#   TEST_DATASETS
#   SPLIT_MANIFESTS
#   SPLIT_SUMMARY_DF
#   TRAIN_PREPROCESSING_COLUMNS
#   TRAIN_PREPROCESSING_METADATA
#   TRAIN_PREPROCESSORS
#   TRANSFORMED_TRAIN_DATASETS
#   TRANSFORMED_VALIDATION_DATASETS
#   TRANSFORMED_TEST_DATASETS
#   NATIVE_FINAL_DATASETS
#   ENCODED_FINAL_DATASETS
#
# IMPORTANT:
#   This section does NOT recreate legacy aliases.
#   This section does NOT alter any Notebook 02 artifacts.
#   This section only verifies the frozen canonical architecture.
#
# ==================================================================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.compose import ColumnTransformer


print("=" * 100)
print("24. FINAL INTEGRITY VERIFICATION")
print("=" * 100)


# ==================================================================================================
# 24.1 — REQUIRED CANONICAL OBJECTS
# ==================================================================================================

REQUIRED_CANONICAL_OBJECTS = [
    "DATASET_IDS",
    "RAW_DATASETS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",

    "TRAIN_DATASETS",
    "VALIDATION_DATASETS",
    "TEST_DATASETS",

    "SPLIT_MANIFESTS",
    "SPLIT_SUMMARY_DF",

    "TRAIN_PREPROCESSING_COLUMNS",
    "TRAIN_PREPROCESSING_METADATA",
    "TRAIN_PREPROCESSORS",

    "TRANSFORMED_TRAIN_DATASETS",
    "TRANSFORMED_VALIDATION_DATASETS",
    "TRANSFORMED_TEST_DATASETS",

    "NATIVE_FINAL_DATASETS",
    "ENCODED_FINAL_DATASETS",

    "NB02_DIRECTORIES",
]


missing_canonical_objects = [
    name
    for name in REQUIRED_CANONICAL_OBJECTS
    if name not in globals()
]


if missing_canonical_objects:

    print(
        "\nMissing canonical objects:"
    )

    for name in missing_canonical_objects:
        print(f"  - {name}")

    raise RuntimeError(
        "Section 24 cannot run because required canonical objects "
        "are missing."
    )


print(
    "Required canonical objects: PASS"
)


# ==================================================================================================
# 24.2 — FINAL CHECK REGISTRY
# ==================================================================================================

FINAL_CHECKS = {}


def register_check(name, result):

    FINAL_CHECKS[name] = bool(result)

    print(
        f"{'✓' if result else '✗'} "
        f"{name:<38} : "
        f"{'PASS' if result else 'FAIL'}"
    )


# ==================================================================================================
# 24.3 — DATASET COUNT
# ==================================================================================================

dataset_count_valid = (
    len(DATASET_IDS) == 3
)

register_check(
    "dataset_count",
    dataset_count_valid
)


# ==================================================================================================
# 24.4 — DATASET REGISTRY CONSISTENCY
# ==================================================================================================

registry_consistency_valid = all([
    set(DATASET_IDS) == set(RAW_DATASETS.keys()),
    set(DATASET_IDS) == set(TARGET_COLUMNS.keys()),
    set(DATASET_IDS) == set(IDENTIFIER_COLUMNS.keys()),
    set(DATASET_IDS) == set(TRAIN_DATASETS.keys()),
    set(DATASET_IDS) == set(VALIDATION_DATASETS.keys()),
    set(DATASET_IDS) == set(TEST_DATASETS.keys()),
    set(DATASET_IDS) == set(TRAIN_PREPROCESSORS.keys()),
    set(DATASET_IDS) == set(TRAIN_PREPROCESSING_COLUMNS.keys()),
    set(DATASET_IDS) == set(TRAIN_PREPROCESSING_METADATA.keys()),
    set(DATASET_IDS) == set(NATIVE_FINAL_DATASETS.keys()),
    set(DATASET_IDS) == set(ENCODED_FINAL_DATASETS.keys()),
    set(DATASET_IDS) == set(TRANSFORMED_TRAIN_DATASETS.keys()),
    set(DATASET_IDS) == set(TRANSFORMED_VALIDATION_DATASETS.keys()),
    set(DATASET_IDS) == set(TRANSFORMED_TEST_DATASETS.keys()),
])


register_check(
    "canonical_registry_consistency",
    registry_consistency_valid
)


# ==================================================================================================
# 24.5 — RAW DATASET AVAILABILITY
# ==================================================================================================

raw_datasets_loaded = all(
    dataset_id in RAW_DATASETS
    for dataset_id in DATASET_IDS
)


register_check(
    "raw_datasets_loaded",
    raw_datasets_loaded
)


# ==================================================================================================
# 24.6 — TARGET VALIDATION
# ==================================================================================================

target_validation_valid = True


for dataset_id in DATASET_IDS:

    target = TARGET_COLUMNS[dataset_id]

    raw_df = RAW_DATASETS[dataset_id]

    if target not in raw_df.columns:

        target_validation_valid = False

        print(
            f"  Target missing: "
            f"{dataset_id} → {target}"
        )


register_check(
    "targets_validated",
    target_validation_valid
)


# ==================================================================================================
# 24.7 — IDENTIFIER POLICY
# ==================================================================================================

identifier_policy_valid = True


for dataset_id in DATASET_IDS:

    identifiers = list(
        IDENTIFIER_COLUMNS[dataset_id]
    )

    schema = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]

    modeling_columns = list(
        schema["all_columns"]
    )

    for identifier in identifiers:

        if identifier in modeling_columns:

            identifier_policy_valid = False

            print(
                f"  Identifier leakage: "
                f"{dataset_id} → {identifier}"
            )


register_check(
    "identifier_policy_valid",
    identifier_policy_valid
)


# ==================================================================================================
# 24.8 — SPLIT OBJECT COVERAGE
# ==================================================================================================

split_coverage_valid = all(
    dataset_id in SPLIT_MANIFESTS
    for dataset_id in DATASET_IDS
)


register_check(
    "split_manifest_coverage",
    split_coverage_valid
)


# ==================================================================================================
# 24.9 — SPLIT ROW COUNTS
# ==================================================================================================

split_row_count_valid = True


for dataset_id in DATASET_IDS:

    train_df = TRAIN_DATASETS[dataset_id]
    validation_df = VALIDATION_DATASETS[dataset_id]
    test_df = TEST_DATASETS[dataset_id]

    raw_rows = len(
        RAW_DATASETS[dataset_id]
    )

    split_total = (
        len(train_df)
        + len(validation_df)
        + len(test_df)
    )

    if split_total != raw_rows:

        split_row_count_valid = False

        print(
            f"  Row-count mismatch: "
            f"{dataset_id} | "
            f"raw={raw_rows}, "
            f"splits={split_total}"
        )


register_check(
    "split_row_count_preservation",
    split_row_count_valid
)


# ==================================================================================================
# 24.10 — PROVENANCE COLUMN EXISTENCE
# ==================================================================================================

provenance_split_valid = True

PROVENANCE_COLUMN = "__original_row_id__"


for dataset_id in DATASET_IDS:

    for split_name, split_df in [
        ("train", TRAIN_DATASETS[dataset_id]),
        ("validation", VALIDATION_DATASETS[dataset_id]),
        ("test", TEST_DATASETS[dataset_id]),
    ]:

        if PROVENANCE_COLUMN not in split_df.columns:

            provenance_split_valid = False

            print(
                f"  Missing provenance: "
                f"{dataset_id} | {split_name}"
            )


register_check(
    "split_provenance_present",
    provenance_split_valid
)


# ==================================================================================================
# 24.11 — PROVENANCE UNIQUENESS
# ==================================================================================================

provenance_uniqueness_valid = True


for dataset_id in DATASET_IDS:

    train_ids = set(
        TRAIN_DATASETS[dataset_id][
            PROVENANCE_COLUMN
        ]
    )

    validation_ids = set(
        VALIDATION_DATASETS[dataset_id][
            PROVENANCE_COLUMN
        ]
    )

    test_ids = set(
        TEST_DATASETS[dataset_id][
            PROVENANCE_COLUMN
        ]
    )

    if (
        len(train_ids)
        != len(TRAIN_DATASETS[dataset_id])
    ):

        provenance_uniqueness_valid = False

    if (
        len(validation_ids)
        != len(VALIDATION_DATASETS[dataset_id])
    ):

        provenance_uniqueness_valid = False

    if (
        len(test_ids)
        != len(TEST_DATASETS[dataset_id])
    ):

        provenance_uniqueness_valid = False


register_check(
    "provenance_uniqueness",
    provenance_uniqueness_valid
)


# ==================================================================================================
# 24.12 — SPLIT DISJOINTNESS
# ==================================================================================================

split_disjointness_valid = True


for dataset_id in DATASET_IDS:

    train_ids = set(
        TRAIN_DATASETS[dataset_id][
            PROVENANCE_COLUMN
        ]
    )

    validation_ids = set(
        VALIDATION_DATASETS[dataset_id][
            PROVENANCE_COLUMN
        ]
    )

    test_ids = set(
        TEST_DATASETS[dataset_id][
            PROVENANCE_COLUMN
        ]
    )

    if train_ids & validation_ids:
        split_disjointness_valid = False

    if train_ids & test_ids:
        split_disjointness_valid = False

    if validation_ids & test_ids:
        split_disjointness_valid = False


register_check(
    "split_disjointness",
    split_disjointness_valid
)


# ==================================================================================================
# 24.13 — PREPROCESSING SCHEMA INTEGRITY
# ==================================================================================================

preprocessing_schema_valid = True


for dataset_id in DATASET_IDS:

    schema = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]

    required_schema_keys = [
        "all_columns",
        "numeric_columns",
        "categorical_columns",
        "target_column",
        "identifier_columns",
        "provenance_column",
        "transformed_columns",
    ]

    for key in required_schema_keys:

        if key not in schema:

            preprocessing_schema_valid = False

            print(
                f"  Missing schema key: "
                f"{dataset_id} → {key}"
            )


register_check(
    "preprocessing_schema_integrity",
    preprocessing_schema_valid
)


# ==================================================================================================
# 24.14 — MODELING COLUMN POLICY
# ==================================================================================================

modeling_column_policy_valid = True


for dataset_id in DATASET_IDS:

    schema = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]

    modeling_columns = list(
        schema["all_columns"]
    )

    numeric_columns = list(
        schema["numeric_columns"]
    )

    categorical_columns = list(
        schema["categorical_columns"]
    )

    target = schema[
        "target_column"
    ]

    identifiers = list(
        schema["identifier_columns"]
    )

    provenance = schema[
        "provenance_column"
    ]

    # Target must remain a modeling feature.
    if target not in modeling_columns:

        modeling_column_policy_valid = False

    # Identifiers must be excluded.
    if any(
        identifier in modeling_columns
        for identifier in identifiers
    ):

        modeling_column_policy_valid = False

    # Provenance must be excluded.
    if provenance in modeling_columns:

        modeling_column_policy_valid = False

    # Numeric/categorical partition must cover modeling columns.
    partition = (
        numeric_columns
        + categorical_columns
    )

    if set(partition) != set(modeling_columns):

        modeling_column_policy_valid = False


register_check(
    "modeling_column_policy",
    modeling_column_policy_valid
)


# ==================================================================================================
# 24.15 — PREPROCESSOR COUNT
# ==================================================================================================

preprocessor_count_valid = (
    len(TRAIN_PREPROCESSORS)
    == len(DATASET_IDS)
)


register_check(
    "training_preprocessor_count",
    preprocessor_count_valid
)


# ==================================================================================================
# 24.16 — PREPROCESSOR TYPE / FITTED STATE
# ==================================================================================================

preprocessor_fitted_valid = True


for dataset_id in DATASET_IDS:

    preprocessor = TRAIN_PREPROCESSORS[
        dataset_id
    ]

    if not isinstance(
        preprocessor,
        ColumnTransformer
    ):

        preprocessor_fitted_valid = False

        print(
            f"  Invalid preprocessor type: "
            f"{dataset_id}"
        )

        continue

    if not hasattr(
        preprocessor,
        "transformers_"
    ):

        preprocessor_fitted_valid = False

        print(
            f"  Unfitted preprocessor: "
            f"{dataset_id}"
        )

        continue

    try:

        names = list(
            preprocessor.get_feature_names_out()
        )

        if len(names) == 0:

            preprocessor_fitted_valid = False

    except Exception:

        preprocessor_fitted_valid = False


register_check(
    "preprocessors_fitted",
    preprocessor_fitted_valid
)


# ==================================================================================================
# 24.17 — TRAIN-ONLY FITTING POLICY
# ==================================================================================================

train_only_fit_valid = True


for dataset_id in DATASET_IDS:

    metadata = TRAIN_PREPROCESSING_METADATA[
        dataset_id
    ]

    fit_dataset = metadata.get(
        "fit_dataset"
    )

    if fit_dataset != "train_only":

        train_only_fit_valid = False

        print(
            f"  Train-only policy failure: "
            f"{dataset_id} → {fit_dataset}"
        )


register_check(
    "train_only_preprocessing",
    train_only_fit_valid
)


# ==================================================================================================
# 24.18 — PREPROCESSOR INPUT SCHEMA
# ==================================================================================================

preprocessor_input_schema_valid = True


for dataset_id in DATASET_IDS:

    preprocessor = TRAIN_PREPROCESSORS[
        dataset_id
    ]

    expected_columns = list(
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]["all_columns"]
    )

    try:

        fitted_columns = list(
            preprocessor.feature_names_in_
        )

    except Exception:

        preprocessor_input_schema_valid = False

        continue

    if fitted_columns != expected_columns:

        preprocessor_input_schema_valid = False

        print(
            f"  Input schema mismatch: "
            f"{dataset_id}"
        )


register_check(
    "preprocessor_input_schema",
    preprocessor_input_schema_valid
)


# ==================================================================================================
# 24.19 — TRANSFORMED FEATURE SCHEMA
# ==================================================================================================

transformed_feature_schema_valid = True


for dataset_id in DATASET_IDS:

    preprocessor = TRAIN_PREPROCESSORS[
        dataset_id
    ]

    expected_names = list(
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]["transformed_columns"]
    )

    try:

        actual_names = list(
            preprocessor.get_feature_names_out()
        )

    except Exception:

        transformed_feature_schema_valid = False

        continue

    if actual_names != expected_names:

        transformed_feature_schema_valid = False

        print(
            f"  Transformed feature schema mismatch: "
            f"{dataset_id}"
        )


register_check(
    "transformed_feature_schema",
    transformed_feature_schema_valid
)


# ==================================================================================================
# 24.20 — TRANSFORMED ARRAY SHAPES
# ==================================================================================================

transformed_shape_valid = True


for dataset_id in DATASET_IDS:

    expected_features = len(
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]["transformed_columns"]
    )

    expected_train_rows = len(
        TRAIN_DATASETS[dataset_id]
    )

    expected_validation_rows = len(
        VALIDATION_DATASETS[dataset_id]
    )

    expected_test_rows = len(
        TEST_DATASETS[dataset_id]
    )

    train_array = (
        TRANSFORMED_TRAIN_DATASETS[
            dataset_id
        ]
    )

    validation_array = (
        TRANSFORMED_VALIDATION_DATASETS[
            dataset_id
        ]
    )

    test_array = (
        TRANSFORMED_TEST_DATASETS[
            dataset_id
        ]
    )

    expected_train_shape = (
        expected_train_rows,
        expected_features
    )

    expected_validation_shape = (
        expected_validation_rows,
        expected_features
    )

    expected_test_shape = (
        expected_test_rows,
        expected_features
    )

    if tuple(train_array.shape) != expected_train_shape:

        transformed_shape_valid = False

        print(
            f"  Train shape mismatch: "
            f"{dataset_id} | "
            f"{train_array.shape} != "
            f"{expected_train_shape}"
        )

    if tuple(validation_array.shape) != expected_validation_shape:

        transformed_shape_valid = False

        print(
            f"  Validation shape mismatch: "
            f"{dataset_id}"
        )

    if tuple(test_array.shape) != expected_test_shape:

        transformed_shape_valid = False

        print(
            f"  Test shape mismatch: "
            f"{dataset_id}"
        )


register_check(
    "transformed_array_shapes",
    transformed_shape_valid
)


# ==================================================================================================
# 24.21 — TRANSFORMED ARRAY FINITE VALUES
# ==================================================================================================

transformed_finite_valid = True


for dataset_id in DATASET_IDS:

    arrays = [
        TRANSFORMED_TRAIN_DATASETS[dataset_id],
        TRANSFORMED_VALIDATION_DATASETS[dataset_id],
        TRANSFORMED_TEST_DATASETS[dataset_id],
    ]

    for array in arrays:

        if not np.isfinite(
            np.asarray(array)
        ).all():

            transformed_finite_valid = False

            print(
                f"  Non-finite transformed values: "
                f"{dataset_id}"
            )

            break


register_check(
    "transformed_values_finite",
    transformed_finite_valid
)


# ==================================================================================================
# 24.22 — NATIVE DATASET STRUCTURE
# ==================================================================================================

native_structure_valid = True


for dataset_id in DATASET_IDS:

    native_splits = (
        NATIVE_FINAL_DATASETS[
            dataset_id
        ]
    )

    expected_splits = {
        "train",
        "validation",
        "test",
    }

    if set(native_splits.keys()) != expected_splits:

        native_structure_valid = False

        print(
            f"  Native split structure mismatch: "
            f"{dataset_id}"
        )


register_check(
    "native_dataset_structure",
    native_structure_valid
)


# ==================================================================================================
# 24.23 — ENCODED DATASET STRUCTURE
# ==================================================================================================

encoded_structure_valid = True


for dataset_id in DATASET_IDS:

    encoded_splits = (
        ENCODED_FINAL_DATASETS[
            dataset_id
        ]
    )

    expected_splits = {
        "train",
        "validation",
        "test",
    }

    if set(encoded_splits.keys()) != expected_splits:

        encoded_structure_valid = False

        print(
            f"  Encoded split structure mismatch: "
            f"{dataset_id}"
        )


register_check(
    "encoded_dataset_structure",
    encoded_structure_valid
)


# ==================================================================================================
# 24.24 — NATIVE ROW COUNT PRESERVATION
# ==================================================================================================

native_row_count_valid = True


for dataset_id in DATASET_IDS:

    expected_counts = {
        "train": len(
            TRAIN_DATASETS[dataset_id]
        ),
        "validation": len(
            VALIDATION_DATASETS[dataset_id]
        ),
        "test": len(
            TEST_DATASETS[dataset_id]
        ),
    }

    for split_name, expected_rows in expected_counts.items():

        actual_rows = len(
            NATIVE_FINAL_DATASETS[
                dataset_id
            ][split_name]
        )

        if actual_rows != expected_rows:

            native_row_count_valid = False

            print(
                f"  Native row mismatch: "
                f"{dataset_id} | "
                f"{split_name}"
            )


register_check(
    "native_row_count_preservation",
    native_row_count_valid
)


# ==================================================================================================
# 24.25 — ENCODED ROW COUNT PRESERVATION
# ==================================================================================================

encoded_row_count_valid = True


for dataset_id in DATASET_IDS:

    expected_counts = {
        "train": len(
            TRAIN_DATASETS[dataset_id]
        ),
        "validation": len(
            VALIDATION_DATASETS[dataset_id]
        ),
        "test": len(
            TEST_DATASETS[dataset_id]
        ),
    }

    for split_name, expected_rows in expected_counts.items():

        encoded_object = (
            ENCODED_FINAL_DATASETS[
                dataset_id
            ][split_name]
        )

        actual_rows = len(
            encoded_object
        )

        if actual_rows != expected_rows:

            encoded_row_count_valid = False

            print(
                f"  Encoded row mismatch: "
                f"{dataset_id} | "
                f"{split_name}"
            )


register_check(
    "encoded_row_count_preservation",
    encoded_row_count_valid
)


# ==================================================================================================
# 24.26 — NATIVE TARGET RETENTION
# ==================================================================================================

native_target_retention_valid = True


for dataset_id in DATASET_IDS:

    target = TARGET_COLUMNS[
        dataset_id
    ]

    for split_name in [
        "train",
        "validation",
        "test",
    ]:

        native_df = (
            NATIVE_FINAL_DATASETS[
                dataset_id
            ][split_name]
        )

        if target not in native_df.columns:

            native_target_retention_valid = False

            print(
                f"  Native target missing: "
                f"{dataset_id} | "
                f"{split_name}"
            )


register_check(
    "native_target_retention",
    native_target_retention_valid
)


# ==================================================================================================
# 24.27 — NATIVE PROVENANCE RETENTION
# ==================================================================================================

native_provenance_valid = True


for dataset_id in DATASET_IDS:

    for split_name in [
        "train",
        "validation",
        "test",
    ]:

        native_df = (
            NATIVE_FINAL_DATASETS[
                dataset_id
            ][split_name]
        )

        if PROVENANCE_COLUMN not in native_df.columns:

            native_provenance_valid = False

            print(
                f"  Native provenance missing: "
                f"{dataset_id} | "
                f"{split_name}"
            )


register_check(
    "native_provenance_retention",
    native_provenance_valid
)


# ==================================================================================================
# 24.28 — NATIVE IDENTIFIER EXCLUSION
# ==================================================================================================

native_identifier_exclusion_valid = True


for dataset_id in DATASET_IDS:

    identifiers = list(
        IDENTIFIER_COLUMNS[
            dataset_id
        ]
    )

    for split_name in [
        "train",
        "validation",
        "test",
    ]:

        native_df = (
            NATIVE_FINAL_DATASETS[
                dataset_id
            ][split_name]
        )

        for identifier in identifiers:

            if identifier in native_df.columns:

                native_identifier_exclusion_valid = False

                print(
                    f"  Native identifier present: "
                    f"{dataset_id} | "
                    f"{split_name} | "
                    f"{identifier}"
                )


register_check(
    "native_identifier_exclusion",
    native_identifier_exclusion_valid
)


# ==================================================================================================
# 24.29 — ENCODED TARGET POLICY
# ==================================================================================================
#
# IMPORTANT:
#   ENCODED_FINAL_DATASETS are transformed NumPy arrays in the current
#   canonical architecture.
#
#   Therefore the raw target column name is NOT expected to appear as a
#   `.columns` label.
#
#   Target retention is verified through:
#       1. target included in canonical modeling schema
#       2. target included in categorical preprocessing
#       3. transformed feature schema generated by the fitted preprocessor
#
# ==================================================================================================

encoded_target_policy_valid = True


for dataset_id in DATASET_IDS:

    target = TARGET_COLUMNS[
        dataset_id
    ]

    schema = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]

    modeling_columns = list(
        schema["all_columns"]
    )

    categorical_columns = list(
        schema["categorical_columns"]
    )

    if target not in modeling_columns:

        encoded_target_policy_valid = False

        print(
            f"  Encoded target absent from modeling schema: "
            f"{dataset_id}"
        )

    if target not in categorical_columns:

        encoded_target_policy_valid = False

        print(
            f"  Encoded target not treated as categorical: "
            f"{dataset_id}"
        )


register_check(
    "encoded_target_policy",
    encoded_target_policy_valid
)


# ==================================================================================================
# 24.30 — ENCODED IDENTIFIER / PROVENANCE EXCLUSION
# ==================================================================================================

encoded_exclusion_valid = True


for dataset_id in DATASET_IDS:

    transformed_names = list(
        TRAIN_PREPROCESSING_COLUMNS[
            dataset_id
        ]["transformed_columns"]
    )

    identifiers = list(
        IDENTIFIER_COLUMNS[
            dataset_id
        ]
    )

    for identifier in identifiers:

        if identifier in transformed_names:

            encoded_exclusion_valid = False

            print(
                f"  Identifier appears in encoded feature schema: "
                f"{dataset_id} | "
                f"{identifier}"
            )

    if PROVENANCE_COLUMN in transformed_names:

        encoded_exclusion_valid = False

        print(
            f"  Provenance appears in encoded feature schema: "
            f"{dataset_id}"
        )


register_check(
    "encoded_identifier_provenance_exclusion",
    encoded_exclusion_valid
)


# ==================================================================================================
# 24.31 — PERSISTED PREPROCESSOR ARTIFACTS
# ==================================================================================================

persisted_preprocessors_valid = True


for dataset_id in DATASET_IDS:

    path = (
        Path(
            NB02_DIRECTORIES[
                "preprocessors"
            ]
        )
        / dataset_id
        / "train_fitted_preprocessor.joblib"
    )

    if not path.exists():

        persisted_preprocessors_valid = False

        print(
            f"  Missing preprocessor artifact: "
            f"{dataset_id}"
        )


register_check(
    "persisted_preprocessors",
    persisted_preprocessors_valid
)


# ==================================================================================================
# 24.32 — PERSISTED SCHEMA ARTIFACTS
# ==================================================================================================

persisted_schemas_valid = True


for dataset_id in DATASET_IDS:

    path = (
        Path(
            NB02_DIRECTORIES[
                "schemas"
            ]
        )
        / dataset_id
        / "preprocessing_schema.json"
    )

    if not path.exists():

        persisted_schemas_valid = False

        print(
            f"  Missing schema artifact: "
            f"{dataset_id}"
        )


register_check(
    "persisted_preprocessing_schemas",
    persisted_schemas_valid
)


# ==================================================================================================
# 24.33 — PERSISTED METADATA ARTIFACTS
# ==================================================================================================

persisted_metadata_valid = True



24. FINAL INTEGRITY VERIFICATION
Required canonical objects: PASS
✓ dataset_count                          : PASS
✓ canonical_registry_consistency         : PASS
✓ raw_datasets_loaded                    : PASS
✓ targets_validated                      : PASS
✓ identifier_policy_valid                : PASS
✓ split_manifest_coverage                : PASS
✓ split_row_count_preservation           : PASS
✓ split_provenance_present               : PASS
✓ provenance_uniqueness                  : PASS
✓ split_disjointness                     : PASS
✓ preprocessing_schema_integrity         : PASS
✓ modeling_column_policy                 : PASS
✓ training_preprocessor_count            : PASS
✓ preprocessors_fitted                   : PASS
✓ train_only_preprocessing               : PASS
✓ preprocessor_input_schema              : PASS
✓ transformed_feature_schema             : PASS
✓ transformed_array_shapes               : PASS
✓ transformed_values_finite              : PASS
✓ native_dataset_struc

In [132]:
# ==============================================================================
# SECTION 24 — FINAL INTEGRITY VERIFICATION
# ==============================================================================
#
# Purpose:
#   Perform the final publication-grade integrity verification of Notebook 02.
#
# Canonical architecture:
#   DATASET_IDS
#   RAW_DATASETS
#   TRAIN_DATASETS
#   VALIDATION_DATASETS
#   TEST_DATASETS
#   TRAIN_PREPROCESSING_DATA
#   TRAIN_PREPROCESSING_COLUMNS
#   TRAIN_PREPROCESSING_METADATA
#   TRAIN_PREPROCESSORS
#   TRANSFORMED_TRAIN_DATASETS
#   TRANSFORMED_VALIDATION_DATASETS
#   TRANSFORMED_TEST_DATASETS
#   NATIVE_FINAL_DATASETS
#   ENCODED_FINAL_DATASETS
#
# Output:
#   FINAL_INTEGRITY_DF
#   OVERALL_FINAL_INTEGRITY_PASS
#   results/preprocessed/schemas/final_integrity_verification.csv
#   results/preprocessed/schemas/final_integrity_metadata.json
#
# IMPORTANT:
#   This section does not create compatibility aliases for obsolete objects.
# ==============================================================================

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

print("=" * 100)
print("24. FINAL INTEGRITY VERIFICATION")
print("=" * 100)


# ==============================================================================
# 24.1 REQUIRED CANONICAL OBJECTS
# ==============================================================================

REQUIRED_OBJECTS = [
    "PROJECT_ROOT",
    "DATASET_IDS",
    "RAW_DATASETS",
    "TARGET_COLUMNS",
    "IDENTIFIER_COLUMNS",

    "TRAIN_DATASETS",
    "VALIDATION_DATASETS",
    "TEST_DATASETS",

    "TRAIN_PREPROCESSING_DATA",
    "TRAIN_PREPROCESSING_COLUMNS",
    "TRAIN_PREPROCESSING_METADATA",
    "TRAIN_PREPROCESSORS",

    "TRANSFORMED_TRAIN_DATASETS",
    "TRANSFORMED_VALIDATION_DATASETS",
    "TRANSFORMED_TEST_DATASETS",

    "NATIVE_FINAL_DATASETS",
    "ENCODED_FINAL_DATASETS",

    "NB02_DIRECTORIES",
]

missing_objects = [
    name for name in REQUIRED_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 24 cannot start because required canonical objects are missing:\n"
        + "\n".join(f"  - {x}" for x in missing_objects)
    )

print("\nRequired canonical objects : PASS")


# ==============================================================================
# 24.2 INITIALIZE INTEGRITY RECORDS
# ==============================================================================

FINAL_INTEGRITY_RECORDS = []


def add_check(
    check_name,
    status,
    details="",
    dataset_id=None,
):
    """
    Add one integrity check without interrupting the entire verification.
    """

    FINAL_INTEGRITY_RECORDS.append({
        "check": check_name,
        "dataset_id": dataset_id,
        "status": "PASS" if bool(status) else "FAIL",
        "details": str(details),
    })


# ==============================================================================
# 24.3 DATASET COUNT
# ==============================================================================

expected_dataset_count = 3

dataset_count_pass = (
    isinstance(DATASET_IDS, (list, tuple))
    and len(DATASET_IDS) == expected_dataset_count
)

add_check(
    "dataset_count",
    dataset_count_pass,
    f"Expected {expected_dataset_count}; found {len(DATASET_IDS)}."
)

print(
    f"✓ dataset_count : "
    f"{'PASS' if dataset_count_pass else 'FAIL'}"
)


# ==============================================================================
# 24.4 CANONICAL DATASET REGISTRY CONSISTENCY
# ==============================================================================

registry_containers = {
    "RAW_DATASETS": RAW_DATASETS,
    "TRAIN_DATASETS": TRAIN_DATASETS,
    "VALIDATION_DATASETS": VALIDATION_DATASETS,
    "TEST_DATASETS": TEST_DATASETS,
    "TRAIN_PREPROCESSING_DATA": TRAIN_PREPROCESSING_DATA,
    "TRAIN_PREPROCESSING_COLUMNS": TRAIN_PREPROCESSING_COLUMNS,
    "TRAIN_PREPROCESSING_METADATA": TRAIN_PREPROCESSING_METADATA,
    "TRAIN_PREPROCESSORS": TRAIN_PREPROCESSORS,
    "TRANSFORMED_TRAIN_DATASETS": TRANSFORMED_TRAIN_DATASETS,
    "TRANSFORMED_VALIDATION_DATASETS": TRANSFORMED_VALIDATION_DATASETS,
    "TRANSFORMED_TEST_DATASETS": TRANSFORMED_TEST_DATASETS,
    "NATIVE_FINAL_DATASETS": NATIVE_FINAL_DATASETS,
    "ENCODED_FINAL_DATASETS": ENCODED_FINAL_DATASETS,
}

registry_failures = []

for container_name, container in registry_containers.items():

    if not isinstance(container, dict):
        registry_failures.append(
            f"{container_name} is {type(container).__name__}, expected dict"
        )
        continue

    missing = [
        dataset_id
        for dataset_id in DATASET_IDS
        if dataset_id not in container
    ]

    if missing:
        registry_failures.append(
            f"{container_name} missing: {missing}"
        )


registry_pass = len(registry_failures) == 0

add_check(
    "canonical_registry_consistency",
    registry_pass,
    "All canonical dataset containers contain all registered datasets."
    if registry_pass
    else "; ".join(registry_failures)
)

print(
    f"✓ canonical_registry_consistency : "
    f"{'PASS' if registry_pass else 'FAIL'}"
)


# ==============================================================================
# 24.5 DATASET-BY-DATASET VERIFICATION
# ==============================================================================

for dataset_id in DATASET_IDS:

    print("\n" + "-" * 100)
    print(f"VERIFYING DATASET : {dataset_id}")
    print("-" * 100)

    target_column = TARGET_COLUMNS[dataset_id]
    identifier_columns = IDENTIFIER_COLUMNS.get(dataset_id, [])

    raw_df = RAW_DATASETS[dataset_id]

    train_df = TRAIN_DATASETS[dataset_id]
    validation_df = VALIDATION_DATASETS[dataset_id]
    test_df = TEST_DATASETS[dataset_id]

    schema = TRAIN_PREPROCESSING_COLUMNS[dataset_id]

    # ==========================================================================
    # RAW DATASET
    # ==========================================================================

    raw_pass = isinstance(raw_df, pd.DataFrame) and len(raw_df) > 0

    add_check(
        "raw_dataset_loaded",
        raw_pass,
        f"Shape={raw_df.shape if isinstance(raw_df, pd.DataFrame) else None}",
        dataset_id
    )

    # ==========================================================================
    # TARGET
    # ==========================================================================

    target_pass = (
        target_column in raw_df.columns
        and target_column in train_df.columns
        and target_column in validation_df.columns
        and target_column in test_df.columns
    )

    add_check(
        "target_column_integrity",
        target_pass,
        f"Target={target_column}",
        dataset_id
    )

    # ==========================================================================
    # IDENTIFIER POLICY
    # ==========================================================================

    identifier_policy_pass = all(
        identifier not in schema["all_columns"]
        for identifier in identifier_columns
    )

    add_check(
        "identifier_exclusion_policy",
        identifier_policy_pass,
        f"Excluded identifiers={identifier_columns}",
        dataset_id
    )

    # ==========================================================================
    # SPLIT ROW COUNTS
    # ==========================================================================

    raw_rows = len(raw_df)

    split_rows = (
        len(train_df)
        + len(validation_df)
        + len(test_df)
    )

    split_rows_pass = split_rows == raw_rows

    add_check(
        "split_row_count_preservation",
        split_rows_pass,
        f"Raw={raw_rows}; Train={len(train_df)}; "
        f"Validation={len(validation_df)}; Test={len(test_df)}; "
        f"Combined={split_rows}",
        dataset_id
    )

    # ==========================================================================
    # PROVENANCE
    # ==========================================================================

    provenance_column = schema.get(
        "provenance_column",
        "__original_row_id__"
    )

    provenance_pass = True
    provenance_details = []

    split_provenance_sets = []

    for split_name, split_df in [
        ("train", train_df),
        ("validation", validation_df),
        ("test", test_df),
    ]:

        if provenance_column not in split_df.columns:
            provenance_pass = False
            provenance_details.append(
                f"{split_name}: provenance missing"
            )
            continue

        values = split_df[provenance_column]

        if values.isna().any():
            provenance_pass = False
            provenance_details.append(
                f"{split_name}: provenance contains NaN"
            )

        if not values.is_unique:
            provenance_pass = False
            provenance_details.append(
                f"{split_name}: duplicate provenance values"
            )

        split_provenance_sets.append(
            set(values.astype(np.int64).tolist())
        )

    if len(split_provenance_sets) == 3:

        train_ids, val_ids, test_ids = split_provenance_sets

        if train_ids & val_ids:
            provenance_pass = False
            provenance_details.append("train/validation overlap")

        if train_ids & test_ids:
            provenance_pass = False
            provenance_details.append("train/test overlap")

        if val_ids & test_ids:
            provenance_pass = False
            provenance_details.append("validation/test overlap")

        union_size = len(train_ids | val_ids | test_ids)

        if union_size != raw_rows:
            provenance_pass = False
            provenance_details.append(
                f"union size={union_size}, expected={raw_rows}"
            )

    add_check(
        "split_provenance_integrity",
        provenance_pass,
        "PASS"
        if provenance_pass
        else "; ".join(provenance_details),
        dataset_id
    )

    # ==========================================================================
    # PREPROCESSING SCHEMA
    # ==========================================================================

    required_schema_keys = [
        "all_columns",
        "numeric_columns",
        "categorical_columns",
        "target_column",
        "identifier_columns",
        "provenance_column",
        "transformed_columns",
    ]

    schema_keys_pass = all(
        key in schema
        for key in required_schema_keys
    )

    add_check(
        "preprocessing_schema_integrity",
        schema_keys_pass,
        f"Required keys={required_schema_keys}",
        dataset_id
    )

    # ==========================================================================
    # MODELING COLUMN POLICY
    # ==========================================================================

    modeling_columns = schema.get("all_columns", [])

    modeling_policy_pass = (
        provenance_column not in modeling_columns
        and all(
            identifier not in modeling_columns
            for identifier in identifier_columns
        )
        and target_column in modeling_columns
    )

    add_check(
        "modeling_column_policy",
        modeling_policy_pass,
        f"Modeling columns={len(modeling_columns)}; "
        f"target retained={target_column in modeling_columns}; "
        f"provenance excluded={provenance_column not in modeling_columns}; "
        f"identifiers excluded={all(i not in modeling_columns for i in identifier_columns)}",
        dataset_id
    )

    # ==========================================================================
    # TRAIN PREPROCESSOR
    # ==========================================================================

    preprocessor = TRAIN_PREPROCESSORS[dataset_id]

    preprocessor_type_pass = hasattr(
        preprocessor,
        "transform"
    ) and hasattr(
        preprocessor,
        "fit"
    )

    fitted_pass = hasattr(
        preprocessor,
        "transformers_"
    )

    metadata = TRAIN_PREPROCESSING_METADATA[dataset_id]

    train_only_pass = (
        metadata.get("fit_dataset") == "train_only"
        or metadata.get("fit_policy") == "train_only"
        or "train" in str(metadata).lower()
    )

    add_check(
        "training_preprocessor_valid",
        preprocessor_type_pass and fitted_pass,
        f"Type={type(preprocessor).__name__}; fitted={fitted_pass}",
        dataset_id
    )

    add_check(
        "training_only_preprocessing",
        train_only_pass,
        f"fit_dataset={metadata.get('fit_dataset', 'not specified')}",
        dataset_id
    )

    # ==========================================================================
    # PREPROCESSOR INPUT SCHEMA
    # ==========================================================================

    fitted_input_columns = list(
        getattr(
            preprocessor,
            "feature_names_in_",
            []
        )
    )

    expected_input_columns = list(modeling_columns)

    preprocessor_input_pass = (
        fitted_input_columns == expected_input_columns
    )

    add_check(
        "preprocessor_input_schema",
        preprocessor_input_pass,
        f"Expected={len(expected_input_columns)}; "
        f"fitted={len(fitted_input_columns)}",
        dataset_id
    )

    # ==========================================================================
    # TRANSFORMED FEATURE SCHEMA
    # ==========================================================================

    transformed_columns = schema.get(
        "transformed_columns",
        []
    )

    try:
        fitted_feature_names = list(
            preprocessor.get_feature_names_out()
        )
    except Exception as exc:
        fitted_feature_names = []
        print(
            f"  Warning: get_feature_names_out failed: {exc}"
        )

    transformed_feature_count = len(transformed_columns)

    transformed_schema_pass = (
        transformed_feature_count > 0
        and len(fitted_feature_names) == transformed_feature_count
        and transformed_columns == fitted_feature_names
    )

    add_check(
        "transformed_feature_schema",
        transformed_schema_pass,
        f"Saved={transformed_feature_count}; "
        f"fitted={len(fitted_feature_names)}",
        dataset_id
    )

    # ==========================================================================
    # TRANSFORMED ARRAY SHAPES
    # ==========================================================================

    transformed_sets = {
        "train": TRANSFORMED_TRAIN_DATASETS[dataset_id],
        "validation": TRANSFORMED_VALIDATION_DATASETS[dataset_id],
        "test": TRANSFORMED_TEST_DATASETS[dataset_id],
    }

    transformed_shape_pass = True
    transformed_shape_details = []

    expected_feature_count = len(transformed_columns)

    for split_name, array in transformed_sets.items():

        if not isinstance(array, np.ndarray):
            transformed_shape_pass = False
            transformed_shape_details.append(
                f"{split_name}: not ndarray"
            )
            continue

        expected_rows = len({
            "train": train_df,
            "validation": validation_df,
            "test": test_df,
        }[split_name])

        if array.ndim != 2:
            transformed_shape_pass = False
            transformed_shape_details.append(
                f"{split_name}: ndim={array.ndim}"
            )
            continue

        if array.shape != (
            expected_rows,
            expected_feature_count
        ):
            transformed_shape_pass = False
            transformed_shape_details.append(
                f"{split_name}: {array.shape} != "
                f"({expected_rows}, {expected_feature_count})"
            )

    add_check(
        "transformed_array_shapes",
        transformed_shape_pass,
        "All transformed split shapes are correct."
        if transformed_shape_pass
        else "; ".join(transformed_shape_details),
        dataset_id
    )

    # ==========================================================================
    # TRANSFORMED FINITE VALUES
    # ==========================================================================
    #
    # RAM-safe chunked verification.
    # Do not create a second complete boolean matrix for the large
    # diabetes_130us array.
    # ==========================================================================

    finite_pass = True
    finite_details = []

    for split_name, array in transformed_sets.items():

        if not isinstance(array, np.ndarray):
            finite_pass = False
            finite_details.append(
                f"{split_name}: invalid array"
            )
            continue

        chunk_size = 10000

        for start in range(
            0,
            array.shape[0],
            chunk_size
        ):

            stop = min(
                start + chunk_size,
                array.shape[0]
            )

            chunk = array[start:stop]

            if not np.isfinite(chunk).all():
                finite_pass = False
                finite_details.append(
                    f"{split_name}: non-finite values "
                    f"in rows {start}:{stop}"
                )
                break

    add_check(
        "transformed_finite_values",
        finite_pass,
        "All transformed values are finite."
        if finite_pass
        else "; ".join(finite_details),
        dataset_id
    )

    # ==========================================================================
    # NATIVE DATASET STRUCTURE
    # ==========================================================================

    native_dataset = NATIVE_FINAL_DATASETS[dataset_id]

    native_structure_pass = (
        isinstance(native_dataset, dict)
        and all(
            split in native_dataset
            for split in ["train", "validation", "test"]
        )
    )

    add_check(
        "native_dataset_structure",
        native_structure_pass,
        "Dataset-first native structure with all three splits."
        if native_structure_pass
        else str(
            list(native_dataset.keys())
            if isinstance(native_dataset, dict)
            else type(native_dataset)
        ),
        dataset_id
    )

    # ==========================================================================
    # ENCODED DATASET STRUCTURE
    # ==========================================================================

    encoded_dataset = ENCODED_FINAL_DATASETS[dataset_id]

    encoded_structure_pass = (
        isinstance(encoded_dataset, dict)
        and all(
            split in encoded_dataset
            for split in ["train", "validation", "test"]
        )
    )

    add_check(
        "encoded_dataset_structure",
        encoded_structure_pass,
        "Dataset-first encoded structure with all three splits."
        if encoded_structure_pass
        else str(
            list(encoded_dataset.keys())
            if isinstance(encoded_dataset, dict)
            else type(encoded_dataset)
        ),
        dataset_id
    )

    # ==========================================================================
    # NATIVE ROW COUNTS
    # ==========================================================================

    native_row_count_pass = True
    native_row_details = []

    expected_split_rows = {
        "train": len(train_df),
        "validation": len(validation_df),
        "test": len(test_df),
    }

    if native_structure_pass:

        for split_name, expected_rows in expected_split_rows.items():

            native_split = native_dataset[split_name]

            if not isinstance(native_split, pd.DataFrame):
                native_row_count_pass = False
                native_row_details.append(
                    f"{split_name}: not DataFrame"
                )
                continue

            if len(native_split) != expected_rows:
                native_row_count_pass = False
                native_row_details.append(
                    f"{split_name}: {len(native_split)} != {expected_rows}"
                )

    else:
        native_row_count_pass = False
        native_row_details.append(
            "native structure invalid"
        )

    add_check(
        "native_row_counts",
        native_row_count_pass,
        "Native row counts preserved."
        if native_row_count_pass
        else "; ".join(native_row_details),
        dataset_id
    )

    # ==========================================================================
    # ENCODED ROW COUNTS
    # ==========================================================================

    encoded_row_count_pass = True
    encoded_row_details = []

    if encoded_structure_pass:

        for split_name, expected_rows in expected_split_rows.items():

            encoded_split = encoded_dataset[split_name]

            if not isinstance(
                encoded_split,
                (np.ndarray, pd.DataFrame)
            ):
                encoded_row_count_pass = False
                encoded_row_details.append(
                    f"{split_name}: invalid type"
                )
                continue

            if len(encoded_split) != expected_rows:
                encoded_row_count_pass = False
                encoded_row_details.append(
                    f"{split_name}: {len(encoded_split)} != {expected_rows}"
                )

    else:
        encoded_row_count_pass = False
        encoded_row_details.append(
            "encoded structure invalid"
        )

    add_check(
        "encoded_row_counts",
        encoded_row_count_pass,
        "Encoded row counts preserved."
        if encoded_row_count_pass
        else "; ".join(encoded_row_details),
        dataset_id
    )

    # ==========================================================================
    # NATIVE TARGET + PROVENANCE RETENTION
    # ==========================================================================

    native_policy_pass = True
    native_policy_details = []

    if native_structure_pass:

        for split_name in [
            "train",
            "validation",
            "test"
        ]:

            native_split = native_dataset[split_name]

            if not isinstance(native_split, pd.DataFrame):
                native_policy_pass = False
                native_policy_details.append(
                    f"{split_name}: not DataFrame"
                )
                continue

            if target_column not in native_split.columns:
                native_policy_pass = False
                native_policy_details.append(
                    f"{split_name}: target missing"
                )

            if provenance_column not in native_split.columns:
                native_policy_pass = False
                native_policy_details.append(
                    f"{split_name}: provenance missing"
                )

    add_check(
        "native_target_provenance_retention",
        native_policy_pass,
        "Target and provenance retained in native datasets."
        if native_policy_pass
        else "; ".join(native_policy_details),
        dataset_id
    )

    # ==========================================================================
    # NATIVE IDENTIFIER EXCLUSION
    # ==========================================================================

    native_identifier_pass = True
    native_identifier_details = []

    if native_structure_pass:

        for split_name in [
            "train",
            "validation",
            "test"
        ]:

            native_split = native_dataset[split_name]

            if isinstance(native_split, pd.DataFrame):

                present_identifiers = [
                    identifier
                    for identifier in identifier_columns
                    if identifier in native_split.columns
                ]

                if present_identifiers:
                    native_identifier_pass = False
                    native_identifier_details.append(
                        f"{split_name}: {present_identifiers}"
                    )

    add_check(
        "native_identifier_exclusion",
        native_identifier_pass,
        "All configured identifiers excluded."
        if native_identifier_pass
        else "; ".join(native_identifier_details),
        dataset_id
    )

    # ==========================================================================
    # ENCODED PROVENANCE / IDENTIFIER EXCLUSION
    # ==========================================================================

    encoded_metadata_pass = True
    encoded_metadata_details = []

    # Encoded data are arrays and therefore should not contain raw
    # identifier/provenance columns.

    if encoded_structure_pass:

        for split_name in [
            "train",
            "validation",
            "test"
        ]:

            encoded_split = encoded_dataset[split_name]

            if isinstance(encoded_split, pd.DataFrame):

                if provenance_column in encoded_split.columns:
                    encoded_metadata_pass = False
                    encoded_metadata_details.append(
                        f"{split_name}: provenance present"
                    )

                present_identifiers = [
                    identifier
                    for identifier in identifier_columns
                    if identifier in encoded_split.columns
                ]

                if present_identifiers:
                    encoded_metadata_pass = False
                    encoded_metadata_details.append(
                        f"{split_name}: identifiers present"
                    )

    add_check(
        "encoded_metadata_exclusion",
        encoded_metadata_pass,
        "Provenance and identifiers absent from encoded model arrays."
        if encoded_metadata_pass
        else "; ".join(encoded_metadata_details),
        dataset_id
    )

    # ==========================================================================
    # TARGET POLICY IN ENCODED DATA
    # ==========================================================================

    #
    # The target is intentionally part of the modeling schema and categorical
    # transformation. Therefore it is expected to contribute to transformed
    # features, but must NOT be manually appended a second time.
    #

    target_in_modeling_pass = target_column in modeling_columns

    transformed_target_metadata_pass = (
        target_in_modeling_pass
        and target_column in schema.get(
            "categorical_columns",
            []
        )
    )

    add_check(
        "encoded_target_policy",
        transformed_target_metadata_pass,
        f"Target={target_column}; "
        f"in modeling columns={target_in_modeling_pass}; "
        f"categorical={target_column in schema.get('categorical_columns', [])}; "
        f"manually appended target prohibited.",
        dataset_id
    )


# ==============================================================================
# 24.6 PERSISTED ARTIFACT VERIFICATION
# ==============================================================================

print("\n" + "=" * 100)
print("PERSISTED ARTIFACT VERIFICATION")
print("=" * 100)


schemas_root = Path(
    NB02_DIRECTORIES["schemas"]
)

metadata_root = (
    schemas_root / "metadata"
)

preprocessors_root = Path(
    NB02_DIRECTORIES["preprocessors"]
)

feature_mapping_path = Path(
    NB02_DIRECTORIES.get(
        "feature_mapping",
        schemas_root / "feature_mapping.csv"
    )
)

# ------------------------------------------------------------------------------
# Preprocessor artifacts
# ------------------------------------------------------------------------------

preprocessor_artifact_pass = True
preprocessor_artifact_details = []

for dataset_id in DATASET_IDS:

    path = (
        preprocessors_root
        / dataset_id
        / "train_fitted_preprocessor.joblib"
    )

    if not path.exists():
        preprocessor_artifact_pass = False
        preprocessor_artifact_details.append(
            f"{dataset_id}: {path}"
        )

add_check(
    "persisted_preprocessors",
    preprocessor_artifact_pass,
    "All train-fitted preprocessors exist."
    if preprocessor_artifact_pass
    else "; ".join(preprocessor_artifact_details)
)


# ------------------------------------------------------------------------------
# Schema artifacts
# ------------------------------------------------------------------------------

schema_artifact_pass = True
schema_artifact_details = []

for dataset_id in DATASET_IDS:

    path = (
        schemas_root
        / dataset_id
        / "preprocessing_schema.json"
    )

    if not path.exists():
        schema_artifact_pass = False
        schema_artifact_details.append(
            f"{dataset_id}: {path}"
        )

add_check(
    "persisted_preprocessing_schemas",
    schema_artifact_pass,
    "All preprocessing schemas exist."
    if schema_artifact_pass
    else "; ".join(schema_artifact_details)
)


# ------------------------------------------------------------------------------
# Metadata artifacts
# ------------------------------------------------------------------------------

metadata_artifact_pass = True
metadata_artifact_details = []

for dataset_id in DATASET_IDS:

    path = (
        metadata_root
        / f"{dataset_id}_preprocessing_metadata.json"
    )

    if not path.exists():
        metadata_artifact_pass = False
        metadata_artifact_details.append(
            f"{dataset_id}: {path}"
        )

add_check(
    "persisted_preprocessing_metadata",
    metadata_artifact_pass,
    "All preprocessing metadata files exist."
    if metadata_artifact_pass
    else "; ".join(metadata_artifact_details)
)


# ------------------------------------------------------------------------------
# Feature mapping
# ------------------------------------------------------------------------------

feature_mapping_pass = (
    feature_mapping_path.exists()
)

add_check(
    "feature_mapping_artifact",
    feature_mapping_pass,
    f"Path={feature_mapping_path}"
)


# ------------------------------------------------------------------------------
# Section 22 split manifest
# ------------------------------------------------------------------------------

split_manifest_path = (
    Path(NB02_DIRECTORIES["manifests"])
    / "split_manifest.csv"
)

split_manifest_pass = (
    split_manifest_path.exists()
)

add_check(
    "split_manifest_artifact",
    split_manifest_pass,
    f"Path={split_manifest_path}"
)


# ------------------------------------------------------------------------------
# Section 23 reload validation
# ------------------------------------------------------------------------------

reload_validation_path = (
    schemas_root
    / "reload_validation.csv"
)

reload_validation_pass = (
    reload_validation_path.exists()
)

add_check(
    "section_23_reload_validation",
    reload_validation_pass,
    f"Path={reload_validation_path}"
)


# ==============================================================================
# 24.7 BUILD FINAL INTEGRITY DATAFRAME
# ==============================================================================

FINAL_INTEGRITY_DF = pd.DataFrame(
    FINAL_INTEGRITY_RECORDS
)

if FINAL_INTEGRITY_DF.empty:
    raise RuntimeError(
        "Section 24 generated no integrity records."
    )


# ==============================================================================
# 24.8 OVERALL FINAL INTEGRITY STATUS
# ==============================================================================

OVERALL_FINAL_INTEGRITY_PASS = bool(
    (
        FINAL_INTEGRITY_DF["status"]
        == "PASS"
    ).all()
)


# ==============================================================================
# 24.9 SAVE FINAL INTEGRITY REPORT
# ==============================================================================

final_integrity_path = (
    schemas_root
    / "final_integrity_verification.csv"
)

final_integrity_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_INTEGRITY_DF.to_csv(
    final_integrity_path,
    index=False
)


# ==============================================================================
# 24.10 SAVE FINAL INTEGRITY METADATA
# ==============================================================================

final_integrity_metadata = {
    "section": "24",
    "section_name": "Final Integrity Verification",
    "status": (
        "PASS"
        if OVERALL_FINAL_INTEGRITY_PASS
        else "FAIL"
    ),
    "datasets": list(DATASET_IDS),
    "dataset_count": len(DATASET_IDS),
    "master_seed": int(
        globals().get(
            "MASTER_SEED",
            2025
        )
    ),
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "checks_total": int(
        len(FINAL_INTEGRITY_DF)
    ),
    "checks_passed": int(
        (
            FINAL_INTEGRITY_DF["status"]
            == "PASS"
        ).sum()
    ),
    "checks_failed": int(
        (
            FINAL_INTEGRITY_DF["status"]
            == "FAIL"
        ).sum()
    ),
    "final_integrity_report": str(
        final_integrity_path
    ),
}


final_integrity_metadata_path = (
    schemas_root
    / "final_integrity_metadata.json"
)

with open(
    final_integrity_metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_integrity_metadata,
        f,
        indent=2
    )


# ==============================================================================
# 24.11 DISPLAY RESULTS
# ==============================================================================

print("\n" + "=" * 100)
print("FINAL INTEGRITY RESULTS")
print("=" * 100)

print(
    f"\nTotal checks  : "
    f"{final_integrity_metadata['checks_total']}"
)

print(
    f"Passed checks : "
    f"{final_integrity_metadata['checks_passed']}"
)

print(
    f"Failed checks : "
    f"{final_integrity_metadata['checks_failed']}"
)

print("\nCheck summary:")

check_summary = (
    FINAL_INTEGRITY_DF
    .groupby("status")
    .size()
    .to_dict()
)

print(
    f"  PASS : {check_summary.get('PASS', 0)}"
)

print(
    f"  FAIL : {check_summary.get('FAIL', 0)}"
)


# ==============================================================================
# 24.12 DISPLAY FAILURES IF ANY
# ==============================================================================

FAILED_INTEGRITY_CHECKS = (
    FINAL_INTEGRITY_DF[
        FINAL_INTEGRITY_DF["status"] == "FAIL"
    ].copy()
)

if not FAILED_INTEGRITY_CHECKS.empty:

    print("\n" + "=" * 100)
    print("FAILED INTEGRITY CHECKS")
    print("=" * 100)

    display(
        FAILED_INTEGRITY_CHECKS[
            [
                "check",
                "dataset_id",
                "details",
            ]
        ]
    )


# ==============================================================================
# 24.13 FINAL STATUS
# ==============================================================================

print("\n" + "=" * 100)

if OVERALL_FINAL_INTEGRITY_PASS:

    print("FINAL INTEGRITY STATUS: PASS")
    print("=" * 100)

    print(
        "\n✓ Notebook 02 final integrity verification completed successfully."
    )

    print(
        f"\nSaved final integrity report:\n"
        f"  {final_integrity_path}"
    )

    print(
        f"\nSaved final integrity metadata:\n"
        f"  {final_integrity_metadata_path}"
    )

else:

    print("FINAL INTEGRITY STATUS: FAIL")
    print("=" * 100)

    print(
        "\n✗ One or more final integrity checks failed."
    )

    print(
        "\nNotebook 02 MUST NOT be treated as complete."
    )

    print(
        "\nReview FAILED INTEGRITY CHECKS above before proceeding."
    )

24. FINAL INTEGRITY VERIFICATION

Required canonical objects : PASS
✓ dataset_count : PASS
✓ canonical_registry_consistency : PASS

----------------------------------------------------------------------------------------------------
VERIFYING DATASET : adult_income
----------------------------------------------------------------------------------------------------

----------------------------------------------------------------------------------------------------
VERIFYING DATASET : bank_marketing
----------------------------------------------------------------------------------------------------

----------------------------------------------------------------------------------------------------
VERIFYING DATASET : diabetes_130us
----------------------------------------------------------------------------------------------------

PERSISTED ARTIFACT VERIFICATION

FINAL INTEGRITY RESULTS

Total checks  : 71
Passed checks : 71
Failed checks : 0

Check summary:
  PASS : 71
  FAIL : 0

FI

In [137]:
# ==============================================================================
# SECTION 25 — NOTEBOOK 02 FINAL COMPLETION GATE
# ==============================================================================

from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np

print("=" * 100)
print("SECTION 25 — NOTEBOOK 02 FINAL COMPLETION GATE")
print("=" * 100)


# ==============================================================================
# 25.1 — REQUIRED CANONICAL OBJECTS
# ==============================================================================

required_objects = [
    "DATASET_IDS",
    "TRAIN_DATASETS",
    "VALIDATION_DATASETS",
    "TEST_DATASETS",
    "TRAIN_PREPROCESSING_DATA",
    "TRAIN_PREPROCESSING_COLUMNS",
    "TRAIN_PREPROCESSING_METADATA",
    "TRAIN_PREPROCESSORS",
    "TRANSFORMED_TRAIN_DATASETS",
    "TRANSFORMED_VALIDATION_DATASETS",
    "TRANSFORMED_TEST_DATASETS",
    "NATIVE_FINAL_DATASETS",
    "ENCODED_FINAL_DATASETS",
    "NB02_DIRECTORIES",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 25 cannot continue.\n"
        f"Missing canonical objects: {missing_objects}"
    )

print("\n[25.1] Required canonical objects")
print("-" * 100)

for name in required_objects:
    print(f"  ✓ {name}")

print("  Required final objects : PASS")


# ==============================================================================
# 25.2 — SECTION 24 FINAL INTEGRITY GATE
# ==============================================================================

print("\n[25.2] Section 24 final integrity gate")
print("-" * 100)

if "OVERALL_FINAL_INTEGRITY_PASS" not in globals():
    raise RuntimeError(
        "OVERALL_FINAL_INTEGRITY_PASS not found. "
        "Run Section 24 before Section 25."
    )

section_24_pass = bool(
    OVERALL_FINAL_INTEGRITY_PASS
)

print(
    f"  Section 24 final integrity gate : "
    f"{'PASS' if section_24_pass else 'FAIL'}"
)

if not section_24_pass:
    raise RuntimeError(
        "Section 25 stopped because Section 24 final integrity gate failed."
    )


# ==============================================================================
# 25.3 — DATASET COMPLETENESS VERIFICATION
# ==============================================================================

print("\n[25.3] Dataset completeness verification")
print("-" * 100)

dataset_completion_records = []

for dataset_id in DATASET_IDS:

    print(f"\n  Dataset: {dataset_id}")

    # --------------------------------------------------------------------------
    # Required dataset objects
    # --------------------------------------------------------------------------

    if dataset_id not in TRAIN_DATASETS:
        raise RuntimeError(
            f"{dataset_id}: train dataset missing."
        )

    if dataset_id not in VALIDATION_DATASETS:
        raise RuntimeError(
            f"{dataset_id}: validation dataset missing."
        )

    if dataset_id not in TEST_DATASETS:
        raise RuntimeError(
            f"{dataset_id}: test dataset missing."
        )

    if dataset_id not in NATIVE_FINAL_DATASETS:
        raise RuntimeError(
            f"{dataset_id}: native final dataset missing."
        )

    if dataset_id not in ENCODED_FINAL_DATASETS:
        raise RuntimeError(
            f"{dataset_id}: encoded final dataset missing."
        )

    if dataset_id not in TRAIN_PREPROCESSORS:
        raise RuntimeError(
            f"{dataset_id}: training preprocessor missing."
        )

    if dataset_id not in TRAIN_PREPROCESSING_COLUMNS:
        raise RuntimeError(
            f"{dataset_id}: preprocessing schema missing."
        )

    # --------------------------------------------------------------------------
    # Schema
    # --------------------------------------------------------------------------

    schema = TRAIN_PREPROCESSING_COLUMNS[
        dataset_id
    ]

    model_columns = list(
        schema["all_columns"]
    )

    numeric_columns = list(
        schema["numeric_columns"]
    )

    categorical_columns = list(
        schema["categorical_columns"]
    )

    target_column = schema["target_column"]

    identifier_columns = list(
        schema["identifier_columns"]
    )

    transformed_columns = list(
        schema["transformed_columns"]
    )

    # --------------------------------------------------------------------------
    # Split sizes
    # --------------------------------------------------------------------------

    train_rows = len(
        TRAIN_DATASETS[dataset_id]
    )

    validation_rows = len(
        VALIDATION_DATASETS[dataset_id]
    )

    test_rows = len(
        TEST_DATASETS[dataset_id]
    )

    original_rows = (
        train_rows
        + validation_rows
        + test_rows
    )

    # --------------------------------------------------------------------------
    # Final dataset shapes
    # --------------------------------------------------------------------------

    native_train = (
        NATIVE_FINAL_DATASETS[
            dataset_id
        ]["train"]
    )

    encoded_train = (
        ENCODED_FINAL_DATASETS[
            dataset_id
        ]["train"]
    )

    expected_native_columns = (
        len(model_columns) + 1
    )

    expected_encoded_columns = (
        len(transformed_columns)
    )

    native_shape = tuple(
        native_train.shape
    )

    encoded_shape = tuple(
        encoded_train.shape
    )

    native_pass = (
        native_shape[0] == train_rows
        and
        native_shape[1] == expected_native_columns
    )

    encoded_pass = (
        encoded_shape[0] == train_rows
        and
        encoded_shape[1] == expected_encoded_columns
    )

    dataset_pass = (
        train_rows > 0
        and validation_rows > 0
        and test_rows > 0
        and len(model_columns) > 0
        and target_column in model_columns
        and native_pass
        and encoded_pass
    )

    identifier_display = (
        ", ".join(identifier_columns)
        if identifier_columns
        else "None"
    )

    # --------------------------------------------------------------------------
    # Report
    # --------------------------------------------------------------------------

    print(
        f"    Original rows       : "
        f"{original_rows:,}"
    )

    print(
        f"    Modeling columns    : "
        f"{len(model_columns)}"
    )

    print(
        f"    Numeric columns     : "
        f"{len(numeric_columns)}"
    )

    print(
        f"    Categorical columns : "
        f"{len(categorical_columns)}"
    )

    print(
        f"    Transformed columns : "
        f"{len(transformed_columns)}"
    )

    print(
        f"    Target              : "
        f"{target_column}"
    )

    print(
        f"    Identifiers excluded: "
        f"{identifier_display}"
    )

    print(
        f"    Train / Val / Test  : "
        f"{train_rows:,} / "
        f"{validation_rows:,} / "
        f"{test_rows:,}"
    )

    print(
        f"    Native train shape  : "
        f"{native_shape}"
    )

    print(
        f"    Encoded train shape : "
        f"{encoded_shape}"
    )

    print(
        f"    Dataset status      : "
        f"{'PASS' if dataset_pass else 'FAIL'}"
    )

    dataset_completion_records.append(
        {
            "dataset_id": dataset_id,
            "original_rows": original_rows,
            "train_rows": train_rows,
            "validation_rows": validation_rows,
            "test_rows": test_rows,
            "modeling_columns": len(model_columns),
            "numeric_columns": len(numeric_columns),
            "categorical_columns": len(categorical_columns),
            "transformed_columns": len(transformed_columns),
            "target_column": target_column,
            "identifier_count": len(identifier_columns),
            "native_train_rows": native_shape[0],
            "native_train_columns": native_shape[1],
            "encoded_train_rows": encoded_shape[0],
            "encoded_train_columns": encoded_shape[1],
            "dataset_complete": dataset_pass,
        }
    )


DATASET_COMPLETENESS_DF = pd.DataFrame(
    dataset_completion_records
)

dataset_completeness_pass = bool(
    DATASET_COMPLETENESS_DF[
        "dataset_complete"
    ].all()
)

print(
    f"\n  Dataset completeness : "
    f"{'PASS' if dataset_completeness_pass else 'FAIL'}"
)


# ==============================================================================
# 25.4 — OUTPUT DIRECTORY VERIFICATION
# ==============================================================================

print("\n[25.4] Output directory verification")
print("-" * 100)

required_directory_keys = [
    "processed_root",
    "native",
    "encoded",
    "splits",
    "preprocessors",
    "schemas",
    "feature_mapping",
]

directory_records = []

for key in required_directory_keys:

    directory_path = Path(
        NB02_DIRECTORIES[key]
    )

    exists = (
        directory_path.exists()
        and directory_path.is_dir()
    )

    directory_records.append(
        {
            "directory_key": key,
            "path": str(directory_path),
            "exists": exists,
        }
    )

    print(
        f"  {'✓' if exists else '✗'} "
        f"{key:<25} : "
        f"{directory_path}"
    )


DIRECTORY_VERIFICATION_DF = pd.DataFrame(
    directory_records
)

directory_pass = bool(
    DIRECTORY_VERIFICATION_DF[
        "exists"
    ].all()
)

print(
    f"\n  Directory verification : "
    f"{'PASS' if directory_pass else 'FAIL'}"
)


# ==============================================================================
# 25.5 — PREPROCESSOR / SCHEMA ARTIFACT VERIFICATION
# ==============================================================================

print("\n[25.5] Preprocessor and schema artifact verification")
print("-" * 100)

schema_root = Path(
    NB02_DIRECTORIES["schemas"]
)

preprocessor_root = Path(
    NB02_DIRECTORIES["preprocessors"]
)

preprocessor_artifact_records = []

for dataset_id in DATASET_IDS:

    preprocessor_path = (
        preprocessor_root
        / dataset_id
        / "train_fitted_preprocessor.joblib"
    )

    schema_path = (
        schema_root
        / dataset_id
        / "preprocessing_schema.json"
    )

    metadata_path = (
        schema_root
        / "metadata"
        / f"{dataset_id}_preprocessing_metadata.json"
    )

    preprocessor_found = (
        preprocessor_path.is_file()
        and preprocessor_path.stat().st_size > 0
    )

    schema_found = (
        schema_path.is_file()
        and schema_path.stat().st_size > 0
    )

    metadata_found = (
        metadata_path.is_file()
        and metadata_path.stat().st_size > 0
    )

    artifacts_pass = (
        preprocessor_found
        and schema_found
        and metadata_found
    )

    print(f"\n  {dataset_id}")

    print(
        f"    {'✓' if preprocessor_found else '✗'} "
        f"Preprocessor : "
        f"{preprocessor_path}"
    )

    print(
        f"    {'✓' if schema_found else '✗'} "
        f"Schema       : "
        f"{schema_path}"
    )

    print(
        f"    {'✓' if metadata_found else '✗'} "
        f"Metadata     : "
        f"{metadata_path}"
    )

    print(
        f"    Status       : "
        f"{'PASS' if artifacts_pass else 'FAIL'}"
    )

    preprocessor_artifact_records.append(
        {
            "dataset_id": dataset_id,
            "preprocessor": preprocessor_found,
            "schema": schema_found,
            "metadata": metadata_found,
            "artifacts_complete": artifacts_pass,
        }
    )


PREPROCESSOR_ARTIFACTS_DF = pd.DataFrame(
    preprocessor_artifact_records
)

preprocessor_artifacts_pass = bool(
    PREPROCESSOR_ARTIFACTS_DF[
        "artifacts_complete"
    ].all()
)

print(
    f"\n  Preprocessor artifacts : "
    f"{'PASS' if preprocessor_artifacts_pass else 'FAIL'}"
)


# ==============================================================================
# 25.6 — SECTION 21 FEATURE MAPPING ARTIFACT VERIFICATION
# ==============================================================================

print("\n[25.6] Section 21 feature mapping artifact verification")
print("-" * 100)

feature_mapping_root = Path(
    NB02_DIRECTORIES["feature_mapping"]
)

feature_mapping_csv = (
    feature_mapping_root
    / "feature_mapping.csv"
)

feature_mapping_summary_csv = (
    feature_mapping_root
    / "feature_mapping_summary.csv"
)

feature_mapping_metadata_json = (
    feature_mapping_root
    / "feature_mapping_metadata.json"
)

feature_mapping_paths = [
    (
        "Section 21 | feature mapping",
        feature_mapping_csv,
    ),
    (
        "Section 21 | feature mapping summary",
        feature_mapping_summary_csv,
    ),
    (
        "Section 21 | feature mapping metadata",
        feature_mapping_metadata_json,
    ),
]

feature_mapping_records = []

for artifact_name, artifact_path in feature_mapping_paths:

    found = (
        artifact_path.is_file()
        and artifact_path.stat().st_size > 0
    )

    print(
        f"  {'✓' if found else '✗'} "
        f"{artifact_name:<45} : "
        f"{'FOUND' if found else 'MISSING'}"
    )

    feature_mapping_records.append(
        {
            "artifact": artifact_name,
            "path": str(artifact_path),
            "exists": found,
        }
    )


FEATURE_MAPPING_ARTIFACTS_DF = pd.DataFrame(
    feature_mapping_records
)

feature_mapping_pass = bool(
    FEATURE_MAPPING_ARTIFACTS_DF[
        "exists"
    ].all()
)

print(
    f"\n  Feature mapping artifacts : "
    f"{'PASS' if feature_mapping_pass else 'FAIL'}"
)


# ==============================================================================
# 25.7 — SECTION 22 SPLIT MANIFEST VERIFICATION
# ==============================================================================

print("\n[25.7] Section 22 split manifest verification")
print("-" * 100)

# ------------------------------------------------------------------------------
# IMPORTANT:
# Section 22 stores validation/manifests under:
#
# results/raw_validation/notebook_02_manifests/
#
# This is the canonical location confirmed by the diagnostic.
# ------------------------------------------------------------------------------

project_root = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

split_manifest_root = (
    project_root
    / "results"
    / "raw_validation"
    / "notebook_02_manifests"
)

split_manifest_path = (
    split_manifest_root
    / "split_manifest.csv"
)

split_manifest_found = (
    split_manifest_path.is_file()
    and split_manifest_path.stat().st_size > 0
)

print(
    f"  Manifest root : "
    f"{split_manifest_root}"
)

print(
    f"  {'✓' if split_manifest_found else '✗'} "
    f"Section 22 | split manifest : "
    f"{'FOUND' if split_manifest_found else 'MISSING'}"
)

if split_manifest_found:

    print(
        f"    Size : "
        f"{split_manifest_path.stat().st_size:,} bytes"
    )


# ==============================================================================
# 25.8 — SECTION 23 RELOAD VALIDATION
# ==============================================================================

print("\n[25.8] Section 23 reload validation verification")
print("-" * 100)

reload_validation_path = (
    schema_root
    / "reload_validation.csv"
)

reload_validation_found = (
    reload_validation_path.is_file()
    and reload_validation_path.stat().st_size > 0
)

print(
    f"  {'✓' if reload_validation_found else '✗'} "
    f"Section 23 | reload validation : "
    f"{'FOUND' if reload_validation_found else 'MISSING'}"
)


# ==============================================================================
# 25.9 — SECTION 24 FINAL INTEGRITY ARTIFACTS
# ==============================================================================

print("\n[25.9] Section 24 final integrity artifacts")
print("-" * 100)

final_integrity_path = (
    schema_root
    / "final_integrity_verification.csv"
)

final_integrity_metadata_path = (
    schema_root
    / "final_integrity_metadata.json"
)

final_integrity_found = (
    final_integrity_path.is_file()
    and final_integrity_path.stat().st_size > 0
)

final_integrity_metadata_found = (
    final_integrity_metadata_path.is_file()
    and final_integrity_metadata_path.stat().st_size > 0
)

print(
    f"  {'✓' if final_integrity_found else '✗'} "
    f"Final integrity verification : "
    f"{'FOUND' if final_integrity_found else 'MISSING'}"
)

print(
    f"  {'✓' if final_integrity_metadata_found else '✗'} "
    f"Final integrity metadata     : "
    f"{'FOUND' if final_integrity_metadata_found else 'MISSING'}"
)


# ==============================================================================
# 25.10 — GLOBAL ARTIFACT COMPLETENESS GATE
# ==============================================================================

artifact_completeness_pass = all(
    [
        directory_pass,
        preprocessor_artifacts_pass,
        feature_mapping_pass,
        split_manifest_found,
        reload_validation_found,
        final_integrity_found,
        final_integrity_metadata_found,
    ]
)

print("\n[25.10] Global artifact completeness")
print("-" * 100)

print(
    f"  Output directories       : "
    f"{'PASS' if directory_pass else 'FAIL'}"
)

print(
    f"  Preprocessor artifacts   : "
    f"{'PASS' if preprocessor_artifacts_pass else 'FAIL'}"
)

print(
    f"  Feature mapping          : "
    f"{'PASS' if feature_mapping_pass else 'FAIL'}"
)

print(
    f"  Split manifest           : "
    f"{'PASS' if split_manifest_found else 'FAIL'}"
)

print(
    f"  Reload validation        : "
    f"{'PASS' if reload_validation_found else 'FAIL'}"
)

print(
    f"  Final integrity report   : "
    f"{'PASS' if final_integrity_found else 'FAIL'}"
)

print(
    f"  Final integrity metadata : "
    f"{'PASS' if final_integrity_metadata_found else 'FAIL'}"
)

print(
    f"\n  Artifact completeness    : "
    f"{'PASS' if artifact_completeness_pass else 'FAIL'}"
)


# ==============================================================================
# 25.11 — FINAL NOTEBOOK STATUS
# ==============================================================================

notebook_02_complete = (
    section_24_pass
    and dataset_completeness_pass
    and artifact_completeness_pass
)

print("\n" + "=" * 100)
print("NOTEBOOK 02 FINAL STATUS")
print("=" * 100)

print(
    f"  Section 24 final integrity gate : "
    f"{'PASS' if section_24_pass else 'FAIL'}"
)

print(
    f"  Dataset completeness             : "
    f"{'PASS' if dataset_completeness_pass else 'FAIL'}"
)

print(
    f"  Artifact completeness            : "
    f"{'PASS' if artifact_completeness_pass else 'FAIL'}"
)

print(
    f"\n  NOTEBOOK 02 STATUS               : "
    f"{'COMPLETE' if notebook_02_complete else 'FAIL'}"
)


# ==============================================================================
# 25.12 — SAVE FINAL COMPLETION ARTIFACTS
# ==============================================================================

completion_root = schema_root

completion_root.mkdir(
    parents=True,
    exist_ok=True
)

completion_timestamp = (
    datetime.utcnow()
    .replace(microsecond=0)
    .isoformat()
    + "Z"
)

NOTEBOOK_02_COMPLETION_SUMMARY = {
    "notebook": "02",
    "notebook_name":
        "Preprocessing, Encoding & Data Splits",
    "completion_timestamp_utc":
        completion_timestamp,
    "datasets_processed":
        len(DATASET_IDS),
    "section_24_final_integrity_pass":
        section_24_pass,
    "dataset_completeness_pass":
        dataset_completeness_pass,
    "artifact_completeness_pass":
        artifact_completeness_pass,
    "notebook_02_complete":
        notebook_02_complete,
    "split_policy":
        "70/15/15 stratified",
    "random_seed":
        int(
            globals().get(
                "RANDOM_SEED",
                globals().get(
                    "MASTER_SEED",
                    2025
                )
            )
        ),
    "identifier_policy":
        "Explicit identifiers excluded from modeling",
    "provenance_policy":
        "__original_row_id__ retained only in native audit datasets",
    "target_policy":
        "Target retained in modeling schema and categorically encoded",
    "preprocessing_fit_policy":
        "Training split only; validation/test never refit",
    "split_manifest_path":
        str(split_manifest_path),
}

completion_metadata_path = (
    completion_root
    / "notebook_02_completion_metadata.json"
)

with open(
    completion_metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        NOTEBOOK_02_COMPLETION_SUMMARY,
        f,
        indent=2
    )

DATASET_COMPLETENESS_DF.to_csv(
    completion_root
    / "notebook_02_dataset_completion.csv",
    index=False
)

PREPROCESSOR_ARTIFACTS_DF.to_csv(
    completion_root
    / "notebook_02_preprocessor_artifacts.csv",
    index=False
)

FEATURE_MAPPING_ARTIFACTS_DF.to_csv(
    completion_root
    / "notebook_02_feature_mapping_artifacts.csv",
    index=False
)

DIRECTORY_VERIFICATION_DF.to_csv(
    completion_root
    / "notebook_02_directory_verification.csv",
    index=False
)


# ==============================================================================
# 25.13 — HARD FINAL GATE
# ==============================================================================

if not notebook_02_complete:

    raise RuntimeError(
        "\n"
        + "=" * 100
        + "\n"
        + "NOTEBOOK 02 COMPLETION GATE FAILED"
        + "\n"
        + "=" * 100
        + "\n"
        + f"Section 24 integrity : "
        + f"{section_24_pass}\n"
        + f"Dataset completeness : "
        + f"{dataset_completeness_pass}\n"
        + f"Artifact completeness: "
        + f"{artifact_completeness_pass}\n"
        + "\n"
        + "Inspect the failed gate(s) above."
    )


print("\n" + "=" * 100)
print("NOTEBOOK 02 COMPLETION GATE PASSED")
print("=" * 100)

print(
    f"\nCompletion metadata saved to:\n"
    f"{completion_metadata_path}"
)

print(
    "\nNotebook 02 is COMPLETE and ready "
    "for downstream notebooks."
)

print("=" * 100)

SECTION 25 — NOTEBOOK 02 FINAL COMPLETION GATE

[25.1] Required canonical objects
----------------------------------------------------------------------------------------------------
  ✓ DATASET_IDS
  ✓ TRAIN_DATASETS
  ✓ VALIDATION_DATASETS
  ✓ TEST_DATASETS
  ✓ TRAIN_PREPROCESSING_DATA
  ✓ TRAIN_PREPROCESSING_COLUMNS
  ✓ TRAIN_PREPROCESSING_METADATA
  ✓ TRAIN_PREPROCESSORS
  ✓ TRANSFORMED_TRAIN_DATASETS
  ✓ TRANSFORMED_VALIDATION_DATASETS
  ✓ TRANSFORMED_TEST_DATASETS
  ✓ NATIVE_FINAL_DATASETS
  ✓ ENCODED_FINAL_DATASETS
  ✓ NB02_DIRECTORIES
  Required final objects : PASS

[25.2] Section 24 final integrity gate
----------------------------------------------------------------------------------------------------
  Section 24 final integrity gate : PASS

[25.3] Dataset completeness verification
----------------------------------------------------------------------------------------------------

  Dataset: adult_income
    Original rows       : 48,842
    Modeling columns    : 15
    Num